# 動能基準研究：24 交易日回測
固定參數第一輪比較，不以最好窗口或年化 Sharpe 選策略。每個窗口重新從 10 億現金開始，持續 24 個交易日，逐日向前移動。
候選：等權流動性參考、20/60 日動能、動能加法人、動能加低波動。皆持有 25 檔、目標現金 10%；A/B/C 每五日調整，AH/BH 僅初次買入後持有。後兩者是觀察換手成本後新增的敏感性實驗，不能視為獨立樣本外選模。
限制：目前官方股票名單回推有存活偏差；歷史 ETF 持股不足，不能驗證 Active Share；資料更新時間不能代表當年實際可得時間；本研究不模擬 Agent 理由品質、提交失敗與官方整批作廢。

本批以動能為研究起點：REF 為參考組，A/B/C/AH/BH 為動能的訊號或持有頻率變體，不代表六種獨立策略家族。

In [ ]:
# Locate this competition repository; do not restart an existing kernel.
import sys as _repo_sys, importlib.util as _repo_importlib
from pathlib import Path as _PlainPath
_candidates = [_PlainPath.cwd(), *_PlainPath.cwd().parents]
_repo_root = next((p for p in _candidates if (p / '99_維護與備份' / '工具' / '研究路徑.py').exists()), None)
if _repo_root is None:
    raise RuntimeError('Open Jupyter from this repository root or its 04_Notebook directory.')
if 'research_layout' not in _repo_sys.modules:
    _spec = _repo_importlib.spec_from_file_location('research_layout', _repo_root / '99_維護與備份' / '工具' / '研究路徑.py')
    _mod = _repo_importlib.module_from_spec(_spec)
    _repo_sys.modules['research_layout'] = _mod
    _spec.loader.exec_module(_mod)
from research_layout import ResearchPath as Path
%whos
import sys, numpy as np
print(sys.executable, np.__version__)
%pip -q install pymysql
import pymysql
print('pymysql ready')


## 集中參數與唯讀資料
密碼只從本機既有連線說明讀取，Notebook 不含密碼，也不輸出連線資訊。來源為團隊 market DB 的 daily_price、institutional、dividend、stocks、universe。查詢在唯讀交易中執行，完成後 rollback 並關閉。


In [ ]:
from research_layout import ResearchPath as Path
import re, json, hashlib, datetime
HORIZON, LOOKBACK, COUNT = 24, 60, 25
INITIAL, INVESTED, FEE, TAX, LOT = 1e9, .90, .001425, .003, 1000
REBALANCE, SEED = 5, 20261013
config_text = next((Path.home()/'Downloads').glob('AI_CUP_DB*.md')).read_text(encoding='utf-8-sig')
config = {k:v.strip().strip('`').strip('"').strip("'") for k,v in re.findall(r'(?m)^(DB_[A-Z_]+)\s*=\s*(.*)$',config_text)}
del config_text
try:
    conn = pymysql.connect(host=config['DB_HOST'], port=int(config['DB_PORT']), user=config['DB_USER'], password=config['DB_PASSWORD'], database=config['DB_NAME'], charset='utf8mb4', connect_timeout=15, read_timeout=120, autocommit=False)
    del config
    with conn.cursor() as cur:
        cur.execute('SET SESSION TRANSACTION READ ONLY')
        cur.execute('START TRANSACTION WITH CONSISTENT SNAPSHOT, READ ONLY')
        queries = {
          'universe':'SELECT stock_id FROM universe WHERE active=1 ORDER BY stock_id',
          'stocks':'SELECT stock_id, listed_date FROM stocks ORDER BY stock_id',
          'price':'SELECT stock_id,trade_date,close,volume,turnover,vwap,high,low FROM daily_price WHERE trade_date <= %s ORDER BY trade_date,stock_id',
          'institution':'SELECT stock_id,trade_date,foreign_net,trust_net FROM institutional WHERE trade_date <= %s ORDER BY trade_date,stock_id',
          'dividend':'SELECT stock_id,ex_date,cash_dividend,stock_dividend_ratio,before_close,reference_price,status FROM dividend WHERE ex_date <= %s ORDER BY ex_date,stock_id'}
        raw_data = {}
        for name,query in queries.items():
            cur.execute(query, ('2026-10-12',) if '%s' in query else None)
            raw_data[name] = cur.fetchall()
    conn.rollback(); conn.close(); del conn
    print({k:len(v) for k,v in raw_data.items()})
except Exception as exc:
    if 'conn' in globals(): conn.close(); del conn
    if 'config' in globals(): del config
    raise RuntimeError('Read-only extraction failed; details suppressed to protect credentials') from None


## 資料品質與公司行動
價格矩陣不對缺漏成交價補值。僅估值價格可沿用前收；選股須有完整 60 日訊號。交易日均價優先採成交金額/成交股數，並檢查落在高低價內。配息配股按除權息日前持股計入應收，24 日內不加入可交易現金或股數；應收股隨收盤估值。動能使用除權息修正日報酬累積，避免把配息當成下跌。


In [ ]:
stock_ids = [str(r[0]) for r in raw_data['universe']]
dates = sorted({r[1] for r in raw_data['price']})
di, si = {d:i for i,d in enumerate(dates)}, {s:i for i,s in enumerate(stock_ids)}
shape = (len(dates),len(stock_ids))
close, volume, turnover, vwap, high, low = [np.full(shape,np.nan) for _ in range(6)]
for sid,date,*values in raw_data['price']:
    if sid in si:
        for arr,value in zip((close,volume,turnover,vwap,high,low), values):
            arr[di[date],si[sid]] = float(value) if value is not None else np.nan
flow = np.full(shape,np.nan)
for sid,date,f,t in raw_data['institution']:
    if sid in si and date in di and f is not None and t is not None:
        flow[di[date],si[sid]]=float(f)+float(t)
cash_div, stock_div = np.zeros(shape), np.zeros(shape)
div_status = sorted({str(r[-1]) for r in raw_data['dividend']})
for sid,date,c,s,*rest in raw_data['dividend']:
    if sid in si and date in di:
        cash_div[di[date],si[sid]] += float(c or 0)
        stock_div[di[date],si[sid]] += float(s or 0)
execution = np.divide(turnover,volume,out=np.full(shape,np.nan),where=volume>0)
good_exec = np.isfinite(execution)&(execution>=low-.02)&(execution<=high+.02)&(close>0)
mark = close.copy()
for t in range(1,len(dates)):
    mark[t]=np.where(np.isfinite(mark[t])&(mark[t]>0),mark[t],mark[t-1])
returns = np.full(shape,np.nan)
returns[1:] = (close[1:]*(1+stock_div[1:])+cash_div[1:])/close[:-1]-1
gross_index = np.ones(shape)
for t in range(1,len(dates)):
    gross_index[t] = gross_index[t-1]*np.where(np.isfinite(returns[t]),1+returns[t],1)
print('Dates:', str(dates[0]),str(dates[-1]),'sessions:',len(dates),'stocks:',len(stock_ids))
print('Missing closes:',int(np.isnan(close).sum()),'invalid execution rows:',int((np.isfinite(close)&~good_exec).sum()))
print('Dividend statuses:',div_status,'max stock dividend ratio:',float(stock_div.max()))
print('VWAP mismatch >0.02:',int((np.isfinite(vwap)&(np.abs(execution-vwap)>.02)).sum()))
print('Corporate action samples:',[(str(r[0]),str(r[1]),float(r[2] or 0),float(r[3] or 0),str(r[-1])) for r in raw_data['dividend'][:5]])
print('Annual trading rows:', {y:sum(d.year==y for d in dates) for y in sorted({d.year for d in dates})})


In [ ]:
print('Zero distribution rows:',sum(not (r[2] or r[3]) for r in raw_data['dividend']))
print('Reference samples:',[(str(r[0]),str(r[1]),*[float(x or 0) for x in r[2:6]]) for r in raw_data['dividend'][:12]])
print('Large falls:',[(str(dates[t]),stock_ids[j],round(float(returns[t,j]),3)) for t,j in np.argwhere(returns < -.25)][:30])

## 固定候選與缺漏處理
參考組在窗口開始用過去 20 日平均成交金額選 25 檔，等權買入並持有。A 為 20/60 日動能百分位平均；B 為 75% 動能＋25% 過去 20 日（外資＋投信）淨買股數/成交股數百分位；C 為 75% 動能＋25% 過去 20 日低波動百分位。A/B/C 每 5 日調整，皆等權。
資料顯示 56 筆配息配股皆零，部分除權參考價卻降低。不能據此擅自填配股比例。除權參考價僅用於訊號的當日價格調整；若持股遇到應收不足以解釋參考價的公司行動，或無公司行動解釋的單日跌幅超過 25%，標記該窗口為不可判定。四策略採共同有效窗口比較。這會造成資料品質相關的樣本選擇偏差，結論只能視為初步。


In [ ]:
bad_action = np.zeros(shape,dtype=bool)
signal_returns = returns.copy()
for sid,date,c,s,before,ref,status in raw_data['dividend']:
    if sid not in si or date not in di: continue
    t,j = di[date],si[sid]
    if before and ref and float(ref)>0:
        expected=(float(before)-float(c or 0))/(1+float(s or 0))
        bad_action[t,j] = abs(expected-float(ref)) > max(.03,float(before)*.0002)
        signal_returns[t,j] = close[t,j]/float(ref)-1
bad_action |= (returns < -.25)
def percentile(x, eligible):
    out=np.full(len(x),np.nan)
    idx=np.flatnonzero(eligible & np.isfinite(x))
    # Equal values receive average ranks, so stock-code order does not affect factor scores.
    if len(idx):
        val=x[idx]
        out[idx]=(np.sum(val[:,None]>val[None,:],axis=1)+.5*np.sum(val[:,None]==val[None,:],axis=1))/len(idx)
    return out
scores={k:np.full(shape,np.nan) for k in ('REF','A','B','C')}
for t in range(60,len(dates)):
    rr=signal_returns[t-59:t+1]
    ff=flow[t-19:t+1]
    vv=volume[t-19:t+1]
    eligible=np.all(np.isfinite(rr)&(rr>-1),axis=0)&np.all(np.isfinite(ff)&np.isfinite(vv)&(vv>0),axis=0)&(close[t]>0)
    eligible &= ~np.any(bad_action[t-59:t+1],axis=0)
    m20=np.prod(1+rr[-20:],axis=0)-1
    m60=np.prod(1+rr,axis=0)-1
    momentum=.5*percentile(m20,eligible)+.5*percentile(m60,eligible)
    fscore=np.sum(ff,axis=0)/np.sum(vv,axis=0)
    vol=np.std(rr[-20:],axis=0)
    scores['REF'][t]=percentile(np.divide(np.nansum(turnover[t-19:t+1],axis=0),np.sum(np.isfinite(turnover[t-19:t+1]),axis=0),out=np.full(shape[1],np.nan),where=np.sum(np.isfinite(turnover[t-19:t+1]),axis=0)>0),eligible)
    scores['A'][t]=momentum
    scores['B'][t]=.75*momentum+.25*percentile(fscore,eligible)
    scores['C'][t]=.75*momentum+.25*percentile(-vol,eligible)
print('Potentially unsupported corporate actions:',int(bad_action.sum()))
print('Signal eligibility last day:',{k:int(np.isfinite(v[-1]).sum()) for k,v in scores.items()})


## 成交與帳戶引擎
每日上午使用 t−1 訊號、前收與可交易淨值換算整千股；當日均價成交、收盤估值，買賣費率與賣出税均扣除。同日賣出淨所得可支付買入，應收配息不得用於下單。無成交價格的委託留在原持股並標記不可判定；不自行聲稱符合官方作廢規則。每日記錄現金、持股數、权重超限與未知公司行動。


In [ ]:
def settle(cash, shares, target, prices, available, fee=FEE, tax=TAX):
    delta=target-shares
    executed=np.where(available,delta,0)
    safe_prices=np.where(available,prices,0)
    buys=np.sum(np.maximum(executed,0)*safe_prices)
    sells=np.sum(np.maximum(-executed,0)*safe_prices)
    cost=buys*fee+sells*(fee+tax)
    return cash+sells-buys-cost, shares+executed, buys+sells, cost, bool(np.any((delta!=0)&~available))
def simulate(start,strategy,horizon=HORIZON):
    shares=np.zeros(len(stock_ids),dtype=np.int64)
    cash=INITIAL; receivable_cash=0.; receivable_shares=np.zeros(len(stock_ids))
    values=[INITIAL]; gross_traded=0.; fees=0.; invalid=False
    violations=0; min_cash=1.; max_cash=0.; cap_age=np.zeros(len(stock_ids),dtype=int)
    for offset,t in enumerate(range(start,start+horizon)):
        before=shares.copy()
        invalid |= bool(np.any(bad_action[t] & ((shares+receivable_shares)>0)))
        receivable_cash += float(np.dot(before,cash_div[t]))
        receivable_shares += before*stock_div[t]
        previous_nav=cash+np.dot(shares,np.nan_to_num(mark[t-1]))
        rebalance=offset==0 or (strategy!='REF' and not strategy.endswith('H') and offset%REBALANCE==0)
        if rebalance:
            rank=scores[strategy][t-1]
            eligible=np.flatnonzero(np.isfinite(rank))
            if len(eligible)<COUNT:
                invalid=True; break
            selected=eligible[np.argsort(-rank[eligible],kind='stable')[:COUNT]]
            target=np.zeros_like(shares)
            target[selected]=(np.floor((INVESTED/COUNT)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)
            cash,shares,traded,cost,missing=settle(cash,shares,target,execution[t],good_exec[t])
            gross_traded += traded; fees += cost; invalid |= missing
        nav=cash+np.dot(shares,np.nan_to_num(mark[t]))
        aum=nav+receivable_cash+np.dot(receivable_shares,np.nan_to_num(mark[t]))
        invalid |= bool(np.any((shares>0)&~np.isfinite(mark[t])))
        weights=shares*np.nan_to_num(mark[t])/nav
        caps=np.array([.25 if s=='2330' else .10 for s in stock_ids])
        cap_age=np.where(weights>caps+1e-12,cap_age+1,0)
        violations += int(cash<0 or cash/nav>=.25 or not 20<=np.count_nonzero(shares)<=30 or np.any(cap_age>5))
        min_cash=min(min_cash,cash/nav);max_cash=max(max_cash,cash/nav)
        values.append(aum)
    vals=np.array(values)
    mdd=np.min(vals/np.maximum.accumulate(vals)-1)
    return {'start':start,'end':start+horizon-1,'return':vals[-1]/INITIAL-1,'mdd':mdd,'turnover':gross_traded/INITIAL,'cost':fees/INITIAL,'invalid':invalid,'violations':violations,'min_cash':min_cash,'max_cash':max_cash,'path':vals}
# Independent accounting checks: entry fee, same-day netting, unavailable prices.
c,q,tr,fee,missing=settle(100000.,np.array([0]),np.array([1000]),np.array([90.]),np.array([True]))
assert abs(c-9871.75)<1e-8 and q[0]==1000 and abs(fee-128.25)<1e-8
c,q,_,fee,_=settle(1000.,np.array([1000,0]),np.array([0,1000]),np.array([100.,90.]),np.array([True,True]))
assert abs(c-10429.25)<1e-8 and np.array_equal(q,[0,1000])
c,q,_,_,missing=settle(1000.,np.array([1000]),np.array([0]),np.array([np.nan]),np.array([False]))
assert c==1000 and q[0]==1000 and missing
print('3 accounting checks passed')
window_starts=np.arange(61,len(dates)-HORIZON+1)
backtest_results={k:[simulate(int(s),k) for s in window_starts] for k in scores}
common_valid=np.array([all(not backtest_results[k][i]['invalid'] for k in scores) for i in range(len(window_starts))])
print('Sliding windows:',len(window_starts),'common valid:',int(common_valid.sum()))
print('Invalid by strategy:',{k:sum(r['invalid'] for r in v) for k,v in backtest_results.items()})
print('Violation days:',{k:sum(r['violations'] for r in v if not r['invalid']) for k,v in backtest_results.items()})


## 24 日持有敏感性實驗
看到週調整成本偏高後，新增 AH（同 A 選股，24 日持有）與 BH（同 B 選股，24 日持有）。沿用訊號、股票數與權重，僅更改調整頻率。這是事後探索，下一次須使用未參與選模的期間驗證；價格漂移若造成五日以上超限，仍計違規。


In [ ]:
scores['AH']=scores['A'].copy()
scores['BH']=scores['B'].copy()
backtest_results['AH']=[simulate(int(s),'AH') for s in window_starts]
backtest_results['BH']=[simulate(int(s),'BH') for s in window_starts]
common_valid=np.array([all(not backtest_results[k][i]['invalid'] for k in scores) for i in range(len(window_starts))])
print('Six-strategy common windows:',int(common_valid.sum()))
print('Invalid by strategy:',{k:sum(r['invalid'] for r in v) for k,v in backtest_results.items()})
print('Violation days on common windows:',{k:sum(r['violations'] for i,r in enumerate(v) if common_valid[i]) for k,v in backtest_results.items()})


## 24 日統計與相依性
固定参数先跑較早年份，再獨立報告 2026 年作為時間穩定性檢查；未做參數搜尋，亦非事先封存的正式樣本外驗證。只計入四組共同可判定窗口。窗口跨越年界不列入時間切分。
勝率是超越「25 檔高流動性股票、90% 投資、24 日持有」參考組的比例，並非競賽奪冠率。全部數字為 24 日未年化、扣成本而不假設期末賣出的總資產報酬。
重疊窗口不獨立：對完整時間軸（保留缺漏）做 24/48 個窗口位置的 moving-block bootstrap，配對超額報酬只用相同日期。另列 24 種起始相位、每 24 日一窗口的不重疊結果；不同相位也不是 24 組獨立研究。


In [ ]:
from IPython.display import display, Markdown
labels={'REF':'參考：流動性等權持有','A':'A：每週動能','B':'B：動能＋法人','C':'C：動能＋低波動','AH':'AH：動能持有24日','BH':'BH：動能法人持有24日'}
metrics={k:{m:np.array([r[m] for r in v]) for m in ('return','mdd','turnover','cost','min_cash','max_cash','violations')} for k,v in backtest_results.items()}
train=common_valid & np.array([dates[s+HORIZON-1]<datetime.date(2026,1,1) for s in window_starts])
test=common_valid & np.array([dates[s]>=datetime.date(2026,1,1) for s in window_starts])
def table(mask):
    lines=['|策略|窗口數|報酬中位數|報酬P10|勝過參考|MDD中位數|MDD P10|平均雙邊換手|平均成本|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    for k in scores:
        a=metrics[k]; r=a['return'][mask]
        if not len(r):continue
        vals=[np.median(r),np.quantile(r,.1),np.mean(r>metrics['REF']['return'][mask]),np.median(a['mdd'][mask]),np.quantile(a['mdd'][mask],.1),np.mean(a['turnover'][mask]),np.mean(a['cost'][mask])]
        lines.append('|'+labels[k]+'|'+str(len(r))+'|'+'|'.join(f'{v:.2%}' for v in vals)+'|')
    return '\n'.join(lines)
for title,mask in [('全部共同有效窗口',common_valid),('2024–2025時間區間',train),('2026時間檢查區間',test)]:
    display(Markdown('### '+title+'\n\n'+table(mask)))
def block_ci(x,block,n_boot=2000):
    rng=np.random.default_rng(SEED+block)
    n=len(x); draws=[]
    for _ in range(n_boot):
        starts=rng.integers(0,n-block+1,size=int(np.ceil(n/block)))
        sample=np.concatenate([x[s:s+block] for s in starts])[:n]
        if np.any(np.isfinite(sample)):draws.append(np.nanmean(sample))
    return np.quantile(draws,[.025,.975])
bootstrap_results={}
test_positions=np.flatnonzero(np.array([dates[s]>=datetime.date(2026,1,1) for s in window_starts]))
for k in ('A','B','C','AH','BH'):
    excess=metrics[k]['return']-metrics['REF']['return']
    series=np.where(common_valid,excess,np.nan)[test_positions]
    bootstrap_results[k]={b:block_ci(series,b).tolist() for b in (24,48)}
    phase_means=[];phase_counts=[]
    for phase in range(24):
        idx=np.arange(phase,len(series),24)
        usable=series[idx][np.isfinite(series[idx])]
        if len(usable): phase_means.append(float(np.mean(usable)));phase_counts.append(len(usable))
    print(labels[k], '2026 paired mean excess:',f'{np.nanmean(series):.2%}', 'CI24:',[f'{x:.2%}' for x in bootstrap_results[k][24]],'CI48:',[f'{x:.2%}' for x in bootstrap_results[k][48]],'nonoverlap phase mean range:',[f'{x:.2%}' for x in (min(phase_means),max(phase_means))], 'windows per phase:',(min(phase_counts),max(phase_counts)))
print('Common valid windows by year:',{y:int(sum(common_valid[i] and dates[s].year==y for i,s in enumerate(window_starts))) for y in (2024,2025,2026)})
print('Cash fraction range:',{k:[round(float(metrics[k]['min_cash'][common_valid].min()),4),round(float(metrics[k]['max_cash'][common_valid].max()),4)] for k in scores})


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(12,4))
for k in ('A','B','C','AH','BH'):
    excess=(metrics[k]['return']-metrics['REF']['return'])*100
    yy=np.where(common_valid,excess,np.nan)
    ax[0].plot([dates[s] for s in window_starts],yy,label=k,alpha=.7)
    x=np.sort(metrics[k]['return'][test]*100)
    ax[1].plot(x,np.arange(1,len(x)+1)/len(x),label=k)
ref=np.sort(metrics['REF']['return'][test]*100)
ax[1].plot(ref,np.arange(1,len(ref)+1)/len(ref),label='REF',color='black')
ax[0].axhline(0,color='black',lw=.7);ax[0].set_title('24-session paired excess return');ax[0].set_ylabel('Percentage points');ax[0].legend()
ax[1].set_title('2026 valid windows: return CDF');ax[1].set_xlabel('24-session return (%)');ax[1].legend()
fig.autofmt_xdate();fig.tight_layout();plt.show()


## 初步判斷與驗證狀態
六組共同有效窗口 375 個，其中 2026 年僅 71 個。全部窗口報酬中位數 A 5.15%、B 4.94%、C 4.29%、AH 4.65%、BH 4.68%，參考組 4.60%。2026 年 B 中位數 17.56% 較高，但對參考組的平均配對超額僅 1.20 個百分點，24 日區塊重抽樣 95% 區間約 −6.09 至 8.67 個百分點，無法判定有穩定優勢。
週調整 A/B 平均成本約本金 0.79%/0.81%；持有 AH/BH 約 0.13%。AH 的全期勝過參考比例較高（58.13%），但 2026 年只有 50.70%；BH 在 2026 年甚至只有 47.89%。低波動版本在較早年份回撤較小，但 2026 年沒有維持這個優勢。因此目前保留 A、B、AH 作為下一輪研究候選，尚不定案競賽策略；不以單一中位數排名直接部署。
優先補齊配股、分割、減資等公司行動原始資料與實際可得時點，再重新做完整窗口比較；再確認 Active Share 與官方違規判定。泛用競賽案例的繼續蒐集目前不如補齊這些資料有用。營收/新聞事件版尚未回測，避免將月份或更新時間當作實際公告時間。
3 項獨立帳務檢查通過（首次手續費、同日淨額交割、無成交價不成交），共同有效窗口觀察到的現金/股數/價格漂移限制違規天數為零；這不等於完整官方合規。未模擬成交量限制、市場衝擊、Agent 文件檢核或整批作廢。共同有效窗口排除 275/650，存在顯著資料品質樣本選擇偏差，且使用現在官方名單回推有存活偏差。
各程式 cell 已在專用持續 kernel 執行，圖表輸出已嵌入 Notebook。未重新啟動或清除任何 kernel；尚未做使用者需批准的 Restart Kernel and Run All 驗證。Notebook 不保存原始行情或密碼，未來重跑需可讀取相同 DB；用抽取雜湊檢查資料是否改變。


In [ ]:
import importlib.metadata
snapshot_sha256=hashlib.sha256(json.dumps(raw_data,default=str,ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()
print('Data snapshot SHA256:',snapshot_sha256)
print('Snapshot recorded at:',datetime.datetime.now().isoformat(timespec='seconds'))
print('Environment:',{name:importlib.metadata.version(name) for name in ('numpy','pymysql','matplotlib','ipykernel')})
print('Kernel: kernel-aicup-backtest-20261013.json; unrelated kernels left intact')


## 箱形圖＋全部窗口散點
每組策略皆呈現全部 650 個窗口：上圖为共同有效 375 個窗口的箱形圖與全部散點；下圖列出其餘 275 個窗口的暫算結果，僅作資料品質診斷，不參與箱形圖統計或策略排名。
箱體為第 25–75 百分位、中線為中位數，鬚為 1.5×IQR 範圍內最遠資料點。全部有效點（含離群值）均疊加，因此不另外繪製重複的離群標記。左右抖動只是防止點重疊，沒有時間或數值意義。窗口重疊，散點數不能當成獨立樣本數。報酬含應收估值、已扣手續費及賣出稅，期末不清倉。
互動 HTML 中每一點可查看窗口起訖、策略、暫算報酬、費用及排除狀態；另附 650 列的明細表。


In [ ]:
%whos dict
assert len(window_starts)==650 and len(common_valid)==650
assert all(len(backtest_results[k])==650 for k in ('REF','A','B','C','AH','BH'))
print('Reusing dedicated kernel data; no DB reload, no kernel restart.')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import html
root=Path(r'C:\Users\weiha\Desktop\文件\03_課程資料\115-1\資料探勘與應用\B_Final Project (0920~1221)\競賽資料')
boxplot_keys=['REF','A','B','C','AH','BH']
short_names=['流動性等權','每週動能','動能＋法人','動能＋低波動','動能持有24日','動能法人持有24日']
font_path=Path('C:/Windows/Fonts/msjh.ttc')
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    plt.rcParams['font.family']=font_manager.FontProperties(fname=str(font_path)).get_name()
plt.rcParams['axes.unicode_minus']=False
boxplot_colors=['#64748b','#2563eb','#059669','#9333ea','#ea580c','#0891b2']
boxplot_rng=np.random.default_rng(SEED)
boxplot_jitter={k:boxplot_rng.uniform(-.29,.29,len(window_starts)) for k in boxplot_keys}
boxplot_fig,boxplot_axes=plt.subplots(2,1,figsize=(15,11),gridspec_kw={'height_ratios':[1.5,1]},layout='constrained')
boxplot_data=[metrics[k]['return'][common_valid]*100 for k in boxplot_keys]
bp=boxplot_axes[0].boxplot(boxplot_data,tick_labels=short_names,showfliers=False,patch_artist=True,widths=.48,medianprops={'color':'#111827','linewidth':2})
for patch,color in zip(bp['boxes'],boxplot_colors):patch.set_facecolor(color);patch.set_alpha(.18)
point_count=0
for j,k in enumerate(boxplot_keys,1):
    y=metrics[k]['return']*100
    boxplot_axes[0].scatter(j+boxplot_jitter[k][common_valid],y[common_valid],s=10,c=boxplot_colors[j-1],alpha=.55,linewidths=0,zorder=3,rasterized=False)
    boxplot_axes[1].scatter(j+boxplot_jitter[k][~common_valid],y[~common_valid],s=13,c='#94a3b8',alpha=.55,marker='x',linewidths=.65,rasterized=False)
    point_count+=len(y)
for ax in boxplot_axes:
    ax.axhline(0,color='#475569',linewidth=.8,linestyle='--')
    ax.set_ylabel('24 日報酬（%）');ax.grid(axis='y',alpha=.2);ax.set_xlim(.4,6.6)
boxplot_axes[0].set_title('共同有效窗口：每策略 375 點＋箱形圖（交易費用已扣除）',fontsize=15,pad=14)
boxplot_axes[1].set_xticks(range(1,7),short_names)
boxplot_axes[1].set_title('排除窗口：每策略 275 點的暫算值，僅供診斷，不作績效推論',fontsize=13,pad=10)
boxplot_fig.suptitle('650 個 24 交易日滑動窗口｜六策略共 3,900 個散點',fontsize=19)
png_path=root/'動能基準研究_箱形圖_全部窗口_2026-10-13.png'
boxplot_fig.savefig(png_path,dpi=180,facecolor='white')
assert point_count==3900
assert [len(c.get_offsets()) for c in boxplot_axes[0].collections]==[375]*6
assert [len(c.get_offsets()) for c in boxplot_axes[1].collections]==[275]*6
print('Verified 3,900 scatter points: 650 per strategy; 375 included + 275 excluded.')
plt.show()

# Dependency-free SVG/HTML preserves every individual point with a native tooltip.
def make_svg(included):
    mask=common_valid if included else ~common_valid
    values=np.concatenate([metrics[k]['return'][mask]*100 for k in boxplot_keys])
    ymin,ymax=float(values.min()),float(values.max())
    padding=max((ymax-ymin)*.08,1);ymin-=padding;ymax+=padding
    W,H,left,right,top,bottom=1250,620,80,20,30,80
    plot_h=H-top-bottom
    def yp(v):return top+(ymax-v)/(ymax-ymin)*plot_h
    def xp(j):return left+(j+.5)*(W-left-right)/6
    s=[f'<svg viewBox="0 0 {W} {H}" role="img" aria-label="'+('有效窗口箱形圖' if included else '排除窗口散點圖')+'">']
    for v in np.linspace(ymin,ymax,7):
        yy=yp(float(v));s.append(f'<line x1="{left}" y1="{yy}" x2="{W-right}" y2="{yy}" stroke="#e2e8f0"/><text x="{left-10}" y="{yy+5}" text-anchor="end">{v:.1f}%</text>')
    for j,k in enumerate(boxplot_keys):
        xx=xp(j);color=boxplot_colors[j]
        if included:
            a=metrics[k]['return'][mask]*100
            q1,med,q3=np.quantile(a,[.25,.5,.75]);iqr=q3-q1
            lo=float(a[a>=q1-1.5*iqr].min());hi=float(a[a<=q3+1.5*iqr].max())
            s.append(f'<line x1="{xx}" x2="{xx}" y1="{yp(lo)}" y2="{yp(hi)}" stroke="{color}" stroke-width="2"/><rect x="{xx-38}" y="{yp(q3)}" width="76" height="{yp(q1)-yp(q3)}" fill="{color}" fill-opacity=".12" stroke="{color}"/>')
            for v in (lo,hi,med):s.append(f'<line x1="{xx-38}" x2="{xx+38}" y1="{yp(v)}" y2="{yp(v)}" stroke="{color}" stroke-width="2"/>')
        s.append(f'<text x="{xx}" y="{H-42}" text-anchor="middle">{short_names[j]}</text>')
        for i in np.flatnonzero(mask):
            r=backtest_results[k][int(i)];start=int(window_starts[i]);v=float(r['return'])*100
            own_status='本策略資料也不可判定' if r['invalid'] else '本策略可判定，但其他策略有排除情況'
            status='共同有效，納入箱形圖' if included else '未納入統計：'+own_status
            tip=f"窗口 #{i+1}｜{labels[k]}\n{dates[start]} 至 {dates[start+HORIZON-1]}\n{'報酬' if included else '暫算報酬'}：{v:.3f}%\n費用/本金：{r['cost']:.3%}\n{status}"
            cx=xx+float(boxplot_jitter[k][i])*150;cy=yp(v)
            s.append(f'<circle class="point" data-window="{i+1}" cx="{cx:.3f}" cy="{cy:.3f}" r="3.0" fill="{color if included else "#94a3b8"}" fill-opacity=".65"><title>{html.escape(tip)}</title></circle>')
    s.append('</svg>');return ''.join(s)
rows=[]
for i,start in enumerate(window_starts):
    cells=[str(i+1),str(dates[int(start)]),str(dates[int(start)+HORIZON-1]),'納入' if common_valid[i] else '排除']
    for k in boxplot_keys:
        r=backtest_results[k][i]
        cells.append(f"{r['return']:.3%}"+('（本策略不可判定）' if r['invalid'] else ('（未納入共同比較）' if not common_valid[i] else '')))
    rows.append('<tr>'+''.join('<td>'+html.escape(c)+'</td>' for c in cells)+'</tr>')
heads=['窗口','開始日','結束日','共同狀態']+short_names
html_page='''<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><title>24日策略箱形圖</title><style>body{font-family:"Microsoft JhengHei",sans-serif;margin:28px;background:#f8fafc;color:#0f172a}main{max-width:1400px;margin:auto}section{background:white;border:1px solid #e2e8f0;border-radius:12px;padding:18px;margin:20px 0}svg{width:100%;min-width:950px}svg text{font-size:14px;fill:#334155}.chart{overflow:auto}.point:hover{r:6;fill:#ef4444;fill-opacity:1;stroke:white;stroke-width:1}table{border-collapse:collapse;font-size:13px;width:100%}td,th{padding:8px;border-bottom:1px solid #e2e8f0;white-space:nowrap}th{position:sticky;top:0;background:#e2e8f0}input{font-size:16px;padding:8px;width:300px}.note{line-height:1.7;color:#475569}</style><main><h1>24 日滑動窗口：箱形圖＋全部散點</h1><p class="note">六策略各 650 點，共 3,900 點。上圖 375 個共同有效窗口；下圖 275 個排除窗口。將滑鼠移到點上可看日期、報酬與費用。水平位置只是抖動，不代表日期。箱形圖用同一組 375 個窗口，包含所有離群點。重疊窗口不等於獨立樣本。</p><section><h2>共同有效窗口：箱形圖＋375 點／策略</h2><div class="chart">'''+make_svg(True)+'''</div></section><section><h2>排除窗口：275 點／策略，僅作診斷</h2><p class="note">灰點是未採納的暫算值，可能受公司行動資料不完整或成交價缺漏影響。部分策略本身可判定，但因共同日期比較而一併排除；不可把這些灰點當作可靠績效。</p><div class="chart">'''+make_svg(False)+'''</div></section><section><h2>650 個窗口逐一明細</h2><p class="note">報酬已扣交易費用，期末不清倉。排除列的數值僅為暫算值。</p><input id="search" placeholder="搜尋窗口編號或日期"><p id="count">顯示 650／650 列</p><div style="overflow:auto;max-height:650px"><table><thead><tr>'''+''.join('<th>'+h+'</th>' for h in heads)+'''</tr></thead><tbody id="rows">'''+''.join(rows)+'''</tbody></table></div></section><script>const allRows=[...document.querySelectorAll('#rows tr')];document.querySelector('#search').addEventListener('input',e=>{const q=e.target.value.trim();let n=0;allRows.forEach(r=>{const show=r.textContent.includes(q);r.hidden=!show;if(show)n++});document.querySelector('#count').textContent=`顯示 ${n}／650 列`})</script></main></html>'''
html_path=root/'動能基準研究_互動箱形圖_2026-10-13.html'
assert html_page.count('class="point"')==3900
assert len(rows)==650
html_path.write_text(html_page,encoding='utf-8')
print('HTML verified: 3,900 circles with individual tooltips and 650 detail rows.')
print('Charts saved:',png_path.name,html_path.name)


## 歷年競賽季節比較
先核對每年 10/27–11/27 交易日數，以及從 10/27 當日或之後首個交易日開始的固定 24 日窗口。2023 年缺少 60 日暖機資料，2026 年競賽尚未開始。歷年日曆範圍未必恰好有 24 個交易日，因此「固定日期」和「固定持有 24 日」需區分。


In [ ]:
%whos dict
seasonal_probe=[]
for year in (2023,2024,2025,2026):
    lo=datetime.date(year,10,27); hi=datetime.date(year,11,27)
    inside=[t for t,d in enumerate(dates) if lo<=d<=hi]
    possible=[t for t,d in enumerate(dates) if d>=lo and d.year==year]
    anchor=possible[0] if possible else None
    start_map={int(s):i for i,s in enumerate(window_starts)}
    anchor_i=start_map.get(anchor)
    nearby=[i for i,s in enumerate(window_starts) if anchor is not None and abs(int(s)-anchor)<=5]
    entry={'year':year,'calendar_sessions':len(inside),'anchor_start':str(dates[anchor]) if anchor is not None else None,'anchor_end_24':str(dates[anchor+23]) if anchor is not None and anchor+23<len(dates) else None,'anchor_available':anchor_i is not None,'nearby_n':len(nearby),'nearby_common_valid':sum(bool(common_valid[i]) for i in nearby),'anchor_common_valid':bool(common_valid[anchor_i]) if anchor_i is not None else None}
    seasonal_probe.append(entry)
    print(entry)
    if anchor_i is not None:
        print('Anchor results:',{k:(round(backtest_results[k][anchor_i]['return']*100,3),backtest_results[k][anchor_i]['invalid']) for k in boxplot_keys})


## 同一張圖的兩種時間對齊
上方：嚴格使用各年 10/27–11/27 內的交易日；2024 年实际起始為 10/28，共 22 日，2025 年為 10/27 起共 24 日。每年每策略只有一點，不適合畫箱形圖；天數不同，僅作曆日季節觀察。
下方：每年以 10/27 當日或之後首個交易日為錨點，起始日前後各 5 個交易日，共 11 個固定 24 日窗口，全部點保留。箱形圖僅採六策略共同可判定窗口（2024 年 8 點，2025 年 4 點），灰色為排除窗口；菱形為錨點窗口。2024 年錨點持有至 11/29，附近窗口可能延伸至 12 月，因此下圖是季節附近的日期敏感性檢查，不是嚴格競賽日期內的窗口。
2023 年缺少 60 日暖機資料，2026 年尚未發生，均不畫虛構結果。2025 年參考組資料不可判定，嚴格日期的該點標 X；其餘策略的可判定單點保留，但不能做完整六組配對排名。


In [ ]:
from matplotlib.lines import Line2D
seasonal_years=[2024,2025]
seasonal_colors=['#2563eb','#ea580c']
seasonal_indices={}
seasonal_anchors={}
calendar_results={}
for year in seasonal_years:
    lo,hi=datetime.date(year,10,27),datetime.date(year,11,27)
    inside=[t for t,d in enumerate(dates) if lo<=d<=hi]
    anchor=inside[0]
    seasonal_anchors[year]=anchor
    seasonal_indices[year]=np.array([i for i,s in enumerate(window_starts) if abs(int(s)-anchor)<=5])
    calendar_results[year]={k:simulate(anchor,k,horizon=len(inside)) for k in boxplot_keys}
    assert len(seasonal_indices[year])==11
    assert all(len(r['path'])==len(inside)+1 for r in calendar_results[year].values())
seasonal_fig=plt.figure(figsize=(16,15),layout='constrained')
gs=seasonal_fig.add_gridspec(3,3,height_ratios=[1.05,1,1])
calendar_ax=seasonal_fig.add_subplot(gs[0,:])
for yj,year in enumerate(seasonal_years):
    for j,k in enumerate(boxplot_keys):
        r=calendar_results[year][k]
        x=j+1+(-.14 if yj==0 else .14);v=r['return']*100
        calendar_ax.scatter(x,v,color=seasonal_colors[yj] if not r['invalid'] else '#94a3b8',marker='o' if not r['invalid'] else 'x',s=90,zorder=3)
        calendar_ax.annotate(f'{v:.2f}%'+(' 暫算' if r['invalid'] else ''),(x,v),xytext=(0,10 if yj else -20),textcoords='offset points',ha='center',fontsize=10)
calendar_ax.axhline(0,color='#64748b',linestyle='--',linewidth=.8)
calendar_ax.set_xticks(range(1,7),short_names)
calendar_ax.set_xlim(.5,6.5);calendar_ax.set_ylabel('該日期範圍報酬（%）')
calendar_ax.set_title('嚴格日期：各年 10/27～11/27，每年每策略一點（2024：22 日；2025：24 日）',fontsize=15,pad=15)
calendar_values=[r['return']*100 for year in seasonal_years for r in calendar_results[year].values()]
calendar_ax.set_ylim(min(calendar_values)-1.4,max(calendar_values)+3.2)
calendar_ax.grid(axis='y',alpha=.2)
calendar_ax.legend(handles=[Line2D([],[],color=seasonal_colors[0],marker='o',ls='',label='2024 年：10/28～11/27（22 日）'),Line2D([],[],color=seasonal_colors[1],marker='o',ls='',label='2025 年：10/27～11/27（24 日）'),Line2D([],[],color='#94a3b8',marker='x',ls='',label='不可判定：暫算值')],loc='upper right',fontsize=10)
all_season_values=np.concatenate([metrics[k]['return'][seasonal_indices[y]]*100 for k in boxplot_keys for y in seasonal_years])
season_ylim=(min(0,float(all_season_values.min()))-1.5,float(all_season_values.max())+1.5)
seasonal_scatter_count=0
for j,k in enumerate(boxplot_keys):
    ax=seasonal_fig.add_subplot(gs[1+j//3,j%3])
    for yj,year in enumerate(seasonal_years):
        idx=seasonal_indices[year];valid=common_valid[idx];good=metrics[k]['return'][idx[valid]]*100
        box=ax.boxplot([good],positions=[yj+1],widths=.44,showfliers=False,patch_artist=True,manage_ticks=False,medianprops={'color':'#0f172a','linewidth':2})
        box['boxes'][0].set(facecolor=seasonal_colors[yj],alpha=.15)
        for i in idx:
            relative=int(window_starts[i])-seasonal_anchors[year]
            x=yj+1+relative*.042;v=metrics[k]['return'][i]*100;anchor=(relative==0)
            ax.scatter(x,v,s=75 if anchor else 30,color=seasonal_colors[yj] if common_valid[i] else '#94a3b8',marker='D' if anchor else ('o' if common_valid[i] else 'x'),zorder=4,alpha=.95)
            seasonal_scatter_count+=1
        ax.text(yj+1,season_ylim[0]+.3,f'納入 {int(valid.sum())}／11 點',ha='center',fontsize=10,color='#475569')
    ax.axhline(0,color='#64748b',linestyle='--',linewidth=.8)
    ax.set_xticks([1,2],['2024','2025']);ax.set_xlim(.5,2.5);ax.set_ylim(season_ylim)
    ax.set_title(short_names[j],fontsize=13);ax.set_ylabel('固定 24 日報酬（%）');ax.grid(axis='y',alpha=.2)
seasonal_fig.suptitle('競賽季節：歷年日期比較＋起始日前後 5 日的箱形圖\n下方每年 11 個窗口全數呈現；菱形＝錨點，灰色＝不納入箱形圖',fontsize=18)
assert seasonal_scatter_count==132
assert sum(len(ax.collections) for ax in seasonal_fig.axes)==144
seasonal_png=root/'動能基準研究_競賽季節_各年份箱形圖與散點_2026-10-13.png'
seasonal_fig.savefig(seasonal_png,dpi=180,facecolor='white')
plt.show()
print('Verified: 12 calendar-window points + 132 nearby 24-day-window points; no subsampling.')
calendar_lines=['|策略|2024：22 日|2025：24 日|','|---|---:|---:|']
for k in boxplot_keys:
    vals=[f"{calendar_results[y][k]['return']:.2%}"+('（不可判定，暫算）' if calendar_results[y][k]['invalid'] else '') for y in seasonal_years]
    calendar_lines.append('|'+labels[k]+'|'+'|'.join(vals)+'|')
calendar_table='\n'.join(calendar_lines)
display(Markdown(calendar_table))
seasonal_report=root/'動能基準研究_競賽季節_歷年比較說明_2026-10-13.md'
seasonal_report.write_text('# 競賽季節：歷年比較\n\n'+calendar_table+'\n\n'+'''比較嚴格日期範圍時，每年每策略只有一個結果，且 2024 為 22 日、2025 為 24 日，不能把這兩點當成充分樣本畫箱形圖或做顯著性檢定。2025 參考組不可判定，因此也不能完整計算當年的六組配對勝負。

圖的下半部改用固定 24 日、起始日前後各 5 交易日，共每年 11 個窗口，全部點保留。2024 共同有效 8 點，2025 共同有效 4 點；菱形為原定起始日錨點，灰色是排除窗口。所有窗口高度重疊，不能把 22 個窗口視為 22 次獨立比賽。

2024 錨點：10/28～11/29；2025 錨點：10/27～11/27。附近窗口會超出嚴格日期範圍。2023 缺少 60 日暖機資料，2026 競賽尚未開始，均無可用年度結果。

同季節比較更貼近問題，但目前只有兩個可回測年份，且資料缺漏影響配對。建議將這張圖作為原本全年 24 日回測的敏感性檢查，不能只根據這兩年選出競賽策略。下一步優先補齊更早年份與公司行動資料；全年比較、同季節比較與不同市況比較應互相核對。

所有結果已扣手續費及賣出稅；期末依收盤與應收估值，不清倉。沿用原 Notebook 專用 kernel，未重新載入 DB 或清除 kernel。程式與圖表已保存至 動能基準研究_24日回測_2026-10-13.ipynb。
''',encoding='utf-8')
print('Saved:',seasonal_png.name,seasonal_report.name)


## 預設視圖與比賽期間快捷鍵
預設顯示全部共同有效窗口，不需操作篩選。所有篩選收在「調整篩選」。主畫面「比賽期間」快捷鍵會重設其他條件，選取各年開始日介於 10/27–11/27 的 24 日窗口；結束日可能超出範圍，因此不是要求整段持有都在競賽日期內。可一鍵切回「全部窗口」，也可展開按年份、月份或日期細調。
全部散點、中位數桃紅線與 0% 基準線保留，排除資料與明細預設收合。HTML 模板直接嵌入下方 cell。


修正空白預覽：模板檔原先未填入資料；其預覽副本已補入完整資料。正式 HTML 與 Notebook 模板預先嵌入全部 3,900 個散點，初始圖表不必等 JavaScript 執行後才出現；JavaScript 仍負責互動篩選。已驗證預載點數及互動篩選；瀏覽器工具不允許直接讀取 file URL，因此未聲稱完成實際分頁視覺檢查。

比較尺度修正：所有快捷鍵與篩選共用全部 650 個窗口、六策略計算出的固定 Y 軸範圍，包含排除窗口的暫算值。有效圖與排除圖共用座標尺度，0% 線位置與策略排列固定；不隨篩選自動縮放。

修正空白預覽：模板檔原先未填入資料；其預覽副本已補入完整資料。正式 HTML 與 Notebook 模板預先嵌入全部 3,900 個散點，初始圖表不必等 JavaScript 執行後才出現；JavaScript 仍負責互動篩選。已驗證預載點數及互動篩選；瀏覽器工具不允許直接讀取 file URL，因此未聲稱完成實際分頁視覺檢查。

滑鼠提示修正：散點移除 SVG 原生 title，僅保留自訂浮動資訊框，避免重複提示。

頁面最後新增六策略白話說明：選股訊號與比例、調整頻率、比較目的及共同帳務設定。已回查選股與成交引擎；明確註明沒有新聞或 LLM 逐日投資判斷，選股分數比例不等於持股權重。

策略名稱統一：圖表橫軸、滑鼠提示、明細欄名與底部解說全部使用相同「代號｜名稱」。

In [ ]:
strategy_display_names={'REF': 'REF｜流動性等權', 'A': 'A｜每週動能', 'B': 'B｜動能＋法人', 'C': 'C｜動能＋低波動', 'AH': 'AH｜動能持有 24 日', 'BH': 'BH｜動能法人持有 24 日'}
filter_payload={'strategies':[{'key':k,'name':strategy_display_names[k],'short':strategy_display_names[k],'color':boxplot_colors[j]} for j,k in enumerate(boxplot_keys)],'windows':[]}
for i,start in enumerate(window_starts):
    filter_payload['windows'].append({'id':i+1,'start':str(dates[int(start)]),'end':str(dates[int(start)+HORIZON-1]),'common':bool(common_valid[i]),'results':[{'return':float(backtest_results[k][i]['return']),'cost':float(backtest_results[k][i]['cost']),'invalid':bool(backtest_results[k][i]['invalid']),'jitter':float(boxplot_jitter[k][i])} for k in boxplot_keys]})
assert len(filter_payload['windows'])==650
filter_template='<!doctype html><html lang="zh-Hant"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>動能基準研究｜24 日回測</title>\n<style>\nbody{font-family:"Microsoft JhengHei",sans-serif;margin:24px;background:#f8fafc;color:#0f172a}main{max-width:1400px;margin:auto}section{background:white;border:1px solid #e2e8f0;border-radius:12px;padding:20px;margin:18px 0}.note{line-height:1.7;color:#475569}.filters{display:flex;flex-wrap:wrap;gap:16px;align-items:flex-end}.field{display:flex;flex-direction:column;gap:6px;font-size:14px}input,select,button{font:inherit;padding:8px;border:1px solid #cbd5e1;border-radius:6px;background:white}button{cursor:pointer}.months{display:flex;flex-wrap:wrap;gap:6px;margin-top:12px}.months label{padding:7px 9px;border:1px solid #cbd5e1;border-radius:6px}.months input{padding:0}.months label:has(input:checked){background:#eff6ff;border-color:#93c5fd}#status{font-weight:bold}#error{color:#b91c1c}.chart{overflow:auto}svg{width:100%;min-width:930px}svg text{font-size:14px;fill:#334155}.point:hover{r:6;fill-opacity:1;stroke:#111827;stroke-width:1.2}.median-label{font-weight:bold}table{border-collapse:collapse;font-size:13px;width:100%}th,td{padding:9px;border-bottom:1px solid #e2e8f0;white-space:nowrap}th{position:sticky;top:0;background:#e2e8f0}#tooltip{position:fixed;z-index:10;background:#0f172a;color:white;padding:12px;border-radius:8px;font-size:13px;line-height:1.6;white-space:pre-line;pointer-events:none;max-width:370px}.table-wrap{overflow:auto;max-height:650px}.swatch{display:inline-block;width:26px;vertical-align:middle;border-top:3px solid #be185d}.zero-swatch{border-top:2px dashed #111827}\nsummary{cursor:pointer;font-weight:600;padding:6px 0}.toolbar{display:flex;flex-wrap:wrap;align-items:center;gap:12px}.toolbar .field{flex-direction:row;align-items:center}.toolbar select{min-width:130px}.secondary{color:#64748b}section{margin:14px 0}.note{font-size:14px}.range{margin-top:12px}.range .filters{padding-top:12px}h1{margin-bottom:8px}.legend{font-size:14px;color:#475569}button[aria-pressed=true]{background:#eff6ff;border-color:#2563eb;color:#1d4ed8;font-weight:bold}\n.strategy-cards{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px}.strategy-cards article{border:1px solid #e2e8f0;border-radius:8px;padding:16px}.strategy-cards h3{margin:0 0 10px;font-size:17px}.strategy-cards p{line-height:1.7;margin:8px 0}@media(max-width:760px){.strategy-cards{grid-template-columns:1fr}}</style><main><h1>動能基準研究｜24 日回測</h1><p class="note">每個點是一段 24 交易日的回測，報酬已扣交易費用。</p>\n<section><div class="toolbar"><button id="competition" aria-pressed="false">比賽期間\u300010/27～11/27</button><button id="reset" aria-pressed="true">全部窗口</button></div>\n<details class="range" id="rangeDetails"><summary>調整篩選</summary><div class="filters">\n<label class="field">期間<select id="scope"><option value="all">全部窗口</option><option value="competition">各年比賽期間</option></select></label>\n<label class="field">年份<select id="year"><option value="">全部年份</option></select></label>\n<label class="field">月份<select id="month"><option value="">全部月份</option></select></label>\n<label class="field">開始日期：從<input id="from" type="date"></label><label class="field">開始日期：至<input id="to" type="date"></label></div></details>\n<p class="note" id="rangeNote">預設顯示全部窗口；需要時再調整。日期依窗口開始日判斷。</p><p id="error" role="alert"></p><p id="status" aria-live="polite"></p></section>\n<section><p class="legend"><i class="swatch"></i> <strong style="color:#be185d">中位數</strong>\u3000<i class="swatch zero-swatch"></i> 0% 基準線\u3000<span class="secondary">固定座標尺度</span></p><div class="chart" id="validChart"><svg viewBox="0 0 1250 620" role="img" aria-label="有效窗口箱形圖"><line x1="82" y1="500.88235294117646" x2="1225" y2="500.88235294117646" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="505.88235294117646" text-anchor="middle" >-30.0%</text><line x1="82" y1="442.6470588235294" x2="1225" y2="442.6470588235294" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="447.6470588235294" text-anchor="middle" >-20.0%</text><line x1="82" y1="384.4117647058824" x2="1225" y2="384.4117647058824" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="389.4117647058824" text-anchor="middle" >-10.0%</text><line x1="82" y1="267.94117647058823" x2="1225" y2="267.94117647058823" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="272.94117647058823" text-anchor="middle" >10.0%</text><line x1="82" y1="209.7058823529412" x2="1225" y2="209.7058823529412" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="214.7058823529412" text-anchor="middle" >20.0%</text><line x1="82" y1="151.47058823529412" x2="1225" y2="151.47058823529412" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="156.47058823529412" text-anchor="middle" >30.0%</text><line x1="82" y1="93.23529411764706" x2="1225" y2="93.23529411764706" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="98.23529411764706" text-anchor="middle" >40.0%</text><line x1="82" y1="35" x2="1225" y2="35" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="40" text-anchor="middle" >50.0%</text><line x1="82" y1="326.1764705882353" x2="1225" y2="326.1764705882353" stroke="#111827" stroke-width="2" class="zero-baseline" stroke-dasharray="7 5"/><text x="50" y="331.1764705882353" text-anchor="middle" font-weight="bold">0%</text><line x1="177.25" y1="434.39264417262837" x2="177.25" y2="162.62109223544914" stroke="#64748b" stroke-width="2" /><rect x="139.25" y="263.31243516733014" width="76" height="68.61055389810247" fill="#64748b" fill-opacity=".12" stroke="#64748b"/><line x1="139.25" y1="434.39264417262837" x2="215.25" y2="434.39264417262837" stroke="#64748b" stroke-width="2" /><line x1="139.25" y1="162.62109223544914" x2="215.25" y2="162.62109223544914" stroke="#64748b" stroke-width="2" /><circle class="point" data-window="18" data-strategy="0" cx="168.899506287621" cy="283.61768408858165" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="19" data-strategy="0" cx="194.78676145990804" cy="289.1570337561905" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="20" data-strategy="0" cx="219.36746405903082" cy="289.6288277095835" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="21" data-strategy="0" cx="145.62507701007752" cy="277.89385801727065" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="22" data-strategy="0" cx="203.7667022015911" cy="277.13109097410916" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="23" data-strategy="0" cx="160.496310948017" cy="293.5254876611841" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="24" data-strategy="0" cx="176.3946287439976" cy="283.43672839327803" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="25" data-strategy="0" cx="158.68138146259335" cy="283.119749274663" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="26" data-strategy="0" cx="181.32967537849686" cy="285.5264905760604" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="27" data-strategy="0" cx="177.3135740215259" cy="287.4589934346069" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="28" data-strategy="0" cx="165.0302183050106" cy="307.80148458874186" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="29" data-strategy="0" cx="173.8896750555297" cy="296.7171186466229" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="30" data-strategy="0" cx="177.51079842240418" cy="285.7064608638673" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="31" data-strategy="0" cx="217.012494119541" cy="281.50340423662755" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="32" data-strategy="0" cx="189.58544876865344" cy="293.84314983400134" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="33" data-strategy="0" cx="138.39524513484372" cy="300.9801812621495" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="34" data-strategy="0" cx="173.3798642455358" cy="303.96499345669844" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="35" data-strategy="0" cx="180.88589114781414" cy="305.9738630774219" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="36" data-strategy="0" cx="138.93269152507227" cy="291.3000604623648" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="37" data-strategy="0" cx="193.78552199261406" cy="289.8397266690011" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="38" data-strategy="0" cx="214.71100657768042" cy="295.0644008445426" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="40" data-strategy="0" cx="185.7636080783534" cy="299.3817720552228" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="41" data-strategy="0" cx="173.83244257786384" cy="309.60395730378514" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="42" data-strategy="0" cx="158.34908972757574" cy="311.56253907818814" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="43" data-strategy="0" cx="140.36051342772828" cy="298.11073017246014" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="45" data-strategy="0" cx="177.74836868669206" cy="339.65075754267355" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="46" data-strategy="0" cx="178.8900520429293" cy="321.68591496618484" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="47" data-strategy="0" cx="152.77067067399983" cy="307.3822447388357" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="48" data-strategy="0" cx="196.26692971043477" cy="333.5220366771168" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="50" data-strategy="0" cx="200.87629666950843" cy="361.10276608009383" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="51" data-strategy="0" cx="134.67081319462545" cy="328.9251361033262" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="52" data-strategy="0" cx="155.6309108644416" cy="345.6023378984744" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="53" data-strategy="0" cx="143.91503553812927" cy="337.5978693006582" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="55" data-strategy="0" cx="186.18023661319626" cy="326.8317940117599" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="56" data-strategy="0" cx="204.13658383162056" cy="326.8994947463274" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="57" data-strategy="0" cx="214.88483061213736" cy="324.07445712945616" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="58" data-strategy="0" cx="150.64427874581938" cy="330.62261325205486" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="60" data-strategy="0" cx="146.35169849581018" cy="327.3273567814638" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="61" data-strategy="0" cx="158.1147040144694" cy="334.1566340342419" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="62" data-strategy="0" cx="202.93318835964962" cy="343.9274598577671" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="63" data-strategy="0" cx="137.7360228591035" cy="346.51103959989416" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="64" data-strategy="0" cx="207.80576050544784" cy="337.5673857642973" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="65" data-strategy="0" cx="150.70559947599116" cy="328.222687930489" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="66" data-strategy="0" cx="219.76279675930274" cy="326.6409914003121" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="68" data-strategy="0" cx="159.237583994384" cy="299.4078418588408" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="69" data-strategy="0" cx="186.6037914345816" cy="305.8409854756482" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="71" data-strategy="0" cx="204.75415013886771" cy="292.43775701257414" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="73" data-strategy="0" cx="182.576004353015" cy="249.1331983326948" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="74" data-strategy="0" cx="135.96075409330965" cy="269.65127720229265" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="76" data-strategy="0" cx="182.29416446836998" cy="294.65327773071033" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="78" data-strategy="0" cx="216.05390178765668" cy="308.0243037891323" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="79" data-strategy="0" cx="151.89649686516907" cy="301.33521721282585" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="81" data-strategy="0" cx="154.9722521777171" cy="300.8795364428137" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="84" data-strategy="0" cx="166.27889623895854" cy="298.9636981694871" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="86" data-strategy="0" cx="138.69617906714197" cy="279.288137313069" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="89" data-strategy="0" cx="154.23688454422074" cy="288.87689783776364" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="91" data-strategy="0" cx="203.94163281193423" cy="280.007770720688" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="92" data-strategy="0" cx="205.0072587825566" cy="289.0362479068592" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="94" data-strategy="0" cx="166.1558499047268" cy="293.998545031152" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="95" data-strategy="0" cx="142.10713339253746" cy="302.25166599555945" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="97" data-strategy="0" cx="208.6550077468556" cy="319.9261828995267" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="99" data-strategy="0" cx="200.43278477615294" cy="305.6764989009861" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="100" data-strategy="0" cx="171.02087253020463" cy="292.2494328192871" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="102" data-strategy="0" cx="155.62753275840666" cy="296.20484510946045" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="104" data-strategy="0" cx="134.13916918884388" cy="296.46594585354853" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="105" data-strategy="0" cx="144.7718790184041" cy="294.1902987314808" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="107" data-strategy="0" cx="182.6181258510166" cy="305.1042826668178" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="108" data-strategy="0" cx="188.3107837412617" cy="316.5632261056709" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="109" data-strategy="0" cx="205.26440212347524" cy="328.2028628692988" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="110" data-strategy="0" cx="167.40733767437283" cy="338.43446032746056" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="112" data-strategy="0" cx="140.67748323139259" cy="375.13295407797824" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="113" data-strategy="0" cx="177.68476907766475" cy="364.6896808249101" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="168" data-strategy="0" cx="170.21647034223508" cy="284.48302707071707" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="170" data-strategy="0" cx="219.46182637892372" cy="283.1755523163225" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="171" data-strategy="0" cx="184.9684081433286" cy="292.04824096326684" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="172" data-strategy="0" cx="200.69593407865509" cy="298.11054631478805" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="173" data-strategy="0" cx="199.39293287609848" cy="291.81586851488765" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="175" data-strategy="0" cx="219.7304829747394" cy="311.86141333674016" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="176" data-strategy="0" cx="205.61664414021868" cy="324.9540350986451" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="177" data-strategy="0" cx="215.17114490260047" cy="332.2658145189334" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="178" data-strategy="0" cx="172.69674421261502" cy="324.05766112319327" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="179" data-strategy="0" cx="144.7026915983909" cy="335.92517097106656" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="180" data-strategy="0" cx="143.72582572770872" cy="337.17997101345884" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="181" data-strategy="0" cx="140.7227660666947" cy="333.1263814319531" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="182" data-strategy="0" cx="156.00143791101914" cy="311.8547598825706" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="183" data-strategy="0" cx="206.80302854461377" cy="303.30524593307956" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="184" data-strategy="0" cx="185.54511391359296" cy="292.60239870567773" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="186" data-strategy="0" cx="181.82123082308556" cy="308.30865200198775" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="187" data-strategy="0" cx="219.94559013156828" cy="316.3170520679984" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="188" data-strategy="0" cx="140.26498075130218" cy="318.2529709656395" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="189" data-strategy="0" cx="213.27953766250818" cy="325.14978058714945" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="191" data-strategy="0" cx="159.7876713314281" cy="327.7820089848556" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="192" data-strategy="0" cx="190.36624297942257" cy="328.6922320135726" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="193" data-strategy="0" cx="137.2982686497773" cy="328.32213868741854" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="194" data-strategy="0" cx="210.48755603318125" cy="328.61474099632056" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="196" data-strategy="0" cx="162.8071760488819" cy="333.41844749116274" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="197" data-strategy="0" cx="137.34494398499322" cy="339.8504059629167" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="198" data-strategy="0" cx="219.87757291428466" cy="345.0773294940933" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="199" data-strategy="0" cx="163.0820503568229" cy="333.55399307686673" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="201" data-strategy="0" cx="198.62722569228748" cy="308.321926499773" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="203" data-strategy="0" cx="140.23671416267683" cy="295.4123572329185" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="204" data-strategy="0" cx="140.42963531111525" cy="300.76388206850856" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="205" data-strategy="0" cx="176.868583350259" cy="308.9971383245526" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="206" data-strategy="0" cx="178.3859485164521" cy="325.4979590018669" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="208" data-strategy="0" cx="185.84405444658216" cy="332.44431326266" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="209" data-strategy="0" cx="180.95109034285662" cy="322.9445353243056" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="210" data-strategy="0" cx="188.30183950138434" cy="328.5766048612975" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="211" data-strategy="0" cx="201.95167993941993" cy="315.23221657054347" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="213" data-strategy="0" cx="159.63053960889192" cy="299.36770905426727" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="214" data-strategy="0" cx="215.99874460919125" cy="314.5326306108034" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="215" data-strategy="0" cx="180.15473332478203" cy="312.1612625460915" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="216" data-strategy="0" cx="193.38551956264564" cy="312.3826456017325" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="218" data-strategy="0" cx="176.0998792014765" cy="318.06367977874726" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="219" data-strategy="0" cx="184.61969995167817" cy="307.86964278784023" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="220" data-strategy="0" cx="162.97688229773462" cy="302.4055608502169" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="221" data-strategy="0" cx="136.12923266603332" cy="297.1049123742153" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="223" data-strategy="0" cx="142.08125677227318" cy="309.0719449358754" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="224" data-strategy="0" cx="159.66283595930037" cy="306.3910509152736" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="225" data-strategy="0" cx="210.46665353411333" cy="308.55555631673707" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="226" data-strategy="0" cx="186.7097718327384" cy="317.19309267813605" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="227" data-strategy="0" cx="146.14585979729813" cy="334.9810027384187" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="228" data-strategy="0" cx="157.2513016347468" cy="342.4046153073114" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="229" data-strategy="0" cx="211.60752672527636" cy="353.80399588600585" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="230" data-strategy="0" cx="210.43835641301712" cy="343.31508950282927" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="231" data-strategy="0" cx="179.9833316713541" cy="355.8462202917712" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="232" data-strategy="0" cx="136.67017211592838" cy="348.9788065613534" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="233" data-strategy="0" cx="189.5304992446682" cy="342.16928158905984" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="234" data-strategy="0" cx="180.48799097615125" cy="336.9424045580747" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="235" data-strategy="0" cx="166.53833451088005" cy="335.3800501857122" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="236" data-strategy="0" cx="199.34241475556436" cy="324.50994988285896" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="237" data-strategy="0" cx="192.8485211606974" cy="352.57608598073574" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="238" data-strategy="0" cx="147.4312338723691" cy="360.29469673218813" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="239" data-strategy="0" cx="198.80048121812837" cy="345.28120700050556" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="240" data-strategy="0" cx="162.08287863262623" cy="341.987197988336" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="241" data-strategy="0" cx="142.85002708510373" cy="331.2658519118465" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="242" data-strategy="0" cx="199.60238256726814" cy="342.0744444147825" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="243" data-strategy="0" cx="177.50362177637624" cy="331.58016361193165" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="244" data-strategy="0" cx="198.81979929721894" cy="333.44905070102493" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="245" data-strategy="0" cx="187.77392055818788" cy="330.9766466954352" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="246" data-strategy="0" cx="178.15855369421513" cy="334.2990675326322" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="247" data-strategy="0" cx="217.15458742193104" cy="333.46213235384835" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="248" data-strategy="0" cx="150.79571104150693" cy="333.700486921476" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="249" data-strategy="0" cx="137.74697769226995" cy="337.8226084290472" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="250" data-strategy="0" cx="162.39091914283517" cy="333.5436713196486" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="251" data-strategy="0" cx="145.69614439353023" cy="321.14459993952784" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="252" data-strategy="0" cx="139.82360794638242" cy="304.32193703515804" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="253" data-strategy="0" cx="209.8066653383081" cy="313.268718693838" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="254" data-strategy="0" cx="216.6738195378039" cy="305.0901752913193" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="255" data-strategy="0" cx="197.07973417995208" cy="328.3538508057675" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="256" data-strategy="0" cx="149.2326470833863" cy="339.7144742750302" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="257" data-strategy="0" cx="211.0887551154805" cy="345.18239398405103" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="258" data-strategy="0" cx="184.05083492278737" cy="343.7695951235342" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="259" data-strategy="0" cx="145.46412871106816" cy="346.6544935354379" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="261" data-strategy="0" cx="188.70872740198047" cy="322.75895444134215" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="262" data-strategy="0" cx="141.8819916355559" cy="347.7801492025295" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="263" data-strategy="0" cx="201.38593285126458" cy="348.52346599255833" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="264" data-strategy="0" cx="200.6972042495391" cy="359.34330888927553" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="265" data-strategy="0" cx="139.10228462412147" cy="359.11486022053623" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="266" data-strategy="0" cx="194.94874941376807" cy="361.4884489660626" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="267" data-strategy="0" cx="183.4515535680475" cy="350.3218958236758" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="268" data-strategy="0" cx="137.18352797872555" cy="368.6483762599843" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="269" data-strategy="0" cx="190.38230074616018" cy="354.69870184864556" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="270" data-strategy="0" cx="169.6386248332842" cy="358.16926425156663" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="271" data-strategy="0" cx="173.35637204173665" cy="370.53752739629863" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="272" data-strategy="0" cx="167.91539078647318" cy="363.6005988417788" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="273" data-strategy="0" cx="142.4166620298205" cy="358.3927688351085" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="274" data-strategy="0" cx="218.4601668751103" cy="369.03767344439126" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="275" data-strategy="0" cx="139.24109187892407" cy="380.0892696590594" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="276" data-strategy="0" cx="191.66400757166048" cy="405.75227857140965" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="277" data-strategy="0" cx="196.85497908823524" cy="400.8641068907163" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="278" data-strategy="0" cx="209.37654287075696" cy="394.1233213501241" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="279" data-strategy="0" cx="208.40582361968325" cy="424.8063154668234" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="280" data-strategy="0" cx="192.60991817826195" cy="455.7292386845872" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="281" data-strategy="0" cx="191.36364470353195" cy="496.34354414097146" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="282" data-strategy="0" cx="148.23251029941525" cy="457.7813801431252" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="283" data-strategy="0" cx="145.4360521337319" cy="433.38511536251275" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="284" data-strategy="0" cx="147.82266529383256" cy="416.7375723344344" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="285" data-strategy="0" cx="191.79421659863831" cy="385.7258906313512" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="286" data-strategy="0" cx="177.25140854179324" cy="414.3459711584394" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="287" data-strategy="0" cx="182.97062874700939" cy="415.08334557324383" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="288" data-strategy="0" cx="199.9528244395176" cy="403.610259264428" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="289" data-strategy="0" cx="136.59955956840605" cy="418.4126569318702" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="290" data-strategy="0" cx="179.4488337609773" cy="434.39264417262837" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="291" data-strategy="0" cx="173.43139482753264" cy="407.9118930580899" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="292" data-strategy="0" cx="159.3274130908734" cy="419.61990868756425" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="293" data-strategy="0" cx="177.94970489954824" cy="401.7406074103533" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="294" data-strategy="0" cx="146.15010006443293" cy="398.52071604678974" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="295" data-strategy="0" cx="175.12827167167666" cy="388.82289443211397" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="296" data-strategy="0" cx="146.70769441195822" cy="399.1164298225637" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="297" data-strategy="0" cx="180.92224667913834" cy="370.80643093741133" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="298" data-strategy="0" cx="150.43205892726817" cy="369.64518324306994" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="299" data-strategy="0" cx="190.16940688536258" cy="335.07222741405405" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="300" data-strategy="0" cx="150.46617484521164" cy="338.63703661512096" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="301" data-strategy="0" cx="182.89758855373276" cy="332.58236738336706" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="304" data-strategy="0" cx="163.40021747301617" cy="166.34077333655998" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="305" data-strategy="0" cx="162.3491947584534" cy="162.62109223544914" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="306" data-strategy="0" cx="144.09889908253706" cy="190.63299002259365" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="307" data-strategy="0" cx="161.5467718498465" cy="236.19400482645506" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="308" data-strategy="0" cx="198.52512124774037" cy="253.01330114762814" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="309" data-strategy="0" cx="182.38207464260344" cy="245.98151797887493" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="310" data-strategy="0" cx="179.47964818739553" cy="224.15915607936134" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="311" data-strategy="0" cx="208.4702792738673" cy="226.04423447331968" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="312" data-strategy="0" cx="219.90556684595924" cy="225.69766377787653" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="313" data-strategy="0" cx="192.2926014929935" cy="223.24942236217498" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="314" data-strategy="0" cx="152.95950867009213" cy="242.58583223266226" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="315" data-strategy="0" cx="212.25325745790582" cy="240.32224873228742" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="316" data-strategy="0" cx="220.42219229617902" cy="252.66499241910122" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="317" data-strategy="0" cx="196.9432174503627" cy="262.87812719772626" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="318" data-strategy="0" cx="180.21318155522516" cy="257.6151996569995" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="319" data-strategy="0" cx="181.67942641482495" cy="243.97073243969172" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="320" data-strategy="0" cx="140.0565983527453" cy="259.1738669493301" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="321" data-strategy="0" cx="145.49090548788809" cy="257.3142011565909" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="322" data-strategy="0" cx="189.29184045236627" cy="255.15305175303405" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="323" data-strategy="0" cx="198.61530221494536" cy="243.82258138454446" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="324" data-strategy="0" cx="161.22506201542967" cy="238.38631806700448" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="325" data-strategy="0" cx="179.1976413704105" cy="240.27848160824036" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="326" data-strategy="0" cx="147.6740978279033" cy="250.58250832865698" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="327" data-strategy="0" cx="188.43474132776086" cy="265.09126311219933" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="328" data-strategy="0" cx="211.4286412356974" cy="276.493164818991" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="329" data-strategy="0" cx="206.56365153795915" cy="277.13461345481153" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="330" data-strategy="0" cx="135.07948340071263" cy="287.7303222724138" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="331" data-strategy="0" cx="218.7651071044154" cy="288.0509769195685" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="332" data-strategy="0" cx="172.2773121998539" cy="288.0560937041229" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="333" data-strategy="0" cx="214.02310465563144" cy="284.68272963092284" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="334" data-strategy="0" cx="137.89751621661983" cy="284.68714765946254" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="335" data-strategy="0" cx="174.9952548126778" cy="289.62387852613347" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="336" data-strategy="0" cx="212.7065693540876" cy="284.4883970803983" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="337" data-strategy="0" cx="215.69858781589363" cy="283.6669216705487" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="338" data-strategy="0" cx="196.7583748412798" cy="278.20760003319026" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="339" data-strategy="0" cx="181.36291829926677" cy="279.55511797464465" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="340" data-strategy="0" cx="204.8771470296713" cy="267.23655509498803" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="341" data-strategy="0" cx="199.63226908980303" cy="277.7072829806949" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="342" data-strategy="0" cx="170.19778342336755" cy="299.48050416038" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="343" data-strategy="0" cx="158.0607238129963" cy="296.1253418475401" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="344" data-strategy="0" cx="146.5472159207883" cy="280.4060863284574" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="345" data-strategy="0" cx="147.50174440688716" cy="274.4905896882263" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="346" data-strategy="0" cx="137.51613618639624" cy="288.786993020817" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="347" data-strategy="0" cx="162.53431231614977" cy="299.94589349535335" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="348" data-strategy="0" cx="178.10733289175135" cy="296.51063540315096" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="349" data-strategy="0" cx="205.47796167042208" cy="285.1712232592801" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="395" data-strategy="0" cx="193.54502221518774" cy="301.6773049108877" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="396" data-strategy="0" cx="158.26335251642058" cy="294.3196824383385" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="397" data-strategy="0" cx="217.34000522239046" cy="245.15771013119277" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="398" data-strategy="0" cx="164.0057939527502" cy="264.4716621155053" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="399" data-strategy="0" cx="177.72835570937667" cy="281.45886000144975" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="400" data-strategy="0" cx="135.13372224280462" cy="298.00781405546223" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="401" data-strategy="0" cx="163.4892157297083" cy="317.2776102563129" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="402" data-strategy="0" cx="215.29208109541042" cy="315.7350243323102" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="403" data-strategy="0" cx="178.2310842792333" cy="306.6381058000908" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="404" data-strategy="0" cx="191.33186819531315" cy="313.8839345702441" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="405" data-strategy="0" cx="184.20308346837993" cy="284.20464141296304" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="407" data-strategy="0" cx="144.414980499284" cy="261.8429437130251" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="408" data-strategy="0" cx="164.79461541891337" cy="263.07230324842135" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="409" data-strategy="0" cx="190.45269314980774" cy="281.42666666773374" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="410" data-strategy="0" cx="217.36581699011097" cy="311.12106964647035" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="412" data-strategy="0" cx="208.35697309113746" cy="296.8830738420685" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="413" data-strategy="0" cx="161.53937965217983" cy="305.50992508784253" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="414" data-strategy="0" cx="219.5341901112385" cy="300.23770875913385" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="415" data-strategy="0" cx="140.88170541834714" cy="282.0860615240399" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="417" data-strategy="0" cx="169.91734753778425" cy="277.39697723084396" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="418" data-strategy="0" cx="175.33238841127155" cy="268.0945926042436" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="419" data-strategy="0" cx="161.92762947588383" cy="267.326463938822" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="420" data-strategy="0" cx="204.4599698483667" cy="260.4394042802903" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="422" data-strategy="0" cx="143.01086387694767" cy="247.8360998232467" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="423" data-strategy="0" cx="190.92270018719063" cy="234.09756926707814" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="424" data-strategy="0" cx="194.08377360016118" cy="229.5715824200648" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="425" data-strategy="0" cx="203.91895467644858" cy="247.6222499519658" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="427" data-strategy="0" cx="203.2672156556198" cy="270.451770201631" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="428" data-strategy="0" cx="142.45399555553834" cy="244.8174349441277" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="429" data-strategy="0" cx="148.0008518434558" cy="260.12168120279966" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="430" data-strategy="0" cx="209.65667054496373" cy="252.85214322596497" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="431" data-strategy="0" cx="174.27474233221625" cy="246.27490314734038" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="443" data-strategy="0" cx="191.5999797665476" cy="307.4231859596048" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="444" data-strategy="0" cx="187.00600104865615" cy="319.6368458879724" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="445" data-strategy="0" cx="159.3217006818979" cy="327.56241016465356" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="446" data-strategy="0" cx="143.34575240956394" cy="336.7868900753425" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="447" data-strategy="0" cx="194.44871372821828" cy="324.65339353545306" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="448" data-strategy="0" cx="135.41525611049804" cy="297.50910328710813" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="449" data-strategy="0" cx="140.3593018197203" cy="313.41507352638405" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="450" data-strategy="0" cx="161.6624965955618" cy="302.1459694954778" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="451" data-strategy="0" cx="212.77958122118324" cy="320.0409021723234" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="452" data-strategy="0" cx="215.11058677158803" cy="326.1750642201994" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="453" data-strategy="0" cx="175.78610907295288" cy="333.07470762351187" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="454" data-strategy="0" cx="213.84007836369557" cy="347.35622618085796" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="455" data-strategy="0" cx="172.3125885609418" cy="327.60600935023405" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="456" data-strategy="0" cx="175.13524914138605" cy="337.12544375088567" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="457" data-strategy="0" cx="192.54345048436826" cy="313.7999436414498" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="458" data-strategy="0" cx="147.30255426836118" cy="284.0949981618147" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="459" data-strategy="0" cx="170.74797804127917" cy="302.25850852504783" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="460" data-strategy="0" cx="136.71955282891116" cy="256.5276818353743" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="461" data-strategy="0" cx="154.98250983852665" cy="249.34180388734094" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="462" data-strategy="0" cx="138.52176545427324" cy="256.3023411250887" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="463" data-strategy="0" cx="163.8532707979299" cy="259.79372075770107" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="465" data-strategy="0" cx="210.33150664807079" cy="268.27494180947895" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="466" data-strategy="0" cx="154.2095921745636" cy="254.83806915821344" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="467" data-strategy="0" cx="152.18546856673024" cy="234.5713783851247" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="468" data-strategy="0" cx="178.5391172298272" cy="227.64214619304326" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="470" data-strategy="0" cx="170.65042594515285" cy="241.90283088342326" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="471" data-strategy="0" cx="209.11863029611746" cy="248.68327283853716" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="472" data-strategy="0" cx="203.69893315937904" cy="260.33464244985225" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="475" data-strategy="0" cx="150.59084588184228" cy="237.5669152470127" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="476" data-strategy="0" cx="144.66618322426785" cy="227.88048326101466" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="477" data-strategy="0" cx="151.08825648357745" cy="207.56105858809227" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="480" data-strategy="0" cx="215.1367699970102" cy="210.7865508717648" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="504" data-strategy="0" cx="152.45364557824357" cy="318.47551047718537" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="505" data-strategy="0" cx="208.48739192271705" cy="290.68526534581923" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="506" data-strategy="0" cx="184.98710050387533" cy="309.37728389742546" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="507" data-strategy="0" cx="154.21041279904688" cy="306.8637691285325" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="508" data-strategy="0" cx="145.18824570410771" cy="292.1834204246568" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="509" data-strategy="0" cx="175.84115328535904" cy="263.82358792589866" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="510" data-strategy="0" cx="138.17094996758712" cy="260.36351896143844" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="511" data-strategy="0" cx="215.04992310333952" cy="263.8683110508609" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="512" data-strategy="0" cx="215.33550092062058" cy="263.55256708623904" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="513" data-strategy="0" cx="136.93207759306586" cy="265.893307334494" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="514" data-strategy="0" cx="186.7973544805837" cy="308.94628587418254" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="515" data-strategy="0" cx="152.3623143849121" cy="285.60264386888355" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="516" data-strategy="0" cx="188.41140515094656" cy="291.76547742770686" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="517" data-strategy="0" cx="181.1727579412095" cy="319.267933387653" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="518" data-strategy="0" cx="184.60003820184926" cy="336.67450245696466" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="519" data-strategy="0" cx="151.02431156249784" cy="383.4521480322973" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="520" data-strategy="0" cx="139.43558229633092" cy="357.5561385714806" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="521" data-strategy="0" cx="134.35898906938107" cy="365.37596788814574" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="522" data-strategy="0" cx="186.4544520965008" cy="343.8098470078284" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="523" data-strategy="0" cx="183.43313670788783" cy="267.7511304319944" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="524" data-strategy="0" cx="175.67038276369527" cy="280.36500845649607" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="525" data-strategy="0" cx="181.49621110644597" cy="266.6636273003778" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="527" data-strategy="0" cx="161.64544885521673" cy="226.97452287352976" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="528" data-strategy="0" cx="181.3217347157064" cy="250.853808834992" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="529" data-strategy="0" cx="162.5696836667205" cy="247.51297081553992" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="530" data-strategy="0" cx="158.22801050910937" cy="253.03974297508597" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="531" data-strategy="0" cx="173.45385508499382" cy="265.56617114557037" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="532" data-strategy="0" cx="156.605768597459" cy="252.4267790304534" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="533" data-strategy="0" cx="178.4351726555432" cy="253.96445565032553" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="535" data-strategy="0" cx="186.56525899329773" cy="233.96838983544524" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="536" data-strategy="0" cx="135.10375149238203" cy="198.00496454929123" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="537" data-strategy="0" cx="173.40157981529873" cy="170.53933384093736" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="538" data-strategy="0" cx="184.92362957606343" cy="172.86487238991364" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="539" data-strategy="0" cx="187.92157935285985" cy="186.60582460111584" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="540" data-strategy="0" cx="144.32779208832162" cy="130.70953135700586" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="541" data-strategy="0" cx="137.59548992097615" cy="115.29844512296725" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="542" data-strategy="0" cx="183.53316909605684" cy="83.28187028499683" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="543" data-strategy="0" cx="208.97534023471343" cy="84.09825239047441" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="544" data-strategy="0" cx="169.12141835060268" cy="128.6377529774256" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="545" data-strategy="0" cx="205.65946239639277" cy="83.30851957886657" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="546" data-strategy="0" cx="153.67979749767855" cy="119.22814136538189" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="547" data-strategy="0" cx="176.74068589701676" cy="159.43800230447596" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="548" data-strategy="0" cx="138.9537887633869" cy="144.3809680780975" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="549" data-strategy="0" cx="195.30181734883016" cy="183.95657139155693" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="550" data-strategy="0" cx="173.78918063619" cy="193.62584843554424" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="551" data-strategy="0" cx="215.3999348115825" cy="226.983242579888" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="552" data-strategy="0" cx="168.57865253118695" cy="219.0464842521515" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="553" data-strategy="0" cx="197.3396446457904" cy="188.88556473159275" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="554" data-strategy="0" cx="177.3453358400731" cy="159.80050948870255" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="555" data-strategy="0" cx="183.66976328331728" cy="138.19206830693696" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="556" data-strategy="0" cx="207.59407444985462" cy="143.54924297257006" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="557" data-strategy="0" cx="207.64707816468" cy="127.22482872430193" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="558" data-strategy="0" cx="183.6023126902664" cy="150.8399988622921" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="559" data-strategy="0" cx="209.96903497676917" cy="154.13989038703667" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="560" data-strategy="0" cx="139.19026018920516" cy="148.32124520141014" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="561" data-strategy="0" cx="184.15478340781664" cy="155.6002837542634" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="562" data-strategy="0" cx="165.9314051531681" cy="168.89797220548002" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="563" data-strategy="0" cx="216.98853807442876" cy="203.87564425803836" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="564" data-strategy="0" cx="218.0300105789618" cy="246.01916134637725" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="565" data-strategy="0" cx="179.13220062902778" cy="273.4445601645853" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="566" data-strategy="0" cx="158.84196022701587" cy="257.53660880296195" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="567" data-strategy="0" cx="197.39467808296845" cy="284.6980100369619" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="568" data-strategy="0" cx="141.87278629068723" cy="296.3088929416975" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="569" data-strategy="0" cx="165.3310197478338" cy="290.3185430943929" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="570" data-strategy="0" cx="189.5878625486859" cy="256.3446132274022" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="571" data-strategy="0" cx="218.06705443238803" cy="254.7866149756152" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="572" data-strategy="0" cx="149.11439692581732" cy="235.26127283581008" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="573" data-strategy="0" cx="145.74843538166556" cy="192.6993414862561" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="648" data-strategy="0" cx="167.9533721672897" cy="263.8500017175886" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="649" data-strategy="0" cx="153.02282119666884" cy="258.48308795435" r="3" fill="#64748b" fill-opacity=".65"></circle><circle class="point" data-window="650" data-strategy="0" cx="186.75788790462107" cy="251.35109958098008" r="3" fill="#64748b" fill-opacity=".65"></circle><line x1="133.25" y1="299.3817720552228" x2="221.25" y2="299.3817720552228" stroke="#be185d" stroke-width="3.5" class="median-line"/><text x="177.25" y="572" text-anchor="middle" >REF｜流動性等權</text><text x="177.25" y="596" text-anchor="middle" class="median-label" style="fill:#be185d">中位數 4.60%</text><line x1="367.75" y1="407.43317785724224" x2="367.75" y2="177.55148702456313" stroke="#2563eb" stroke-width="2" /><rect x="329.75" y="264.5266975812697" width="76" height="58.45154700028172" fill="#2563eb" fill-opacity=".12" stroke="#2563eb"/><line x1="329.75" y1="407.43317785724224" x2="405.75" y2="407.43317785724224" stroke="#2563eb" stroke-width="2" /><line x1="329.75" y1="177.55148702456313" x2="405.75" y2="177.55148702456313" stroke="#2563eb" stroke-width="2" /><circle class="point" data-window="18" data-strategy="1" cx="377.493669028804" cy="203.28787158247812" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="19" data-strategy="1" cx="371.2884241789267" cy="209.1892341203565" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="20" data-strategy="1" cx="386.26681352623217" cy="226.73377788303975" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="21" data-strategy="1" cx="387.9853066978248" cy="220.8156049422879" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="22" data-strategy="1" cx="395.2189811840831" cy="220.10542047659735" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="23" data-strategy="1" cx="374.08231864910704" cy="227.04348128765434" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="24" data-strategy="1" cx="406.51320386900807" cy="243.54377275962824" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="25" data-strategy="1" cx="325.01575518150986" cy="251.32416812747925" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="26" data-strategy="1" cx="382.1451667414925" cy="270.3873828123642" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="27" data-strategy="1" cx="366.5219452100699" cy="269.44720095132993" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="28" data-strategy="1" cx="328.82654416805025" cy="294.7847150013814" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="29" data-strategy="1" cx="330.20934222326173" cy="283.1024338231264" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="30" data-strategy="1" cx="341.3485117639006" cy="282.54190749734033" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="31" data-strategy="1" cx="370.86216145548" cy="302.67059218336414" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="32" data-strategy="1" cx="409.8919279908771" cy="312.7862738933886" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="33" data-strategy="1" cx="369.8792055164425" cy="304.24706227192905" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="34" data-strategy="1" cx="404.59814377567534" cy="307.2707847047643" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="35" data-strategy="1" cx="392.55796699575546" cy="308.33610260326924" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="36" data-strategy="1" cx="338.64633228647057" cy="321.7028395864025" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="37" data-strategy="1" cx="350.9995219393625" cy="312.05877846419395" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="38" data-strategy="1" cx="387.7878273556803" cy="321.8267269605887" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="40" data-strategy="1" cx="380.3717114944511" cy="319.46035389123324" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="41" data-strategy="1" cx="325.0683244202348" cy="354.3534699034671" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="42" data-strategy="1" cx="327.10212964541586" cy="350.26693865674383" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="43" data-strategy="1" cx="357.2292764285862" cy="320.01839103980626" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="45" data-strategy="1" cx="362.25929549652415" cy="352.54036200280177" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="46" data-strategy="1" cx="339.626451443485" cy="357.1829673761955" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="47" data-strategy="1" cx="409.8913502984642" cy="328.9222376540465" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="48" data-strategy="1" cx="330.2467466990996" cy="343.8646275921399" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="50" data-strategy="1" cx="354.29781120881825" cy="379.3752583269697" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="51" data-strategy="1" cx="402.0200067137348" cy="355.11463617569257" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="52" data-strategy="1" cx="332.03853530745505" cy="363.24391054362064" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="53" data-strategy="1" cx="367.02741276881125" cy="350.4171458962948" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="55" data-strategy="1" cx="403.3369796070076" cy="336.52125168489613" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="56" data-strategy="1" cx="369.2557524495031" cy="339.7186493724151" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="57" data-strategy="1" cx="406.73605684881164" cy="341.70876584783036" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="58" data-strategy="1" cx="354.43944149130215" cy="340.28168613826404" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="60" data-strategy="1" cx="340.75309579187456" cy="341.44243053199625" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="61" data-strategy="1" cx="370.04402498518664" cy="349.065694977079" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="62" data-strategy="1" cx="388.87745197086383" cy="353.4005426765921" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="63" data-strategy="1" cx="395.09568918236823" cy="356.0426072710275" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="64" data-strategy="1" cx="348.3756230906829" cy="344.893474027719" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="65" data-strategy="1" cx="373.80660691913596" cy="334.48755558539557" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="66" data-strategy="1" cx="353.3864411526974" cy="340.1543380829299" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="68" data-strategy="1" cx="372.7844458676355" cy="319.12074252306695" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="69" data-strategy="1" cx="376.40709911997214" cy="330.03481188161766" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="71" data-strategy="1" cx="343.0164179595977" cy="303.7635997680954" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="73" data-strategy="1" cx="385.8128967426619" cy="260.29204635401476" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="74" data-strategy="1" cx="327.03670075787636" cy="278.1399679979374" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="76" data-strategy="1" cx="351.2246019876181" cy="295.96819994676093" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="78" data-strategy="1" cx="326.05194349368986" cy="295.39788477626234" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="79" data-strategy="1" cx="356.802869896384" cy="307.92353092820184" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="81" data-strategy="1" cx="345.3149534667166" cy="300.0167283588607" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="84" data-strategy="1" cx="327.33922825567544" cy="294.4938687272575" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="86" data-strategy="1" cx="337.60447096562615" cy="274.12812705181443" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="89" data-strategy="1" cx="384.7413788547914" cy="278.4563616896851" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="91" data-strategy="1" cx="398.68089461090136" cy="269.4013667649218" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="92" data-strategy="1" cx="358.2976102150498" cy="296.1847224283556" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="94" data-strategy="1" cx="354.2264440757362" cy="282.2408127953054" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="95" data-strategy="1" cx="355.424542795683" cy="303.5807841633306" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="97" data-strategy="1" cx="333.60511821426934" cy="311.2072754935851" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="99" data-strategy="1" cx="382.33012790322954" cy="302.68155042071174" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="100" data-strategy="1" cx="348.6715753969399" cy="295.4816176968399" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="102" data-strategy="1" cx="327.4980449422141" cy="277.45429965900246" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="104" data-strategy="1" cx="400.0223357004568" cy="276.2226632436805" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="105" data-strategy="1" cx="362.2803884063329" cy="274.09563423834425" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="107" data-strategy="1" cx="410.0871966936964" cy="292.93384673072876" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="108" data-strategy="1" cx="347.08596441595137" cy="300.2199893081898" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="109" data-strategy="1" cx="369.08142484562694" cy="297.8414828509336" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="110" data-strategy="1" cx="357.4436856346817" cy="311.04033520031606" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="112" data-strategy="1" cx="348.2682552895474" cy="357.9336679495057" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="113" data-strategy="1" cx="342.9648828728969" cy="343.0214353376647" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="168" data-strategy="1" cx="361.1329544425892" cy="299.92019014625083" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="170" data-strategy="1" cx="363.79257309319615" cy="299.68808308529566" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="171" data-strategy="1" cx="345.0332861323155" cy="301.4318757367099" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="172" data-strategy="1" cx="349.5422187660993" cy="306.8788352637992" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="173" data-strategy="1" cx="330.6913736247916" cy="312.5766150507387" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="175" data-strategy="1" cx="373.88414247238865" cy="326.9250091711514" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="176" data-strategy="1" cx="408.14879571911064" cy="342.5794559042365" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="177" data-strategy="1" cx="399.2742884761803" cy="346.7043027198296" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="178" data-strategy="1" cx="389.40871287356106" cy="346.74528435163245" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="179" data-strategy="1" cx="332.6752199040743" cy="338.068415883574" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="180" data-strategy="1" cx="342.24016665795307" cy="337.5881405305513" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="181" data-strategy="1" cx="406.22865090189117" cy="328.9035157631416" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="182" data-strategy="1" cx="364.93039034308305" cy="304.6317107798786" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="183" data-strategy="1" cx="408.3587767379271" cy="304.7247991531625" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="184" data-strategy="1" cx="398.33689008703857" cy="296.86929166504774" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="186" data-strategy="1" cx="402.5811964155439" cy="315.30578255789004" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="187" data-strategy="1" cx="366.5779179754841" cy="308.60957293664416" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="188" data-strategy="1" cx="392.6466487543815" cy="318.19626762631333" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="189" data-strategy="1" cx="348.8172851465556" cy="333.37620393469166" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="191" data-strategy="1" cx="391.64445617610454" cy="320.990680365305" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="192" data-strategy="1" cx="387.5272629605823" cy="323.1892133354397" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="193" data-strategy="1" cx="359.1068823146953" cy="309.3440724090658" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="194" data-strategy="1" cx="368.91623689504274" cy="320.2621493526049" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="196" data-strategy="1" cx="397.6164256886363" cy="316.0935560015886" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="197" data-strategy="1" cx="399.85440284241986" cy="320.6500181188057" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="198" data-strategy="1" cx="344.0578354861897" cy="321.19318874626686" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="199" data-strategy="1" cx="374.8433276559368" cy="312.24415428143686" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="201" data-strategy="1" cx="410.0098880170144" cy="274.7118738789394" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="203" data-strategy="1" cx="343.77892514270985" cy="276.16457432859636" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="204" data-strategy="1" cx="358.0553573920276" cy="284.65827085888526" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="205" data-strategy="1" cx="407.83094274418073" cy="288.9172046550777" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="206" data-strategy="1" cx="405.0528595733773" cy="307.07693604738034" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="208" data-strategy="1" cx="334.10063040624897" cy="312.5452922861551" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="209" data-strategy="1" cx="359.29751109635515" cy="299.0374531072789" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="210" data-strategy="1" cx="327.0625091197178" cy="303.7150887868284" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="211" data-strategy="1" cx="331.62860809479946" cy="288.7070859914773" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="213" data-strategy="1" cx="368.25737103325054" cy="278.47939906405816" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="214" data-strategy="1" cx="407.2299939493292" cy="287.74287340033277" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="215" data-strategy="1" cx="393.8504753109914" cy="286.0672982954934" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="216" data-strategy="1" cx="352.42725720295965" cy="304.50457471169915" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="218" data-strategy="1" cx="373.86487690642224" cy="303.7183629117446" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="219" data-strategy="1" cx="362.63263269931963" cy="289.74080668891276" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="220" data-strategy="1" cx="357.10348277257975" cy="283.88378083698467" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="221" data-strategy="1" cx="345.36697872825334" cy="272.1574053855848" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="223" data-strategy="1" cx="329.16439051922663" cy="320.308792562896" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="224" data-strategy="1" cx="401.8557387213198" cy="317.44343143192714" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="225" data-strategy="1" cx="394.0338959889203" cy="312.3045557252737" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="226" data-strategy="1" cx="338.5355005283667" cy="318.4042195826131" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="227" data-strategy="1" cx="373.1683518810191" cy="329.63324898815983" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="228" data-strategy="1" cx="342.00474797890195" cy="333.6908226803716" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="229" data-strategy="1" cx="375.73132876942617" cy="347.16118861388753" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="230" data-strategy="1" cx="387.78799213458296" cy="332.35405469737333" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="231" data-strategy="1" cx="402.0081585626507" cy="335.81653705397764" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="232" data-strategy="1" cx="399.6937435234874" cy="313.10874448682887" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="233" data-strategy="1" cx="401.15495120312136" cy="318.1686117637968" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="234" data-strategy="1" cx="329.22812705654064" cy="315.8361338057371" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="235" data-strategy="1" cx="325.3733324540434" cy="310.2703323201129" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="236" data-strategy="1" cx="353.44796530765603" cy="294.48351105034595" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="237" data-strategy="1" cx="380.4870269375302" cy="310.25909404818077" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="238" data-strategy="1" cx="330.70634350492713" cy="331.6850049612856" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="239" data-strategy="1" cx="325.1204705087639" cy="321.0574366317885" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="240" data-strategy="1" cx="354.3173168265043" cy="305.91921004688714" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="241" data-strategy="1" cx="403.77964608326533" cy="307.420369504801" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="242" data-strategy="1" cx="366.5022160628885" cy="303.7933862573151" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="243" data-strategy="1" cx="392.505158897644" cy="303.21389193633047" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="244" data-strategy="1" cx="371.6076874812978" cy="315.5753680898677" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="245" data-strategy="1" cx="341.35896748377144" cy="296.18992491238635" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="246" data-strategy="1" cx="353.9662582874231" cy="312.691559726002" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="247" data-strategy="1" cx="333.14926551650916" cy="300.41317259803327" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="248" data-strategy="1" cx="370.0813205622355" cy="314.84968723088224" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="249" data-strategy="1" cx="395.6644735153051" cy="317.1263267879185" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="250" data-strategy="1" cx="347.1501649454529" cy="313.4882832654391" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="251" data-strategy="1" cx="335.82338672557984" cy="318.6476342935341" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="252" data-strategy="1" cx="357.0433039760574" cy="281.43364824289085" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="253" data-strategy="1" cx="375.16307373760856" cy="292.55472116922385" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="254" data-strategy="1" cx="404.10581232974107" cy="301.78766221011125" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="255" data-strategy="1" cx="359.3329922271195" cy="317.8714017096875" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="256" data-strategy="1" cx="387.30316607135893" cy="333.5567241365912" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="257" data-strategy="1" cx="357.98374097073116" cy="326.38183510898716" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="258" data-strategy="1" cx="360.61683169470575" cy="332.27062043052894" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="259" data-strategy="1" cx="367.84919871553365" cy="360.92990464728626" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="261" data-strategy="1" cx="364.55867588698106" cy="329.7931169621634" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="262" data-strategy="1" cx="334.04213441978965" cy="351.9059339631803" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="263" data-strategy="1" cx="394.6363321739105" cy="352.8909378005108" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="264" data-strategy="1" cx="368.80132568120854" cy="374.86200076035243" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="265" data-strategy="1" cx="352.0964027007486" cy="362.1103305295285" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="266" data-strategy="1" cx="324.6175490815892" cy="354.99197694693567" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="267" data-strategy="1" cx="398.1216419358273" cy="360.1362527236585" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="268" data-strategy="1" cx="338.238420496784" cy="358.0331274254737" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="269" data-strategy="1" cx="402.7967536400828" cy="360.5903695329931" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="270" data-strategy="1" cx="344.480710748172" cy="351.9458982071113" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="271" data-strategy="1" cx="325.9205655981967" cy="351.63385345598425" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="272" data-strategy="1" cx="404.8581545851408" cy="366.67719574163016" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="273" data-strategy="1" cx="345.08993662058396" cy="355.7624630599625" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="274" data-strategy="1" cx="394.8511119854894" cy="371.27199459216286" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="275" data-strategy="1" cx="338.5190413037885" cy="371.1704583602328" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="276" data-strategy="1" cx="333.06161885370676" cy="383.2009580444598" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="277" data-strategy="1" cx="360.0524019891064" cy="383.17895950197203" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="278" data-strategy="1" cx="357.37363063812745" cy="369.5140202509266" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="279" data-strategy="1" cx="356.13702518720663" cy="411.0971415931066" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="280" data-strategy="1" cx="403.09665308780853" cy="413.90940381613893" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="281" data-strategy="1" cx="394.97993462458675" cy="451.50963624016" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="282" data-strategy="1" cx="410.8955203041482" cy="424.9343057394252" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="283" data-strategy="1" cx="347.5359701316681" cy="416.1615113988038" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="284" data-strategy="1" cx="398.7961571128566" cy="423.175360842263" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="285" data-strategy="1" cx="399.0070050520165" cy="395.14923006384447" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="286" data-strategy="1" cx="340.6289759736376" cy="407.43317785724224" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="287" data-strategy="1" cx="377.0502670135262" cy="394.84287888463086" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="288" data-strategy="1" cx="350.9723906028248" cy="394.71572984365736" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="289" data-strategy="1" cx="361.1143977011168" cy="423.2537492597148" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="290" data-strategy="1" cx="374.02636098305106" cy="414.5218482097077" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="291" data-strategy="1" cx="362.52042871330684" cy="412.9457414436891" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="292" data-strategy="1" cx="336.79306229574104" cy="401.1980779011734" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="293" data-strategy="1" cx="394.71931080104355" cy="404.0181016657032" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="294" data-strategy="1" cx="411.1474930906507" cy="418.61812903198864" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="295" data-strategy="1" cx="354.2638399165762" cy="401.73951875861866" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="296" data-strategy="1" cx="383.9233922832613" cy="411.4002310903913" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="297" data-strategy="1" cx="381.52939263599103" cy="381.33671907826647" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="298" data-strategy="1" cx="409.01335929103476" cy="376.81409847205526" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="299" data-strategy="1" cx="404.6495193637986" cy="372.3236168624791" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="300" data-strategy="1" cx="403.6993783443523" cy="374.8234925107834" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="301" data-strategy="1" cx="327.64067626005703" cy="379.90146148233856" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="304" data-strategy="1" cx="371.8237643080349" cy="273.74762512611994" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="305" data-strategy="1" cx="399.1097263133827" cy="299.41177393851973" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="306" data-strategy="1" cx="373.71715139360475" cy="293.1740858878087" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="307" data-strategy="1" cx="339.28792260506026" cy="284.80116492222714" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="308" data-strategy="1" cx="344.71697670533507" cy="300.48416967462015" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="309" data-strategy="1" cx="364.39590665736506" cy="289.6059887242019" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="310" data-strategy="1" cx="404.15696105883995" cy="285.60625964940675" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="311" data-strategy="1" cx="367.85516597642766" cy="282.49649131652905" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="312" data-strategy="1" cx="389.1996691675142" cy="286.6495345950262" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="313" data-strategy="1" cx="384.9715621381688" cy="279.3080847774984" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="314" data-strategy="1" cx="360.0692004599111" cy="291.0081709084588" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="315" data-strategy="1" cx="383.17602531725754" cy="292.253524986237" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="316" data-strategy="1" cx="383.85141005208664" cy="281.10464672093684" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="317" data-strategy="1" cx="334.3162023337851" cy="285.5810378656643" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="318" data-strategy="1" cx="397.09982680708003" cy="277.4974457850784" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="319" data-strategy="1" cx="382.7929377169967" cy="263.65000177264494" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="320" data-strategy="1" cx="333.5327834534358" cy="272.1476537623866" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="321" data-strategy="1" cx="367.4164226570943" cy="269.05775224115473" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="322" data-strategy="1" cx="383.9110432267552" cy="270.51529046093674" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="323" data-strategy="1" cx="389.4556916868792" cy="257.1042685471158" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="324" data-strategy="1" cx="335.763132714877" cy="238.53248898378658" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="325" data-strategy="1" cx="358.0137698116488" cy="248.65078616484053" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="326" data-strategy="1" cx="340.61006323476056" cy="261.77732209898113" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="327" data-strategy="1" cx="379.17068846734384" cy="267.32376665041113" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="328" data-strategy="1" cx="338.3923737836926" cy="274.5209496960754" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="329" data-strategy="1" cx="379.1627974970662" cy="267.75301296002795" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="330" data-strategy="1" cx="385.88121686385114" cy="281.2657666648667" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="331" data-strategy="1" cx="355.64529804065427" cy="279.04844066547554" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="332" data-strategy="1" cx="353.6793153260356" cy="278.017570456331" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="333" data-strategy="1" cx="342.9753179212796" cy="272.63739338669217" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="334" data-strategy="1" cx="381.21024317432546" cy="258.92834900368905" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="335" data-strategy="1" cx="410.9064323601805" cy="264.5799752030238" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="336" data-strategy="1" cx="351.9960513022565" cy="269.3401654782061" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="337" data-strategy="1" cx="335.5829715022186" cy="258.67644768711824" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="338" data-strategy="1" cx="347.91008635110364" cy="267.9349539564653" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="339" data-strategy="1" cx="404.90857935914755" cy="256.82608566937313" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="340" data-strategy="1" cx="347.1136529183994" cy="250.76502060268135" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="341" data-strategy="1" cx="338.77715747000076" cy="270.789508670206" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="342" data-strategy="1" cx="324.74579809833085" cy="283.86135477821904" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="343" data-strategy="1" cx="388.7378866735218" cy="280.81724747395504" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="344" data-strategy="1" cx="351.72533078210927" cy="254.76711721647658" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="345" data-strategy="1" cx="409.17077965894924" cy="259.21077184747594" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="346" data-strategy="1" cx="337.28291890414516" cy="273.6134796106793" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="347" data-strategy="1" cx="356.1196620563156" cy="287.759495012844" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="348" data-strategy="1" cx="393.62936024664" cy="284.5962875730711" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="349" data-strategy="1" cx="353.23069628012007" cy="261.8195850561328" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="395" data-strategy="1" cx="373.38483642451433" cy="326.4817773348601" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="396" data-strategy="1" cx="344.0032918050775" cy="293.51202654859185" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="397" data-strategy="1" cx="346.2181573695945" cy="253.47573527602577" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="398" data-strategy="1" cx="396.5134300618571" cy="272.2616108059324" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="399" data-strategy="1" cx="336.2811512234234" cy="290.44878069414744" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="400" data-strategy="1" cx="346.0043955678325" cy="301.55182745242166" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="401" data-strategy="1" cx="401.33359903485234" cy="304.5831887923003" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="402" data-strategy="1" cx="345.50802139382034" cy="308.7658867979511" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="403" data-strategy="1" cx="355.07951791386273" cy="311.70500426760594" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="404" data-strategy="1" cx="344.03421318097355" cy="319.14883004536046" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="405" data-strategy="1" cx="405.70788066549204" cy="296.77753467463316" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="407" data-strategy="1" cx="334.976581775414" cy="256.64808024985825" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="408" data-strategy="1" cx="327.3275214740653" cy="248.8198936830329" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="409" data-strategy="1" cx="381.26567207596156" cy="286.1324635590438" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="410" data-strategy="1" cx="409.423785020513" cy="307.19046441203466" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="412" data-strategy="1" cx="357.39381157691963" cy="280.18526847600174" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="413" data-strategy="1" cx="330.46271118771574" cy="281.28594637581296" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="414" data-strategy="1" cx="406.4873895245745" cy="270.4785803395633" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="415" data-strategy="1" cx="407.3186325771478" cy="242.60262754100273" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="417" data-strategy="1" cx="375.7728235972385" cy="245.65070326976192" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="418" data-strategy="1" cx="345.0602537806222" cy="226.4609430844859" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="419" data-strategy="1" cx="396.6685436658567" cy="239.12255501311404" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="420" data-strategy="1" cx="369.4899784577386" cy="246.90110326668605" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="422" data-strategy="1" cx="394.41845056736514" cy="222.40687792632954" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="423" data-strategy="1" cx="354.18099773870705" cy="209.6354138502332" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="424" data-strategy="1" cx="333.3709271581815" cy="220.34549224531986" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="425" data-strategy="1" cx="410.5757681522172" cy="242.85273117606425" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="427" data-strategy="1" cx="400.5270741894903" cy="243.51897776841244" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="428" data-strategy="1" cx="387.2498285814921" cy="223.3801420540618" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="429" data-strategy="1" cx="347.0229046217523" cy="251.9689380459712" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="430" data-strategy="1" cx="361.81965476313997" cy="245.49081468818733" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="431" data-strategy="1" cx="401.12431452207574" cy="255.28008576902533" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="443" data-strategy="1" cx="355.8928107247485" cy="320.1764204818434" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="444" data-strategy="1" cx="375.4002547210014" cy="320.7297413377803" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="445" data-strategy="1" cx="347.61572273717525" cy="307.0599184325145" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="446" data-strategy="1" cx="339.09117876739964" cy="342.0028364287087" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="447" data-strategy="1" cx="383.41665755596483" cy="316.83246035113825" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="448" data-strategy="1" cx="397.18231820994623" cy="290.812315116673" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="449" data-strategy="1" cx="382.62735702061553" cy="301.15816656634814" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="450" data-strategy="1" cx="382.4683019405808" cy="293.7016563070766" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="451" data-strategy="1" cx="357.47008980126634" cy="319.3827095987628" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="452" data-strategy="1" cx="397.1143186484049" cy="300.37025111255747" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="453" data-strategy="1" cx="331.0903729150071" cy="312.47844526799656" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="454" data-strategy="1" cx="324.79589698742376" cy="334.29231170091583" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="455" data-strategy="1" cx="366.39397651955187" cy="326.1068003587603" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="456" data-strategy="1" cx="342.53315456245736" cy="355.6195935298783" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="457" data-strategy="1" cx="356.9467800485279" cy="315.172396968696" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="458" data-strategy="1" cx="396.4878398589431" cy="293.7277654574322" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="459" data-strategy="1" cx="329.93729320187276" cy="299.45655150035185" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="460" data-strategy="1" cx="361.99113942966693" cy="274.54250732892865" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="461" data-strategy="1" cx="330.0440480370857" cy="276.0724001758889" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="462" data-strategy="1" cx="347.0570074829682" cy="272.75781814310795" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="463" data-strategy="1" cx="349.00243640921" cy="263.8800669537533" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="465" data-strategy="1" cx="369.568486258562" cy="297.0602010126675" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="466" data-strategy="1" cx="360.92573003109567" cy="281.2200263816595" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="467" data-strategy="1" cx="374.03662208809175" cy="245.82645857106777" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="468" data-strategy="1" cx="389.1893970040109" cy="252.58652047183094" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="470" data-strategy="1" cx="398.89279164661474" cy="271.0664405338209" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="471" data-strategy="1" cx="402.10641846092415" cy="257.48585029317167" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="472" data-strategy="1" cx="389.0721191883591" cy="246.43782373463318" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="475" data-strategy="1" cx="350.5640653179577" cy="241.84726431955264" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="476" data-strategy="1" cx="325.1762004479758" cy="217.4812740287615" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="477" data-strategy="1" cx="355.17496189195134" cy="182.16205954980748" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="480" data-strategy="1" cx="345.90784994830324" cy="204.3160288872802" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="504" data-strategy="1" cx="358.6485313911657" cy="299.8580864697937" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="505" data-strategy="1" cx="364.5205685974406" cy="290.82982924511225" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="506" data-strategy="1" cx="356.6945176827013" cy="277.78754227201546" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="507" data-strategy="1" cx="409.6131007060201" cy="264.4734199595154" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="508" data-strategy="1" cx="333.21410289080893" cy="256.57531398894537" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="509" data-strategy="1" cx="349.93113318717155" cy="232.02802045278185" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="510" data-strategy="1" cx="365.599971758713" cy="232.53712570336637" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="511" data-strategy="1" cx="356.42982002104327" cy="187.88154485114362" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="512" data-strategy="1" cx="358.92440376962577" cy="185.1767992483795" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="513" data-strategy="1" cx="352.9627815559928" cy="204.71045984409673" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="514" data-strategy="1" cx="410.3297028867862" cy="253.07765984320633" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="515" data-strategy="1" cx="356.4627877655128" cy="224.5496707038354" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="516" data-strategy="1" cx="381.5700167138636" cy="218.11497725896533" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="517" data-strategy="1" cx="389.90649452144214" cy="230.2224590784758" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="518" data-strategy="1" cx="367.85978755913436" cy="274.4945807707504" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="519" data-strategy="1" cx="404.0495240751856" cy="328.4376993110743" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="520" data-strategy="1" cx="405.51524465203397" cy="292.7406309455294" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="521" data-strategy="1" cx="328.1314873822285" cy="293.87739943950794" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="522" data-strategy="1" cx="372.296938712934" cy="267.95110492478824" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="523" data-strategy="1" cx="381.3061211025064" cy="198.47536129272802" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="524" data-strategy="1" cx="339.2949432453133" cy="200.40189532799073" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="525" data-strategy="1" cx="360.61535811066386" cy="159.00542391671243" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="527" data-strategy="1" cx="339.25582483927667" cy="124.6962615973284" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="528" data-strategy="1" cx="405.3079407501923" cy="177.55148702456313" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="529" data-strategy="1" cx="337.3141375146783" cy="169.48377588769242" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="530" data-strategy="1" cx="368.4137901527203" cy="125.43921780108764" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="531" data-strategy="1" cx="352.4191964598768" cy="121.23526796083695" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="532" data-strategy="1" cx="398.11737215047526" cy="129.86235765454663" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="533" data-strategy="1" cx="356.0875816038475" cy="166.583492799204" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="535" data-strategy="1" cx="385.5306675870253" cy="167.9828031669716" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="536" data-strategy="1" cx="347.9775069551687" cy="173.51731455927214" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="537" data-strategy="1" cx="409.3864427989809" cy="136.20881809197218" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="538" data-strategy="1" cx="380.69785770548543" cy="184.57648211235883" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="539" data-strategy="1" cx="397.54818362895037" cy="130.97678417152682" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="540" data-strategy="1" cx="398.8226634855716" cy="85.90661593098173" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="541" data-strategy="1" cx="365.75235347647265" cy="78.23732205381083" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="542" data-strategy="1" cx="399.6080820302278" cy="65.6615083170799" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="543" data-strategy="1" cx="329.401289716173" cy="114.21402177264008" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="544" data-strategy="1" cx="406.0128476533222" cy="122.41677847409824" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="545" data-strategy="1" cx="347.6880807287829" cy="114.36822699123076" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="546" data-strategy="1" cx="390.91034762048406" cy="145.49745382975314" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="547" data-strategy="1" cx="325.2996665592035" cy="188.61616655569063" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="548" data-strategy="1" cx="403.4567326545841" cy="225.65888959304192" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="549" data-strategy="1" cx="380.46573937995424" cy="244.73867064240164" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="550" data-strategy="1" cx="359.3961263344713" cy="248.5471274542571" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="551" data-strategy="1" cx="348.3816393836988" cy="257.16573606435617" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="552" data-strategy="1" cx="366.1233836880211" cy="269.712458849314" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="553" data-strategy="1" cx="378.6078434920978" cy="290.1523390204807" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="554" data-strategy="1" cx="379.35676146910083" cy="269.87101568902676" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="555" data-strategy="1" cx="393.3620246915512" cy="244.35841651994832" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="556" data-strategy="1" cx="375.38061082368984" cy="232.9669481831188" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="557" data-strategy="1" cx="392.07208979065945" cy="213.99425664629652" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="558" data-strategy="1" cx="344.22305162088503" cy="264.62507899256184" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="559" data-strategy="1" cx="403.79854881454173" cy="220.74467532008057" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="560" data-strategy="1" cx="371.34290401489113" cy="196.13095243043458" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="561" data-strategy="1" cx="375.01207959461544" cy="211.5465644683094" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="562" data-strategy="1" cx="344.0371050995425" cy="241.97573099988904" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="563" data-strategy="1" cx="342.6242879869065" cy="292.58519974506515" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="564" data-strategy="1" cx="361.785610950629" cy="318.1056329410441" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="565" data-strategy="1" cx="377.1414741024342" cy="326.56406348017646" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="566" data-strategy="1" cx="337.49801064575973" cy="303.6784626820754" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="567" data-strategy="1" cx="401.60269246282337" cy="330.3637334710697" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="568" data-strategy="1" cx="345.9747113613806" cy="347.30367136292693" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="569" data-strategy="1" cx="395.0863269100896" cy="344.09558212803165" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="570" data-strategy="1" cx="334.69094553414664" cy="282.7270316251353" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="571" data-strategy="1" cx="383.56909263415133" cy="280.4342020173316" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="572" data-strategy="1" cx="327.9899122000876" cy="245.16929186653095" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="573" data-strategy="1" cx="362.11250122493533" cy="198.13486083444235" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="648" data-strategy="1" cx="370.8267992407774" cy="322.7672758276632" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="649" data-strategy="1" cx="383.91430238776655" cy="322.1288235838466" r="3" fill="#2563eb" fill-opacity=".65"></circle><circle class="point" data-window="650" data-strategy="1" cx="391.18313858633576" cy="308.4284829090741" r="3" fill="#2563eb" fill-opacity=".65"></circle><line x1="323.75" y1="296.18992491238635" x2="411.75" y2="296.18992491238635" stroke="#be185d" stroke-width="3.5" class="median-line"/><text x="367.75" y="572" text-anchor="middle" >A｜每週動能</text><text x="367.75" y="596" text-anchor="middle" class="median-label" style="fill:#be185d">中位數 5.15%</text><line x1="558.25" y1="407.77385102150726" x2="558.25" y2="182.49903740655952" stroke="#059669" stroke-width="2" /><rect x="520.25" y="266.6159163003151" width="76" height="56.68393311500944" fill="#059669" fill-opacity=".12" stroke="#059669"/><line x1="520.25" y1="407.77385102150726" x2="596.25" y2="407.77385102150726" stroke="#059669" stroke-width="2" /><line x1="520.25" y1="182.49903740655952" x2="596.25" y2="182.49903740655952" stroke="#059669" stroke-width="2" /><circle class="point" data-window="18" data-strategy="2" cx="531.8081490884957" cy="217.11122897609002" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="19" data-strategy="2" cx="561.5916992393335" cy="216.73221876600405" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="20" data-strategy="2" cx="563.2348510384352" cy="235.6587433560268" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="21" data-strategy="2" cx="561.1072119618973" cy="206.02250286429413" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="22" data-strategy="2" cx="542.2564328231749" cy="214.70054606623577" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="23" data-strategy="2" cx="553.1047716946345" cy="234.6866884536555" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="24" data-strategy="2" cx="593.5553319626576" cy="248.7710598195758" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="25" data-strategy="2" cx="549.6059395704165" cy="265.3836414280636" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="26" data-strategy="2" cx="532.4755863814939" cy="252.64306655073278" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="27" data-strategy="2" cx="558.4201599027854" cy="260.4306223710596" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="28" data-strategy="2" cx="545.7103917532414" cy="292.46784094713735" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="29" data-strategy="2" cx="544.9618488652886" cy="290.79366903315696" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="30" data-strategy="2" cx="536.671315139897" cy="292.5160330696053" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="31" data-strategy="2" cx="545.1511799170011" cy="291.5990923362308" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="32" data-strategy="2" cx="552.8742365702163" cy="306.97686488248445" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="33" data-strategy="2" cx="549.2746700280962" cy="299.876675431115" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="34" data-strategy="2" cx="587.7110433697587" cy="317.8263239621373" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="35" data-strategy="2" cx="537.2544319277993" cy="317.9471274145194" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="36" data-strategy="2" cx="593.7496201802209" cy="324.73477775706357" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="37" data-strategy="2" cx="542.8673242503656" cy="320.17250051419245" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="38" data-strategy="2" cx="546.3784521926874" cy="312.82564849785143" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="40" data-strategy="2" cx="572.3140114363316" cy="326.0000511867507" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="41" data-strategy="2" cx="576.5398458550295" cy="353.5905061206388" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="42" data-strategy="2" cx="544.7383978204468" cy="358.93908555518715" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="43" data-strategy="2" cx="527.7275026236576" cy="316.2059036236553" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="45" data-strategy="2" cx="522.4603142408505" cy="361.2339195296874" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="46" data-strategy="2" cx="532.1298874801075" cy="357.7493762944892" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="47" data-strategy="2" cx="589.3079694406099" cy="343.368980501475" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="48" data-strategy="2" cx="545.010500371699" cy="341.88229854802864" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="50" data-strategy="2" cx="581.935156466805" cy="375.6033669154718" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="51" data-strategy="2" cx="537.7690329793998" cy="359.70783487417833" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="52" data-strategy="2" cx="530.6549277107413" cy="378.2258442577098" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="53" data-strategy="2" cx="530.2893101975592" cy="354.9727720177398" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="55" data-strategy="2" cx="521.7259871334853" cy="336.76611366025287" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="56" data-strategy="2" cx="550.0180562265765" cy="342.318698252241" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="57" data-strategy="2" cx="567.9040378368143" cy="350.3291649809792" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="58" data-strategy="2" cx="525.8322150817867" cy="344.8844042009115" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="60" data-strategy="2" cx="586.0272808268747" cy="337.93558893086" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="61" data-strategy="2" cx="539.8699178853162" cy="349.6622461704549" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="62" data-strategy="2" cx="589.7846260582252" cy="353.4060296769519" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="63" data-strategy="2" cx="525.8642274018534" cy="352.24370821327295" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="64" data-strategy="2" cx="515.9099054253448" cy="334.12491874885404" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="65" data-strategy="2" cx="579.7240630358557" cy="330.209889597484" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="66" data-strategy="2" cx="570.0450005918142" cy="344.5149807300987" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="68" data-strategy="2" cx="537.0850424104166" cy="309.1681318672062" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="69" data-strategy="2" cx="569.5876341300034" cy="313.81764060388355" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="71" data-strategy="2" cx="548.4601822494442" cy="309.615906333892" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="73" data-strategy="2" cx="571.4723261635475" cy="260.66999505916544" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="74" data-strategy="2" cx="578.0431035492736" cy="273.43488476311165" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="76" data-strategy="2" cx="530.4440568920629" cy="310.9270015369979" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="78" data-strategy="2" cx="581.0723669106368" cy="295.5081895994288" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="79" data-strategy="2" cx="557.7187851342829" cy="302.04105503647224" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="81" data-strategy="2" cx="557.0064047975416" cy="307.5098305567255" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="84" data-strategy="2" cx="517.7866297648346" cy="292.8897563419047" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="86" data-strategy="2" cx="584.4035410837391" cy="279.3944505246481" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="89" data-strategy="2" cx="535.0915778271014" cy="296.7220149367047" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="91" data-strategy="2" cx="529.9887079337605" cy="272.5275182804628" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="92" data-strategy="2" cx="543.3043432936955" cy="296.12507822729617" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="94" data-strategy="2" cx="562.1950504235155" cy="291.09558381506235" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="95" data-strategy="2" cx="589.0233111093484" cy="302.61711270209656" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="97" data-strategy="2" cx="582.0882737944584" cy="304.67328147821735" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="99" data-strategy="2" cx="555.145884094301" cy="303.3317904470296" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="100" data-strategy="2" cx="554.8196938914629" cy="294.1189033780089" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="102" data-strategy="2" cx="569.0322190272923" cy="268.67781890883646" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="104" data-strategy="2" cx="537.8740590213603" cy="281.8001218208422" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="105" data-strategy="2" cx="570.6625905161578" cy="270.4698407693488" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="107" data-strategy="2" cx="545.9879813060161" cy="281.8449388283824" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="108" data-strategy="2" cx="531.321247725496" cy="294.37737175294956" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="109" data-strategy="2" cx="580.7076631515303" cy="300.59216119578366" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="110" data-strategy="2" cx="552.5998210060579" cy="299.58099827106867" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="112" data-strategy="2" cx="583.9154620514787" cy="346.1233055385817" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="113" data-strategy="2" cx="522.9683412363718" cy="338.49452542584896" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="168" data-strategy="2" cx="551.5722737422584" cy="303.4998248962978" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="170" data-strategy="2" cx="533.146674720229" cy="299.96822538241344" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="171" data-strategy="2" cx="583.8843896589977" cy="303.13089645158885" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="172" data-strategy="2" cx="522.331962520748" cy="309.84363736336803" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="173" data-strategy="2" cx="525.3481151497786" cy="313.4503007885268" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="175" data-strategy="2" cx="522.1566856418276" cy="323.8123427718197" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="176" data-strategy="2" cx="543.7666359055829" cy="335.99558923267574" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="177" data-strategy="2" cx="600.8201407107057" cy="348.1853759448869" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="178" data-strategy="2" cx="573.2022219564782" cy="347.24492450156004" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="179" data-strategy="2" cx="595.649932785692" cy="343.4727989166993" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="180" data-strategy="2" cx="600.583449076202" cy="332.92471410026116" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="181" data-strategy="2" cx="593.4769108851807" cy="326.2536615921044" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="182" data-strategy="2" cx="558.9636483331471" cy="310.3609670923828" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="183" data-strategy="2" cx="530.6748750241579" cy="307.3318184382581" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="184" data-strategy="2" cx="600.4296057623486" cy="303.0250760148697" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="186" data-strategy="2" cx="560.5466453066146" cy="312.6173729087473" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="187" data-strategy="2" cx="580.0290432483417" cy="308.9029021090988" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="188" data-strategy="2" cx="522.4995546946643" cy="316.27594381229824" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="189" data-strategy="2" cx="568.4291277598627" cy="327.49711618117686" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="191" data-strategy="2" cx="579.6844480183836" cy="321.54471211367445" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="192" data-strategy="2" cx="539.079783601555" cy="315.51997928755577" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="193" data-strategy="2" cx="601.550069099613" cy="305.5702557691649" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="194" data-strategy="2" cx="600.0905188842099" cy="313.89506747140433" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="196" data-strategy="2" cx="563.8991257987562" cy="319.083212742054" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="197" data-strategy="2" cx="549.3089040544291" cy="313.9481079398581" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="198" data-strategy="2" cx="550.7977711874064" cy="312.73481425511915" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="199" data-strategy="2" cx="518.8032195410517" cy="306.45179763023316" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="201" data-strategy="2" cx="526.5615973633885" cy="282.2958401220618" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="203" data-strategy="2" cx="565.3406303579338" cy="269.74024313161254" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="204" data-strategy="2" cx="564.6903689658218" cy="283.82977535281236" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="205" data-strategy="2" cx="519.7614049400742" cy="288.8949746937781" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="206" data-strategy="2" cx="561.7993733599017" cy="309.26651527768433" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="208" data-strategy="2" cx="516.9372221474504" cy="301.5269158156295" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="209" data-strategy="2" cx="517.3988765682308" cy="298.951767282929" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="210" data-strategy="2" cx="540.8833180790756" cy="306.5354975617492" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="211" data-strategy="2" cx="596.6743325915876" cy="292.09548249961495" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="213" data-strategy="2" cx="535.5775862292153" cy="276.6954529489925" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="214" data-strategy="2" cx="567.1779754892249" cy="292.17116022969634" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="215" data-strategy="2" cx="598.3340403680801" cy="290.6460033017148" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="216" data-strategy="2" cx="518.5269507748361" cy="305.4493054085416" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="218" data-strategy="2" cx="542.1084411485303" cy="297.78743569231403" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="219" data-strategy="2" cx="566.7496883625264" cy="289.6394596751531" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="220" data-strategy="2" cx="520.3911558649135" cy="288.8046701807865" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="221" data-strategy="2" cx="589.3499390858333" cy="273.6473756288189" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="223" data-strategy="2" cx="600.524624229002" cy="316.10752462551994" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="224" data-strategy="2" cx="587.8954178738898" cy="315.9516701051778" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="225" data-strategy="2" cx="581.3457533255602" cy="311.71653508760926" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="226" data-strategy="2" cx="541.4914354199402" cy="320.96314374954363" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="227" data-strategy="2" cx="522.3997764472798" cy="335.49673816565735" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="228" data-strategy="2" cx="548.8047754887878" cy="329.47092593145567" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="229" data-strategy="2" cx="601.549983169002" cy="345.86697900149494" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="230" data-strategy="2" cx="518.2497064052321" cy="330.802632086866" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="231" data-strategy="2" cx="516.4758435586057" cy="334.5933132161802" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="232" data-strategy="2" cx="588.1903259852817" cy="319.5509224728871" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="233" data-strategy="2" cx="516.2119239792715" cy="311.3778750272757" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="234" data-strategy="2" cx="572.5898853512759" cy="313.6981011950121" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="235" data-strategy="2" cx="518.3725043923407" cy="305.39366206846" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="236" data-strategy="2" cx="591.5666264950603" cy="295.8362411454172" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="237" data-strategy="2" cx="591.5246059543744" cy="306.3894047539915" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="238" data-strategy="2" cx="595.6763459994193" cy="323.9515945091502" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="239" data-strategy="2" cx="518.3974651066192" cy="319.1829811739302" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="240" data-strategy="2" cx="544.2861533809953" cy="303.7941453145631" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="241" data-strategy="2" cx="552.3481295799911" cy="311.6357898471169" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="242" data-strategy="2" cx="577.7620088013211" cy="300.9306929327561" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="243" data-strategy="2" cx="568.221121472732" cy="304.6998517363858" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="244" data-strategy="2" cx="528.1991016577001" cy="318.0021321336928" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="245" data-strategy="2" cx="556.4438442680727" cy="293.0503269705836" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="246" data-strategy="2" cx="567.6801130928741" cy="310.190721727755" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="247" data-strategy="2" cx="532.7663344169562" cy="298.508234911792" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="248" data-strategy="2" cx="525.75435516816" cy="317.65671801358127" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="249" data-strategy="2" cx="561.0057238057766" cy="325.96033510560807" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="250" data-strategy="2" cx="601.1622668921392" cy="315.2845979700721" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="251" data-strategy="2" cx="563.700164088816" cy="319.7816718902865" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="252" data-strategy="2" cx="600.1142194329274" cy="281.5696542894784" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="253" data-strategy="2" cx="588.2058073400713" cy="300.49864877876473" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="254" data-strategy="2" cx="572.8503435179624" cy="310.1595031077601" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="255" data-strategy="2" cx="588.6196079801964" cy="319.6777112124327" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="256" data-strategy="2" cx="586.1575586518684" cy="336.6697833936324" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="257" data-strategy="2" cx="516.7742029486377" cy="326.77797755322916" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="258" data-strategy="2" cx="529.9102985592456" cy="334.8295851903042" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="259" data-strategy="2" cx="574.2498534298734" cy="367.51752874937426" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="261" data-strategy="2" cx="596.1222490561537" cy="329.646770814388" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="262" data-strategy="2" cx="534.3623695444157" cy="348.95774088223004" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="263" data-strategy="2" cx="567.0048332067811" cy="345.48989626795213" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="264" data-strategy="2" cx="589.5956483070005" cy="374.96588483295983" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="265" data-strategy="2" cx="578.8608017971079" cy="363.55086978810476" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="266" data-strategy="2" cx="538.34873927318" cy="358.22643133029214" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="267" data-strategy="2" cx="585.6276687357487" cy="359.3179352921047" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="268" data-strategy="2" cx="515.8588125087621" cy="349.0981945207004" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="269" data-strategy="2" cx="521.6707050333516" cy="359.93102322619643" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="270" data-strategy="2" cx="553.9502829827819" cy="355.43445277734804" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="271" data-strategy="2" cx="573.7236042017726" cy="355.78483251344767" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="272" data-strategy="2" cx="527.8328791998123" cy="365.6013862247417" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="273" data-strategy="2" cx="542.827442915864" cy="345.3436305621358" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="274" data-strategy="2" cx="529.091022703253" cy="363.55770213077335" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="275" data-strategy="2" cx="522.6092182275123" cy="371.3759532831049" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="276" data-strategy="2" cx="546.2550447676432" cy="379.547072849137" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="277" data-strategy="2" cx="560.1037043482395" cy="384.7183913780427" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="278" data-strategy="2" cx="525.5881094723139" cy="364.0836908166702" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="279" data-strategy="2" cx="556.0703411754125" cy="400.5584562129792" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="280" data-strategy="2" cx="521.526348536972" cy="413.9036667706955" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="281" data-strategy="2" cx="587.7623198230306" cy="443.6944140670304" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="282" data-strategy="2" cx="562.5382767321539" cy="421.0747619033965" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="283" data-strategy="2" cx="583.6659859406583" cy="413.72869679213113" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="284" data-strategy="2" cx="563.7063290275905" cy="404.7446772954969" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="285" data-strategy="2" cx="584.8988918100559" cy="389.2747110195049" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="286" data-strategy="2" cx="528.2404830377345" cy="397.2114648782333" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="287" data-strategy="2" cx="565.6774567184457" cy="396.25593785599625" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="288" data-strategy="2" cx="581.2479944865167" cy="394.14237641991906" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="289" data-strategy="2" cx="598.9723014247594" cy="407.77385102150726" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="290" data-strategy="2" cx="598.5548984954144" cy="409.2247412441481" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="291" data-strategy="2" cx="574.3899132026628" cy="397.43459790402636" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="292" data-strategy="2" cx="593.2294974916293" cy="397.95791794665826" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="293" data-strategy="2" cx="549.1639069067436" cy="401.28084235010243" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="294" data-strategy="2" cx="601.676134831635" cy="404.8611937258879" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="295" data-strategy="2" cx="520.1873304913253" cy="394.89415043101224" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="296" data-strategy="2" cx="523.4658550054537" cy="396.7361944700503" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="297" data-strategy="2" cx="530.3299018128556" cy="378.25312188165736" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="298" data-strategy="2" cx="592.7788758160407" cy="374.7006600033597" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="299" data-strategy="2" cx="550.025526672837" cy="370.80619248862223" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="300" data-strategy="2" cx="577.0622993951057" cy="371.74235354330244" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="301" data-strategy="2" cx="515.2011770380163" cy="366.2428528297446" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="304" data-strategy="2" cx="558.2906562622359" cy="278.26604935986893" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="305" data-strategy="2" cx="580.8860675718128" cy="292.93044096749514" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="306" data-strategy="2" cx="572.2601838757435" cy="282.39059323198427" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="307" data-strategy="2" cx="570.8818309080093" cy="287.05844364690546" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="308" data-strategy="2" cx="552.9499748529546" cy="299.7811352303565" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="309" data-strategy="2" cx="540.0427893614374" cy="295.94848568134427" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="310" data-strategy="2" cx="569.5437026165494" cy="287.8688144002407" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="311" data-strategy="2" cx="574.4908125388056" cy="279.3042248186907" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="312" data-strategy="2" cx="584.458551657484" cy="286.0636732862938" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="313" data-strategy="2" cx="581.5222052482067" cy="283.0296270846235" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="314" data-strategy="2" cx="546.0327660742702" cy="298.7762373664488" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="315" data-strategy="2" cx="568.4289502129219" cy="299.59225690631916" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="316" data-strategy="2" cx="574.510374021625" cy="289.71929271353287" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="317" data-strategy="2" cx="549.7099032453995" cy="296.04327606496673" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="318" data-strategy="2" cx="553.9935303546735" cy="282.8074790698938" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="319" data-strategy="2" cx="523.5792775031018" cy="272.59762859984914" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="320" data-strategy="2" cx="539.1411349728698" cy="281.09257716311214" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="321" data-strategy="2" cx="555.2109557322707" cy="277.3661929177179" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="322" data-strategy="2" cx="556.5487888517104" cy="285.68501800382074" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="323" data-strategy="2" cx="531.0923597600114" cy="266.73348800720504" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="324" data-strategy="2" cx="589.1729692626756" cy="246.77419487425507" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="325" data-strategy="2" cx="588.8241768174279" cy="259.5245925495546" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="326" data-strategy="2" cx="535.7957131945709" cy="271.6988862255677" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="327" data-strategy="2" cx="585.0776755096582" cy="280.5747983549994" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="328" data-strategy="2" cx="530.2843661217926" cy="278.5633190570043" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="329" data-strategy="2" cx="601.5925889496285" cy="276.1217963554909" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="330" data-strategy="2" cx="578.4520989698182" cy="289.6294426129057" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="331" data-strategy="2" cx="595.4830425428413" cy="293.0721919989349" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="332" data-strategy="2" cx="546.0731030254543" cy="290.45851230146127" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="333" data-strategy="2" cx="525.8804645455812" cy="276.03931443949546" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="334" data-strategy="2" cx="570.1916586245641" cy="268.47654491529875" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="335" data-strategy="2" cx="569.758653221251" cy="279.8150808138163" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="336" data-strategy="2" cx="548.2776285546117" cy="284.909439430062" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="337" data-strategy="2" cx="596.523737405951" cy="271.7584075741775" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="338" data-strategy="2" cx="527.2646745244824" cy="271.1070759047356" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="339" data-strategy="2" cx="533.2004319864017" cy="269.4527854921124" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="340" data-strategy="2" cx="582.6439214019567" cy="258.0669852216772" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="341" data-strategy="2" cx="578.5015261056668" cy="277.06970046585036" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="342" data-strategy="2" cx="584.3035527678375" cy="286.56626926620095" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="343" data-strategy="2" cx="552.6346453021447" cy="284.6810061437011" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="344" data-strategy="2" cx="583.9440552816363" cy="263.61037100681017" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="345" data-strategy="2" cx="597.4716445867621" cy="266.49834459342514" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="346" data-strategy="2" cx="583.5645153906974" cy="281.65269086146645" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="347" data-strategy="2" cx="569.8727458951504" cy="290.1885983897794" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="348" data-strategy="2" cx="572.2343931008149" cy="285.2212141068545" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="349" data-strategy="2" cx="601.6135458152294" cy="275.10073548046034" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="395" data-strategy="2" cx="519.598364988054" cy="307.11813512089964" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="396" data-strategy="2" cx="581.0637679467404" cy="292.9768888765607" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="397" data-strategy="2" cx="578.3750266752842" cy="241.56404071111035" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="398" data-strategy="2" cx="521.4421878819385" cy="261.6652942807621" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="399" data-strategy="2" cx="516.3281024205157" cy="277.05131804310815" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="400" data-strategy="2" cx="541.1444729582886" cy="289.65662525347005" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="401" data-strategy="2" cx="515.8580311768436" cy="306.45308310676376" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="402" data-strategy="2" cx="564.3658669556365" cy="297.43287149950356" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="403" data-strategy="2" cx="573.2600569570142" cy="296.76178455430426" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="404" data-strategy="2" cx="523.7600244332683" cy="309.6263004827464" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="405" data-strategy="2" cx="593.3406958024866" cy="281.67805507924146" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="407" data-strategy="2" cx="541.0655103683897" cy="242.38847137464558" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="408" data-strategy="2" cx="601.56752232753" cy="242.08326980247494" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="409" data-strategy="2" cx="580.9195253243247" cy="276.09929259464514" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="410" data-strategy="2" cx="584.0473653025819" cy="299.63791859697875" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="412" data-strategy="2" cx="547.6515894485697" cy="256.6009205683216" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="413" data-strategy="2" cx="527.4084078680248" cy="270.4666957706007" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="414" data-strategy="2" cx="523.2052861021834" cy="266.25521849177863" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="415" data-strategy="2" cx="518.3549381582382" cy="250.01363643180628" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="417" data-strategy="2" cx="539.8669576451624" cy="239.38280400627872" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="418" data-strategy="2" cx="585.9867149331719" cy="227.8236060173498" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="419" data-strategy="2" cx="581.3949877095074" cy="242.84245642704232" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="420" data-strategy="2" cx="547.9352987904111" cy="255.13550446251205" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="422" data-strategy="2" cx="586.239690755772" cy="221.76760043690751" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="423" data-strategy="2" cx="566.648910067023" cy="218.62083021697515" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="424" data-strategy="2" cx="549.9727982672683" cy="223.7226498735986" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="425" data-strategy="2" cx="561.8426239224152" cy="248.43514209882122" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="427" data-strategy="2" cx="516.1694724267386" cy="241.7130407142369" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="428" data-strategy="2" cx="596.0031876731025" cy="238.53386335426177" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="429" data-strategy="2" cx="600.693633756764" cy="260.3071975953396" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="430" data-strategy="2" cx="556.6279265511452" cy="259.98758800839863" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="431" data-strategy="2" cx="561.9039611084823" cy="259.75046272180066" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="443" data-strategy="2" cx="551.1532858267699" cy="328.70421654827885" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="444" data-strategy="2" cx="541.3640699817108" cy="328.95451255897734" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="445" data-strategy="2" cx="519.9285094570212" cy="328.9069864146982" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="446" data-strategy="2" cx="574.0309997823288" cy="336.83310457661105" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="447" data-strategy="2" cx="594.317999179014" cy="321.7262783987391" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="448" data-strategy="2" cx="599.5651580523438" cy="299.6778000432689" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="449" data-strategy="2" cx="580.9278974237995" cy="313.60788798285614" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="450" data-strategy="2" cx="589.0465980355588" cy="311.63644962340186" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="451" data-strategy="2" cx="544.0819844136719" cy="316.1619026855155" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="452" data-strategy="2" cx="538.570977480429" cy="309.90146814521955" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="453" data-strategy="2" cx="569.5152044342018" cy="313.0295034752019" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="454" data-strategy="2" cx="560.0634276941789" cy="340.1739774719641" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="455" data-strategy="2" cx="563.6803108638503" cy="332.24043274235817" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="456" data-strategy="2" cx="597.788683406046" cy="346.40403552105613" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="457" data-strategy="2" cx="538.225002913651" cy="320.0858740374926" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="458" data-strategy="2" cx="534.421541663075" cy="296.19104563593123" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="459" data-strategy="2" cx="589.1745350976312" cy="312.8622591617019" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="460" data-strategy="2" cx="550.8329657194091" cy="276.9301864115452" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="461" data-strategy="2" cx="520.5620696006827" cy="275.1573626510186" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="462" data-strategy="2" cx="594.9084757483407" cy="281.10114247006095" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="463" data-strategy="2" cx="582.258304526161" cy="272.83546510212176" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="465" data-strategy="2" cx="600.5582903203228" cy="289.3121446875593" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="466" data-strategy="2" cx="524.7130814866632" cy="284.7086536066409" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="467" data-strategy="2" cx="528.8845453975074" cy="262.3200106981769" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="468" data-strategy="2" cx="582.0102562298364" cy="266.4264002082732" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="470" data-strategy="2" cx="566.5841895223016" cy="268.83491235381285" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="471" data-strategy="2" cx="521.0818151470012" cy="265.49450001563423" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="472" data-strategy="2" cx="532.6010102685232" cy="259.42651795598573" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="475" data-strategy="2" cx="572.8842111443528" cy="242.3558478589563" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="476" data-strategy="2" cx="553.700220605391" cy="220.652599371777" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="477" data-strategy="2" cx="529.1427858406284" cy="196.4121805503925" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="480" data-strategy="2" cx="539.0330225075143" cy="196.0714013646973" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="504" data-strategy="2" cx="521.2158466681033" cy="303.60610142406443" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="505" data-strategy="2" cx="594.6219509268648" cy="276.99328579694793" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="506" data-strategy="2" cx="546.366401667431" cy="279.44606054215376" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="507" data-strategy="2" cx="556.4357324570396" cy="283.48922995735563" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="508" data-strategy="2" cx="585.4149822466438" cy="267.317870966908" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="509" data-strategy="2" cx="527.6912829737145" cy="234.87720655368278" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="510" data-strategy="2" cx="517.155580209202" cy="224.34822040051233" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="511" data-strategy="2" cx="545.2721326486277" cy="209.11035233222037" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="512" data-strategy="2" cx="595.5355346080008" cy="205.72343094425779" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="513" data-strategy="2" cx="578.341210757548" cy="212.42931323126115" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="514" data-strategy="2" cx="593.8202377034852" cy="257.1621375111995" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="515" data-strategy="2" cx="571.6451995835837" cy="222.85993306755873" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="516" data-strategy="2" cx="516.1589296380698" cy="232.6234530683322" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="517" data-strategy="2" cx="523.4600781840256" cy="250.15912004325588" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="518" data-strategy="2" cx="531.3492011926297" cy="282.1929510957287" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="519" data-strategy="2" cx="516.7734423256338" cy="333.9865224523161" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="520" data-strategy="2" cx="596.9705647283105" cy="309.1009487375401" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="521" data-strategy="2" cx="582.40136506468" cy="310.7032686313411" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="522" data-strategy="2" cx="518.1374494960451" cy="302.7060386326741" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="523" data-strategy="2" cx="575.2388102140923" cy="210.96814377753407" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="524" data-strategy="2" cx="555.243481034319" cy="226.27353372439597" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="525" data-strategy="2" cx="581.0040980516468" cy="193.1754965621873" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="527" data-strategy="2" cx="529.8424807058138" cy="173.92269662974994" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="528" data-strategy="2" cx="538.5049746897065" cy="187.92615137636696" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="529" data-strategy="2" cx="564.6557236698573" cy="186.61640234724814" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="530" data-strategy="2" cx="556.4458060584955" cy="164.14646438413405" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="531" data-strategy="2" cx="586.7141821949543" cy="145.41704843211772" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="532" data-strategy="2" cx="560.7245673289611" cy="142.30062268772997" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="533" data-strategy="2" cx="549.306307471204" cy="163.4015555041081" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="535" data-strategy="2" cx="519.7577118341349" cy="184.2360106835028" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="536" data-strategy="2" cx="535.7614542479766" cy="176.0802774896885" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="537" data-strategy="2" cx="521.7731078169639" cy="154.88395007049957" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="538" data-strategy="2" cx="581.9799020605477" cy="173.77125712905928" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="539" data-strategy="2" cx="570.4198478946736" cy="143.192412001546" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="540" data-strategy="2" cx="597.2193415784647" cy="101.93265368594817" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="541" data-strategy="2" cx="550.745531354906" cy="81.9662337541598" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="542" data-strategy="2" cx="529.5314201006431" cy="71.19523004182508" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="543" data-strategy="2" cx="516.4139551837653" cy="90.09220520599746" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="544" data-strategy="2" cx="541.8923481591883" cy="103.68329859237613" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="545" data-strategy="2" cx="599.7788407538864" cy="97.58916116017961" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="546" data-strategy="2" cx="537.2570570079891" cy="139.8546210652819" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="547" data-strategy="2" cx="549.0063324300394" cy="175.99507598662285" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="548" data-strategy="2" cx="554.5768845613445" cy="182.60835979126028" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="549" data-strategy="2" cx="577.1804797713386" cy="215.8208068038316" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="550" data-strategy="2" cx="556.7723620530111" cy="229.67556569414654" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="551" data-strategy="2" cx="563.7813313260735" cy="254.42409421564506" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="552" data-strategy="2" cx="559.8589497789965" cy="255.94417076877488" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="553" data-strategy="2" cx="550.1728368198508" cy="243.22473969996733" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="554" data-strategy="2" cx="527.9440119559505" cy="239.59387058697718" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="555" data-strategy="2" cx="579.2677382045564" cy="214.20732062095445" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="556" data-strategy="2" cx="592.6283648629536" cy="237.44038342487548" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="557" data-strategy="2" cx="567.1851028681683" cy="208.07761530436335" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="558" data-strategy="2" cx="574.211394932071" cy="231.7317896021037" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="559" data-strategy="2" cx="565.5748112150592" cy="182.49903740655952" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="560" data-strategy="2" cx="584.4834305684385" cy="174.78543835907283" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="561" data-strategy="2" cx="565.1691230179935" cy="212.0949093098036" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="562" data-strategy="2" cx="582.8040959209103" cy="211.7948171797303" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="563" data-strategy="2" cx="584.4239846403274" cy="260.08129938381194" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="564" data-strategy="2" cx="597.3517896196737" cy="281.5196112872969" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="565" data-strategy="2" cx="530.1686885417979" cy="310.9224401398078" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="566" data-strategy="2" cx="588.3524430019263" cy="307.4345483087516" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="567" data-strategy="2" cx="516.6486517733813" cy="310.98177883538085" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="568" data-strategy="2" cx="584.7007780195117" cy="331.806848241759" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="569" data-strategy="2" cx="527.5720851893977" cy="322.7873560588294" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="570" data-strategy="2" cx="561.0759359172343" cy="285.5882989169039" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="571" data-strategy="2" cx="592.7722731525773" cy="282.5359443051657" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="572" data-strategy="2" cx="554.5871152322763" cy="223.94397710962932" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="573" data-strategy="2" cx="599.2144935488972" cy="196.20573644824483" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="648" data-strategy="2" cx="521.2873927987534" cy="317.7808168247147" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="649" data-strategy="2" cx="558.2811029530744" cy="312.4110987339799" r="3" fill="#059669" fill-opacity=".65"></circle><circle class="point" data-window="650" data-strategy="2" cx="529.7757134663575" cy="306.39878249108574" r="3" fill="#059669" fill-opacity=".65"></circle><line x1="514.25" y1="297.43287149950356" x2="602.25" y2="297.43287149950356" stroke="#be185d" stroke-width="3.5" class="median-line"/><text x="558.25" y="572" text-anchor="middle" >B｜動能＋法人</text><text x="558.25" y="596" text-anchor="middle" class="median-label" style="fill:#be185d">中位數 4.94%</text><line x1="748.75" y1="407.54539058825713" x2="748.75" y2="180.86381397724108" stroke="#9333ea" stroke-width="2" /><rect x="710.75" y="270.53333134963736" width="76" height="59.8060008997698" fill="#9333ea" fill-opacity=".12" stroke="#9333ea"/><line x1="710.75" y1="407.54539058825713" x2="786.75" y2="407.54539058825713" stroke="#9333ea" stroke-width="2" /><line x1="710.75" y1="180.86381397724108" x2="786.75" y2="180.86381397724108" stroke="#9333ea" stroke-width="2" /><circle class="point" data-window="18" data-strategy="3" cx="765.106380431236" cy="213.85309706436436" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="19" data-strategy="3" cx="723.8171626131254" cy="212.14626720150557" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="20" data-strategy="3" cx="768.7344379704549" cy="223.778868253485" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="21" data-strategy="3" cx="774.1256621344504" cy="208.4809870709821" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="22" data-strategy="3" cx="718.9658629312395" cy="210.1708052683105" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="23" data-strategy="3" cx="714.5454638620091" cy="222.11541822963432" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="24" data-strategy="3" cx="746.9124678817121" cy="231.82605671452694" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="25" data-strategy="3" cx="716.8784579407518" cy="240.35481177499935" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="26" data-strategy="3" cx="788.8448032637173" cy="231.0415372332856" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="27" data-strategy="3" cx="734.1635553524043" cy="245.80702101809004" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="28" data-strategy="3" cx="769.9600716812406" cy="278.9244509337417" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="29" data-strategy="3" cx="783.1453315253714" cy="268.7819629468229" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="30" data-strategy="3" cx="707.5555658462436" cy="263.6060550540984" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="31" data-strategy="3" cx="771.1009583119348" cy="263.02231922794476" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="32" data-strategy="3" cx="782.8382729554348" cy="288.1051344595361" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="33" data-strategy="3" cx="712.2413105841125" cy="284.391766351376" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="34" data-strategy="3" cx="762.3464267935344" cy="288.6946442408139" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="35" data-strategy="3" cx="787.2393660666908" cy="291.9163571932913" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="36" data-strategy="3" cx="747.9793286508822" cy="294.81717533494754" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="37" data-strategy="3" cx="736.7002413006018" cy="302.24032424631463" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="38" data-strategy="3" cx="789.6132184566969" cy="308.90943375930715" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="40" data-strategy="3" cx="763.2545778101737" cy="299.87559319600746" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="41" data-strategy="3" cx="725.7129763582598" cy="322.7885326896245" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="42" data-strategy="3" cx="769.2952503512151" cy="340.9942061420129" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="43" data-strategy="3" cx="738.7831118787553" cy="316.5292153467767" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="45" data-strategy="3" cx="790.4851759109379" cy="347.6277784140061" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="46" data-strategy="3" cx="739.9931415997148" cy="343.9393121683823" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="47" data-strategy="3" cx="765.5712777602316" cy="333.9286348569592" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="48" data-strategy="3" cx="748.4863800385795" cy="343.79353615092685" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="50" data-strategy="3" cx="760.6039117417342" cy="364.62630125151543" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="51" data-strategy="3" cx="791.4653973814861" cy="349.65326610843954" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="52" data-strategy="3" cx="728.8051796394973" cy="366.5339916990559" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="53" data-strategy="3" cx="765.6829848728198" cy="366.92529345270134" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="55" data-strategy="3" cx="717.7924659119257" cy="330.82908342800215" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="56" data-strategy="3" cx="789.7204037047226" cy="333.17712298381423" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="57" data-strategy="3" cx="751.8491599226819" cy="346.9227721722248" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="58" data-strategy="3" cx="743.3436956234915" cy="347.34061842690835" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="60" data-strategy="3" cx="712.8416437443545" cy="329.1605654385891" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="61" data-strategy="3" cx="741.1444443074681" cy="341.37104878431336" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="62" data-strategy="3" cx="738.3551302255112" cy="349.0062109868481" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="63" data-strategy="3" cx="791.8628795712057" cy="354.3189497064215" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="64" data-strategy="3" cx="754.7189065547614" cy="343.70007589982066" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="65" data-strategy="3" cx="748.1057258585057" cy="318.30813348671904" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="66" data-strategy="3" cx="705.61490450263" cy="326.694418989682" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="68" data-strategy="3" cx="736.1532245332434" cy="316.71812615377524" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="69" data-strategy="3" cx="712.5054808744025" cy="328.9026366218557" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="71" data-strategy="3" cx="773.7819428763447" cy="299.49800427946036" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="73" data-strategy="3" cx="725.1414368428445" cy="273.542362431166" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="74" data-strategy="3" cx="787.4077945300694" cy="286.0787073541638" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="76" data-strategy="3" cx="730.6789747647198" cy="303.0884329528505" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="78" data-strategy="3" cx="790.0908889012611" cy="292.33521001276023" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="79" data-strategy="3" cx="746.4618180601615" cy="309.2400470667156" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="81" data-strategy="3" cx="735.6186620480306" cy="304.19596725850636" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="84" data-strategy="3" cx="711.724345073768" cy="304.7616615521779" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="86" data-strategy="3" cx="787.8662319115779" cy="288.05702620892316" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="89" data-strategy="3" cx="777.7719018178881" cy="297.48477293550553" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="91" data-strategy="3" cx="779.9785253663737" cy="276.56930007729244" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="92" data-strategy="3" cx="774.271658302934" cy="289.1217426495065" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="94" data-strategy="3" cx="763.3130528310475" cy="287.93836081143655" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="95" data-strategy="3" cx="773.4275551676969" cy="300.26608517274616" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="97" data-strategy="3" cx="790.1411799621176" cy="297.0362830063053" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="99" data-strategy="3" cx="769.803322319777" cy="295.5287872673397" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="100" data-strategy="3" cx="738.5326496660919" cy="293.2719384079519" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="102" data-strategy="3" cx="714.2364518086929" cy="264.1422285231721" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="104" data-strategy="3" cx="777.1211696561305" cy="277.9311398703604" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="105" data-strategy="3" cx="741.3031580922328" cy="278.0629373821139" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="107" data-strategy="3" cx="777.0605849208386" cy="280.50184204014255" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="108" data-strategy="3" cx="716.1146363109056" cy="301.1746415455968" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="109" data-strategy="3" cx="706.1402635297109" cy="290.0447524239138" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="110" data-strategy="3" cx="710.5643042879303" cy="305.1883734982316" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="112" data-strategy="3" cx="710.773205098021" cy="341.35469087295104" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="113" data-strategy="3" cx="760.318987745674" cy="341.2684394590285" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="168" data-strategy="3" cx="778.6796367761375" cy="302.47187811524293" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="170" data-strategy="3" cx="726.1708981492308" cy="307.4916203887825" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="171" data-strategy="3" cx="762.8580808092788" cy="316.43594775587013" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="172" data-strategy="3" cx="706.2669315688448" cy="311.86668079688866" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="173" data-strategy="3" cx="759.3321718874089" cy="315.6335554028855" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="175" data-strategy="3" cx="787.703644804144" cy="326.4932790202872" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="176" data-strategy="3" cx="751.4428749577122" cy="345.8148886553485" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="177" data-strategy="3" cx="738.7621535753236" cy="349.9136767207601" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="178" data-strategy="3" cx="739.1512366927373" cy="345.96046670320163" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="179" data-strategy="3" cx="707.1586283114583" cy="343.7207489414234" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="180" data-strategy="3" cx="729.2926006459436" cy="334.9802420121067" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="181" data-strategy="3" cx="719.5689912123989" cy="331.71782681377607" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="182" data-strategy="3" cx="716.2182188863193" cy="322.38148033537163" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="183" data-strategy="3" cx="746.2430319885225" cy="313.4327049011669" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="184" data-strategy="3" cx="716.4799753003883" cy="303.47900964225835" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="186" data-strategy="3" cx="769.8780366422077" cy="318.3836840359918" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="187" data-strategy="3" cx="787.16175308942" cy="312.8186324538184" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="188" data-strategy="3" cx="787.3757445867367" cy="323.1625497735368" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="189" data-strategy="3" cx="710.0312839281868" cy="328.7174490627289" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="191" data-strategy="3" cx="755.4141803869769" cy="322.8253265113766" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="192" data-strategy="3" cx="728.6995363073688" cy="320.33128993329103" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="193" data-strategy="3" cx="758.204868746594" cy="314.6708805199924" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="194" data-strategy="3" cx="714.9286626725273" cy="313.99359857554776" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="196" data-strategy="3" cx="751.9269723444567" cy="322.70581852101657" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="197" data-strategy="3" cx="746.8886035006894" cy="312.8854994051389" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="198" data-strategy="3" cx="717.8282454984446" cy="317.68472600842915" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="199" data-strategy="3" cx="744.0205526069169" cy="304.3774108316969" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="201" data-strategy="3" cx="725.1200913790987" cy="285.26278377475626" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="203" data-strategy="3" cx="741.7884819985254" cy="272.48401911769014" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="204" data-strategy="3" cx="743.4007871660925" cy="286.4954700620579" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="205" data-strategy="3" cx="723.5500893105819" cy="281.58537411027265" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="206" data-strategy="3" cx="708.6781429162417" cy="312.1865127060393" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="208" data-strategy="3" cx="761.5634642872329" cy="304.6437824579456" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="209" data-strategy="3" cx="729.4016072834938" cy="303.64518392631476" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="210" data-strategy="3" cx="736.6623024377069" cy="308.54851026030684" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="211" data-strategy="3" cx="726.345681545693" cy="304.3622071829553" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="213" data-strategy="3" cx="772.1120760639213" cy="292.6259004874741" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="214" data-strategy="3" cx="773.4254303184671" cy="309.16680519347705" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="215" data-strategy="3" cx="744.5955612265445" cy="306.0432402239334" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="216" data-strategy="3" cx="735.2773341035415" cy="316.4784452627472" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="218" data-strategy="3" cx="779.6496172301028" cy="309.63488981833535" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="219" data-strategy="3" cx="774.7308867495201" cy="306.2517043430374" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="220" data-strategy="3" cx="752.5299499159715" cy="307.3940615063319" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="221" data-strategy="3" cx="758.1644922820402" cy="300.0383537994288" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="223" data-strategy="3" cx="780.0791408971955" cy="329.5559028131755" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="224" data-strategy="3" cx="791.1452654914542" cy="330.3701299068663" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="225" data-strategy="3" cx="744.0261317882192" cy="325.7954989626914" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="226" data-strategy="3" cx="780.7901048689578" cy="336.20140796165447" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="227" data-strategy="3" cx="721.5782358021829" cy="348.47961431852764" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="228" data-strategy="3" cx="765.7176229271696" cy="353.0798488691135" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="229" data-strategy="3" cx="722.1441191329019" cy="353.7532072799811" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="230" data-strategy="3" cx="726.3180560399916" cy="342.84206879846596" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="231" data-strategy="3" cx="731.4158712233402" cy="341.43476790343425" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="232" data-strategy="3" cx="747.2025635918291" cy="331.246929049804" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="233" data-strategy="3" cx="706.0628644844966" cy="341.69571284223434" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="234" data-strategy="3" cx="751.8853925565838" cy="325.2238703213807" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="235" data-strategy="3" cx="735.052369568159" cy="318.70435855956777" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="236" data-strategy="3" cx="719.3193119089937" cy="300.2912979837712" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="237" data-strategy="3" cx="736.2433315466808" cy="318.9677848882302" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="238" data-strategy="3" cx="726.4991723534625" cy="336.2407061605432" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="239" data-strategy="3" cx="774.5228409883553" cy="317.6887161660351" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="240" data-strategy="3" cx="745.092748607771" cy="313.3221760564976" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="241" data-strategy="3" cx="730.3438032452851" cy="310.45451815004935" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="242" data-strategy="3" cx="740.871670133827" cy="315.620582488785" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="243" data-strategy="3" cx="773.1634277491938" cy="326.05480253901715" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="244" data-strategy="3" cx="747.482478156669" cy="317.7255437946847" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="245" data-strategy="3" cx="784.3005933827678" cy="308.2607581724026" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="246" data-strategy="3" cx="759.5260939344486" cy="310.9004727781476" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="247" data-strategy="3" cx="768.9036380004574" cy="318.1820459987465" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="248" data-strategy="3" cx="717.7777438533378" cy="339.95824859189133" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="249" data-strategy="3" cx="725.5735349937132" cy="329.9520679666651" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="250" data-strategy="3" cx="716.3390780517966" cy="325.562943650669" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="251" data-strategy="3" cx="782.2362967665506" cy="312.1530824340643" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="252" data-strategy="3" cx="746.8853655002085" cy="295.7931691757227" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="253" data-strategy="3" cx="763.8782730768752" cy="315.82383170438857" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="254" data-strategy="3" cx="730.5418456168342" cy="315.9772901051953" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="255" data-strategy="3" cx="745.9967574248699" cy="327.1911997344131" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="256" data-strategy="3" cx="765.2910232584599" cy="331.1888673485624" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="257" data-strategy="3" cx="739.4644754824586" cy="337.631655907716" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="258" data-strategy="3" cx="716.0659342652948" cy="338.0840786779782" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="259" data-strategy="3" cx="769.4990362509651" cy="352.95834643620236" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="261" data-strategy="3" cx="724.6870842797615" cy="338.03457006443585" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="262" data-strategy="3" cx="729.3916307873279" cy="353.1483814656013" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="263" data-strategy="3" cx="771.0154445671905" cy="356.9904318600524" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="264" data-strategy="3" cx="722.2139709983401" cy="364.98986052203065" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="265" data-strategy="3" cx="788.260505397723" cy="357.18162004319913" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="266" data-strategy="3" cx="705.7883349483792" cy="353.1176882594324" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="267" data-strategy="3" cx="715.1713854627674" cy="356.3258361579996" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="268" data-strategy="3" cx="737.2717225481741" cy="354.47333157145783" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="269" data-strategy="3" cx="739.1482992910304" cy="348.9267504077795" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="270" data-strategy="3" cx="707.0787449925244" cy="347.62898532054874" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="271" data-strategy="3" cx="708.4078952768053" cy="348.3656787888843" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="272" data-strategy="3" cx="773.3735385940066" cy="360.0590861174787" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="273" data-strategy="3" cx="790.4981337990299" cy="352.6168174890987" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="274" data-strategy="3" cx="747.2838746477129" cy="354.40713866634906" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="275" data-strategy="3" cx="771.0155637445312" cy="363.874436240879" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="276" data-strategy="3" cx="771.1196323232731" cy="366.5138628648248" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="277" data-strategy="3" cx="726.1929733188127" cy="370.052487698636" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="278" data-strategy="3" cx="749.0973994008475" cy="361.6577958526138" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="279" data-strategy="3" cx="709.8713039292455" cy="400.1345132903614" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="280" data-strategy="3" cx="741.1491068704618" cy="400.5097672186813" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="281" data-strategy="3" cx="758.7276264561029" cy="425.47593378077374" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="282" data-strategy="3" cx="716.7157950320442" cy="401.5267568135212" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="283" data-strategy="3" cx="713.9997720416081" cy="393.9438634049378" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="284" data-strategy="3" cx="749.7730194710994" cy="402.81617939494504" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="285" data-strategy="3" cx="733.2135707665878" cy="385.25924983380287" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="286" data-strategy="3" cx="749.1982409598373" cy="380.5194123532459" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="287" data-strategy="3" cx="775.9604518753839" cy="379.8327022403273" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="288" data-strategy="3" cx="782.5153763403803" cy="376.34753147696085" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="289" data-strategy="3" cx="790.8014082035597" cy="407.54539058825713" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="290" data-strategy="3" cx="768.0237630367415" cy="400.37937147290523" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="291" data-strategy="3" cx="706.3743943659596" cy="386.5721002002298" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="292" data-strategy="3" cx="757.8558045513081" cy="381.60499638211616" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="293" data-strategy="3" cx="783.4388456223389" cy="379.2484876301612" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="294" data-strategy="3" cx="788.1878442461027" cy="403.55696188057533" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="295" data-strategy="3" cx="772.2231479344699" cy="384.52397146441723" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="296" data-strategy="3" cx="726.7346546445365" cy="378.6194125913419" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="297" data-strategy="3" cx="766.5921987099974" cy="366.717757059005" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="298" data-strategy="3" cx="722.4231318390121" cy="358.9836286579013" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="299" data-strategy="3" cx="734.7848930255894" cy="366.01422884096735" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="300" data-strategy="3" cx="760.8368250057605" cy="365.9803432551254" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="301" data-strategy="3" cx="722.8931706297869" cy="357.4370593536724" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="304" data-strategy="3" cx="758.5411687816585" cy="285.0103012632841" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="305" data-strategy="3" cx="763.6144926104531" cy="300.2050352502682" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="306" data-strategy="3" cx="783.102790395666" cy="295.6527774470533" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="307" data-strategy="3" cx="765.3063716342201" cy="298.0493284176684" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="308" data-strategy="3" cx="717.4687196958763" cy="306.9477999490585" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="309" data-strategy="3" cx="790.2518521757758" cy="298.2603286822004" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="310" data-strategy="3" cx="778.4675007963714" cy="290.8389206744531" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="311" data-strategy="3" cx="783.5341934984136" cy="287.4020600450052" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="312" data-strategy="3" cx="768.247665901278" cy="296.417185372462" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="313" data-strategy="3" cx="786.6176677767563" cy="291.72205002052254" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="314" data-strategy="3" cx="712.7173730801035" cy="302.1021880836184" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="315" data-strategy="3" cx="744.119392173194" cy="304.49867787563846" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="316" data-strategy="3" cx="776.3627959373233" cy="299.1320508465768" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="317" data-strategy="3" cx="744.0104363561725" cy="309.0216113938629" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="318" data-strategy="3" cx="706.361867160323" cy="298.5272719746505" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="319" data-strategy="3" cx="759.0215918429018" cy="285.42049652584865" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="320" data-strategy="3" cx="728.1775268377274" cy="293.5020398365491" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="321" data-strategy="3" cx="720.7213073649364" cy="292.6468359721574" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="322" data-strategy="3" cx="723.4404164023732" cy="302.3333704629014" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="323" data-strategy="3" cx="734.3762007408766" cy="292.98894707667387" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="324" data-strategy="3" cx="732.1645031306911" cy="268.3141078197348" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="325" data-strategy="3" cx="714.9707272012616" cy="279.74049833765866" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="326" data-strategy="3" cx="751.2069007230639" cy="284.68943298667887" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="327" data-strategy="3" cx="774.1698371408555" cy="294.34345447840764" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="328" data-strategy="3" cx="744.1299492404534" cy="292.24227046312694" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="329" data-strategy="3" cx="715.7072763570976" cy="285.96242544984307" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="330" data-strategy="3" cx="738.5678646654096" cy="300.39875080917625" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="331" data-strategy="3" cx="729.8828791015594" cy="294.67408418923884" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="332" data-strategy="3" cx="768.8509584288381" cy="296.34837961528086" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="333" data-strategy="3" cx="723.6729412725175" cy="286.8977282028609" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="334" data-strategy="3" cx="720.0858352047431" cy="277.09974897392" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="335" data-strategy="3" cx="751.7005318067477" cy="281.1728825380279" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="336" data-strategy="3" cx="766.2269957152354" cy="280.3470411979223" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="337" data-strategy="3" cx="717.4392734761988" cy="271.2162576854607" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="338" data-strategy="3" cx="722.6733828022913" cy="270.4010589498703" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="339" data-strategy="3" cx="752.5028950130949" cy="268.35642795889896" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="340" data-strategy="3" cx="755.093232186406" cy="260.18262920801624" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="341" data-strategy="3" cx="720.110663817886" cy="265.76740587731285" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="342" data-strategy="3" cx="746.3858140416343" cy="281.4756441974162" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="343" data-strategy="3" cx="764.7210672684662" cy="279.41729000581114" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="344" data-strategy="3" cx="721.2742810812364" cy="267.81197077794207" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="345" data-strategy="3" cx="790.9559898991545" cy="266.4311875501822" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="346" data-strategy="3" cx="721.0252312821951" cy="272.76508386057964" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="347" data-strategy="3" cx="723.1110308159425" cy="284.22646020901806" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="348" data-strategy="3" cx="756.7482245200905" cy="279.2533083476877" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="349" data-strategy="3" cx="789.1483058188034" cy="278.0451286920969" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="395" data-strategy="3" cx="712.5899000236429" cy="315.0807850137304" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="396" data-strategy="3" cx="740.287957398502" cy="289.05230837197894" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="397" data-strategy="3" cx="712.8735372165386" cy="248.06371722667808" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="398" data-strategy="3" cx="739.633578446601" cy="276.98930843224025" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="399" data-strategy="3" cx="763.1754628259563" cy="292.70126904954174" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="400" data-strategy="3" cx="766.2978392285189" cy="295.6847252774582" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="401" data-strategy="3" cx="774.4898971993252" cy="303.165308669487" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="402" data-strategy="3" cx="727.3842753647492" cy="295.58396815337335" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="403" data-strategy="3" cx="764.4125346232213" cy="307.35703590956354" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="404" data-strategy="3" cx="782.8443017353289" cy="317.16762517692416" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="405" data-strategy="3" cx="736.3740442832328" cy="287.697897458179" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="407" data-strategy="3" cx="751.4376326505261" cy="237.51029393738636" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="408" data-strategy="3" cx="765.9787158811199" cy="245.20198875325624" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="409" data-strategy="3" cx="732.8351867943979" cy="274.54171759743843" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="410" data-strategy="3" cx="724.5165647368549" cy="302.9197285020668" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="412" data-strategy="3" cx="741.047542929137" cy="257.76987376019974" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="413" data-strategy="3" cx="735.413155251587" cy="270.66560374940445" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="414" data-strategy="3" cx="728.9594851289124" cy="271.0280153494943" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="415" data-strategy="3" cx="758.2951648762022" cy="254.00409409240814" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="417" data-strategy="3" cx="771.6873383724349" cy="241.60145861681198" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="418" data-strategy="3" cx="741.1863952312674" cy="231.22662383850803" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="419" data-strategy="3" cx="763.0823083877058" cy="245.3290334644937" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="420" data-strategy="3" cx="755.4592304793654" cy="255.2119869324304" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="422" data-strategy="3" cx="790.4358488068581" cy="221.81426359785988" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="423" data-strategy="3" cx="757.0996347476204" cy="214.12468137211567" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="424" data-strategy="3" cx="734.0130177235432" cy="230.92423709482946" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="425" data-strategy="3" cx="740.838072169107" cy="250.35058602545135" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="427" data-strategy="3" cx="722.5716759180184" cy="247.64812132631752" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="428" data-strategy="3" cx="736.7690884649403" cy="237.49322547802066" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="429" data-strategy="3" cx="717.0724977999432" cy="261.85496453453237" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="430" data-strategy="3" cx="723.1290164382946" cy="251.95329331134528" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="431" data-strategy="3" cx="730.0270140018899" cy="261.9063983591968" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="443" data-strategy="3" cx="768.5815352183342" cy="341.44650253231595" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="444" data-strategy="3" cx="721.9103870443572" cy="335.69021997113214" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="445" data-strategy="3" cx="787.6627168919291" cy="325.330638602363" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="446" data-strategy="3" cx="755.7238934207162" cy="347.8166241589366" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="447" data-strategy="3" cx="754.7022356626652" cy="326.048346499215" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="448" data-strategy="3" cx="709.082153471842" cy="321.4536127967612" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="449" data-strategy="3" cx="751.7303295048881" cy="331.9469850141067" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="450" data-strategy="3" cx="764.7032533366994" cy="322.961890959897" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="451" data-strategy="3" cx="744.5962832809315" cy="335.34914879853574" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="452" data-strategy="3" cx="750.5779063241808" cy="326.97134101481623" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="453" data-strategy="3" cx="721.7985394494033" cy="336.2648547827402" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="454" data-strategy="3" cx="782.8307020829116" cy="351.42620480132297" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="455" data-strategy="3" cx="710.0266496930171" cy="339.13347861320887" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="456" data-strategy="3" cx="726.3264887681161" cy="360.9820293683702" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="457" data-strategy="3" cx="772.5715573818875" cy="330.3085345919481" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="458" data-strategy="3" cx="761.2298566274762" cy="307.5583152328576" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="459" data-strategy="3" cx="750.5870220887101" cy="319.64404855284016" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="460" data-strategy="3" cx="707.6164610326841" cy="289.2371704413546" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="461" data-strategy="3" cx="780.5048994535609" cy="296.3591603067682" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="462" data-strategy="3" cx="718.3383106445514" cy="296.2212872538908" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="463" data-strategy="3" cx="749.3736379877213" cy="281.42052239454597" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="465" data-strategy="3" cx="763.967975335032" cy="303.5253254208925" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="466" data-strategy="3" cx="789.0306858701766" cy="301.8270302923067" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="467" data-strategy="3" cx="787.8406443524572" cy="280.45031786514085" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="468" data-strategy="3" cx="739.9769704099255" cy="265.0487889451998" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="470" data-strategy="3" cx="705.4886542321716" cy="280.8674908699096" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="471" data-strategy="3" cx="775.2023232855582" cy="282.9397941309502" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="472" data-strategy="3" cx="755.3678206687894" cy="268.7449265327509" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="475" data-strategy="3" cx="706.3684000767093" cy="246.094383756581" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="476" data-strategy="3" cx="784.6345925418202" cy="227.42950655262112" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="477" data-strategy="3" cx="713.2021133968237" cy="202.19065607191118" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="480" data-strategy="3" cx="718.225610705428" cy="225.9990593969373" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="504" data-strategy="3" cx="760.1474769802254" cy="292.60935290378865" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="505" data-strategy="3" cx="773.5120169515562" cy="273.2312487055302" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="506" data-strategy="3" cx="777.2415032107951" cy="276.60463735833866" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="507" data-strategy="3" cx="791.8144458179627" cy="292.7985896153596" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="508" data-strategy="3" cx="729.8975570390201" cy="259.43639072913044" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="509" data-strategy="3" cx="722.3597734177173" cy="226.01573666502293" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="510" data-strategy="3" cx="758.0905228202373" cy="226.26328629616614" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="511" data-strategy="3" cx="763.7743425697834" cy="197.98788294938353" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="512" data-strategy="3" cx="739.4949395526318" cy="226.58522516401868" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="513" data-strategy="3" cx="740.2919231971242" cy="215.4319503516168" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="514" data-strategy="3" cx="748.693549038567" cy="248.39405430822012" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="515" data-strategy="3" cx="729.011625561469" cy="223.80013195135825" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="516" data-strategy="3" cx="783.9283888269548" cy="227.66795243853431" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="517" data-strategy="3" cx="734.7056530380171" cy="266.4238334699312" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="518" data-strategy="3" cx="730.5278150909758" cy="285.00244777785724" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="519" data-strategy="3" cx="740.5623547787151" cy="316.24511989488417" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="520" data-strategy="3" cx="754.2026247238321" cy="302.58155553495345" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="521" data-strategy="3" cx="784.6328216755118" cy="300.01712491426207" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="522" data-strategy="3" cx="718.8691462049653" cy="305.2844920449019" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="523" data-strategy="3" cx="775.0462532155123" cy="225.0340792410612" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="524" data-strategy="3" cx="719.7535584035454" cy="215.05555543040964" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="525" data-strategy="3" cx="739.1595635388936" cy="193.75442040268132" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="527" data-strategy="3" cx="781.5050904461704" cy="183.94078200625432" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="528" data-strategy="3" cx="721.8123337010126" cy="216.72922519368095" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="529" data-strategy="3" cx="770.3033554840305" cy="195.76481381971408" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="530" data-strategy="3" cx="705.7989896500226" cy="168.77890221302013" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="531" data-strategy="3" cx="744.1372273696111" cy="144.74706514688967" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="532" data-strategy="3" cx="756.507143608442" cy="158.0607837907989" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="533" data-strategy="3" cx="791.578748419781" cy="182.91289370419108" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="535" data-strategy="3" cx="732.1776191673066" cy="192.0106488001132" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="536" data-strategy="3" cx="770.9864185029867" cy="183.06050713499104" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="537" data-strategy="3" cx="765.739649811973" cy="155.55822934071733" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="538" data-strategy="3" cx="755.6280008891999" cy="180.86381397724108" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="539" data-strategy="3" cx="774.5310893055009" cy="183.2018635114508" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="540" data-strategy="3" cx="754.5796342017102" cy="123.08243614341012" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="541" data-strategy="3" cx="735.0886220578582" cy="113.0216204482081" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="542" data-strategy="3" cx="717.7881175333833" cy="84.48373253844312" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="543" data-strategy="3" cx="777.5272606945815" cy="104.16601781602841" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="544" data-strategy="3" cx="708.0043452951405" cy="171.29148757028534" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="545" data-strategy="3" cx="752.101669743591" cy="140.54881247480182" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="546" data-strategy="3" cx="762.779134843721" cy="161.75510502867186" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="547" data-strategy="3" cx="714.7769494439667" cy="180.3024507057205" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="548" data-strategy="3" cx="715.1435460365235" cy="198.56734027152422" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="549" data-strategy="3" cx="753.9816196322931" cy="262.44631430056927" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="550" data-strategy="3" cx="737.0367435978503" cy="267.1497722362641" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="551" data-strategy="3" cx="710.5433355046093" cy="264.0561772111434" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="552" data-strategy="3" cx="783.4868929064108" cy="263.25813128994105" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="553" data-strategy="3" cx="791.9550958816321" cy="262.2714059947579" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="554" data-strategy="3" cx="712.0977233684491" cy="282.5291719378302" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="555" data-strategy="3" cx="715.6636455912844" cy="262.5521002040214" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="556" data-strategy="3" cx="748.9529730338583" cy="241.2074137930478" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="557" data-strategy="3" cx="720.9052382328987" cy="219.48782097357446" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="558" data-strategy="3" cx="782.100216488468" cy="259.14731843020536" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="559" data-strategy="3" cx="761.5485214439068" cy="231.33442091388358" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="560" data-strategy="3" cx="729.1586200094248" cy="223.7440967103646" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="561" data-strategy="3" cx="745.2458032784737" cy="225.56429601951507" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="562" data-strategy="3" cx="752.1612256471033" cy="243.19321781555135" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="563" data-strategy="3" cx="785.205668129851" cy="280.16559635477046" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="564" data-strategy="3" cx="775.3033897536458" cy="296.535099300521" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="565" data-strategy="3" cx="725.1610301372422" cy="334.442673373621" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="566" data-strategy="3" cx="706.5753430865647" cy="305.2272369576938" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="567" data-strategy="3" cx="732.2891389161142" cy="337.81625349584414" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="568" data-strategy="3" cx="780.9777042384969" cy="354.91666879686835" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="569" data-strategy="3" cx="743.2622994562691" cy="334.4906651473955" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="570" data-strategy="3" cx="743.7888752074621" cy="297.57456834987147" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="571" data-strategy="3" cx="708.446717172622" cy="297.2823608930114" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="572" data-strategy="3" cx="721.3856746394695" cy="260.23012850738917" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="573" data-strategy="3" cx="705.6206457107573" cy="225.30029016758598" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="648" data-strategy="3" cx="782.3254887979209" cy="315.98098243208796" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="649" data-strategy="3" cx="726.3968814319525" cy="315.6596099824999" r="3" fill="#9333ea" fill-opacity=".65"></circle><circle class="point" data-window="650" data-strategy="3" cx="780.3449401654317" cy="310.7508378562278" r="3" fill="#9333ea" fill-opacity=".65"></circle><line x1="704.75" y1="301.1746415455968" x2="792.75" y2="301.1746415455968" stroke="#be185d" stroke-width="3.5" class="median-line"/><text x="748.75" y="572" text-anchor="middle" >C｜動能＋低波動</text><text x="748.75" y="596" text-anchor="middle" class="median-label" style="fill:#be185d">中位數 4.29%</text><line x1="939.25" y1="405.3814981972799" x2="939.25" y2="187.46019891647805" stroke="#ea580c" stroke-width="2" /><rect x="901.25" y="268.8777217379205" width="76" height="56.05879633994766" fill="#ea580c" fill-opacity=".12" stroke="#ea580c"/><line x1="901.25" y1="405.3814981972799" x2="977.25" y2="405.3814981972799" stroke="#ea580c" stroke-width="2" /><line x1="901.25" y1="187.46019891647805" x2="977.25" y2="187.46019891647805" stroke="#ea580c" stroke-width="2" /><circle class="point" data-window="18" data-strategy="4" cx="903.5984405122805" cy="200.258830112859" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="19" data-strategy="4" cx="955.1119432124193" cy="206.8880910494048" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="20" data-strategy="4" cx="973.847360759443" cy="229.4635402560218" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="21" data-strategy="4" cx="967.7527862321097" cy="218.4274535430264" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="22" data-strategy="4" cx="912.7333705402775" cy="207.12105674534186" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="23" data-strategy="4" cx="956.0220188659737" cy="228.70729571058476" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="24" data-strategy="4" cx="975.4121089357388" cy="256.3886794629405" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="25" data-strategy="4" cx="910.608916401575" cy="248.31186463475638" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="26" data-strategy="4" cx="944.3489427680171" cy="246.88301131267133" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="27" data-strategy="4" cx="965.5400322022275" cy="264.79244654235583" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="28" data-strategy="4" cx="956.7888856516469" cy="291.59640301283457" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="29" data-strategy="4" cx="913.5394121571234" cy="300.303504453012" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="30" data-strategy="4" cx="912.0064322202882" cy="281.25146061083746" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="31" data-strategy="4" cx="919.3449379949324" cy="274.19802188080484" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="32" data-strategy="4" cx="906.7821080984453" cy="300.11772060125793" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="33" data-strategy="4" cx="906.431339308571" cy="295.9862286148632" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="34" data-strategy="4" cx="978.121707932462" cy="297.0492603246255" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="35" data-strategy="4" cx="960.3478429013365" cy="306.47383047606576" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="36" data-strategy="4" cx="975.276670324545" cy="293.59074038094064" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="37" data-strategy="4" cx="930.7103160229053" cy="294.4289773918125" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="38" data-strategy="4" cx="896.4086450662454" cy="308.29648520702307" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="40" data-strategy="4" cx="942.8440502331931" cy="309.03352598082796" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="41" data-strategy="4" cx="972.1020965396139" cy="329.9199864180357" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="42" data-strategy="4" cx="913.4202635108388" cy="329.78466717324653" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="43" data-strategy="4" cx="938.2367580590831" cy="294.25694769747196" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="45" data-strategy="4" cx="953.1960547612946" cy="334.2379611141712" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="46" data-strategy="4" cx="958.9692713249437" cy="340.9644070376442" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="47" data-strategy="4" cx="918.9464807645566" cy="302.06254270811866" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="48" data-strategy="4" cx="937.01169926092" cy="321.00130415332717" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="50" data-strategy="4" cx="941.202677496097" cy="356.0103880087088" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="51" data-strategy="4" cx="960.6757750477448" cy="329.19247621067444" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="52" data-strategy="4" cx="965.8295863207209" cy="334.5721750272313" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="53" data-strategy="4" cx="945.194818232438" cy="324.28283442397475" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="55" data-strategy="4" cx="918.8211090162322" cy="315.8141563291598" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="56" data-strategy="4" cx="932.1214543715823" cy="318.80779224877614" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="57" data-strategy="4" cx="935.5751839216558" cy="318.42306429425236" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="58" data-strategy="4" cx="963.350826274041" cy="322.0086608828883" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="60" data-strategy="4" cx="906.3485429730549" cy="330.0516371612211" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="61" data-strategy="4" cx="924.4075055238017" cy="340.5696506044584" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="62" data-strategy="4" cx="969.7289683203148" cy="340.41100122705126" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="63" data-strategy="4" cx="898.2103440542064" cy="353.2628385134242" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="64" data-strategy="4" cx="935.4148232999344" cy="338.62309063936755" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="65" data-strategy="4" cx="914.4088922193029" cy="335.9430074682126" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="66" data-strategy="4" cx="959.3517644950813" cy="332.57928472010235" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="68" data-strategy="4" cx="913.5799524771024" cy="318.58289871885444" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="69" data-strategy="4" cx="976.477622261715" cy="324.0401417496568" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="71" data-strategy="4" cx="926.1588054084775" cy="297.9381196940056" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="73" data-strategy="4" cx="973.0254128534772" cy="274.9590252848633" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="74" data-strategy="4" cx="911.2175364458694" cy="288.5402400289204" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="76" data-strategy="4" cx="932.1306128482299" cy="293.16968633043194" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="78" data-strategy="4" cx="934.3736017971784" cy="299.70416931246876" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="79" data-strategy="4" cx="964.0712161199058" cy="296.6582729849628" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="81" data-strategy="4" cx="963.4729741957267" cy="301.71137077457735" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="84" data-strategy="4" cx="975.0293833325373" cy="283.9470619839606" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="86" data-strategy="4" cx="903.1403292325186" cy="266.1228573054301" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="89" data-strategy="4" cx="910.5604807010205" cy="267.7525936979846" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="91" data-strategy="4" cx="923.5406671645892" cy="267.6543158027946" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="92" data-strategy="4" cx="950.358630195363" cy="279.1140976629914" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="94" data-strategy="4" cx="919.05860974432" cy="283.22421290979867" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="95" data-strategy="4" cx="963.4414720701327" cy="287.4301776866438" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="97" data-strategy="4" cx="931.1451142771093" cy="301.9113647924475" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="99" data-strategy="4" cx="896.5761139657953" cy="295.5995208649132" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="100" data-strategy="4" cx="917.9077982370125" cy="298.26563019451453" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="102" data-strategy="4" cx="943.1370144365195" cy="290.02797838837466" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="104" data-strategy="4" cx="931.4863677216189" cy="296.36446373215705" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="105" data-strategy="4" cx="910.2762586710577" cy="293.73072221116996" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="107" data-strategy="4" cx="972.1541131816211" cy="288.1714344610563" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="108" data-strategy="4" cx="899.7335811532241" cy="302.1444504607863" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="109" data-strategy="4" cx="980.3877083282904" cy="298.9988720856718" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="110" data-strategy="4" cx="934.8827911949837" cy="324.2994555457098" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="112" data-strategy="4" cx="957.9865884347882" cy="348.7654949442149" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="113" data-strategy="4" cx="934.4682919208168" cy="334.805313770917" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="168" data-strategy="4" cx="964.9721096334373" cy="273.0359529635259" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="170" data-strategy="4" cx="961.7966761892949" cy="276.1282009129651" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="171" data-strategy="4" cx="979.8346157901325" cy="282.03923939388824" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="172" data-strategy="4" cx="901.1300267745193" cy="285.1786689344151" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="173" data-strategy="4" cx="967.8123776265373" cy="290.890515637355" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="175" data-strategy="4" cx="904.4886982780048" cy="305.23453158281313" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="176" data-strategy="4" cx="914.8962445184409" cy="326.4026609805685" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="177" data-strategy="4" cx="979.5879318491263" cy="329.71857602879254" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="178" data-strategy="4" cx="949.6501404113151" cy="330.33304457447485" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="179" data-strategy="4" cx="918.0590759094428" cy="327.19200820933037" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="180" data-strategy="4" cx="977.9839447583244" cy="326.10063639912386" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="181" data-strategy="4" cx="902.3180010359771" cy="317.2831494672919" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="182" data-strategy="4" cx="923.5576095866857" cy="309.90917841292656" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="183" data-strategy="4" cx="896.6220015161991" cy="308.02725569318005" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="184" data-strategy="4" cx="961.548242654334" cy="293.364388653712" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="186" data-strategy="4" cx="914.5529396151177" cy="299.12195046076414" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="187" data-strategy="4" cx="943.6511668338431" cy="308.4874767149352" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="188" data-strategy="4" cx="951.3425850836256" cy="322.01351872524737" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="189" data-strategy="4" cx="919.7863111257997" cy="322.97039874881665" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="191" data-strategy="4" cx="951.8139066974203" cy="328.1956964706194" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="192" data-strategy="4" cx="943.5865493455547" cy="320.545173556617" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="193" data-strategy="4" cx="974.0011622210075" cy="312.6751989305792" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="194" data-strategy="4" cx="911.6745148350392" cy="317.88207902237644" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="196" data-strategy="4" cx="936.8692159045598" cy="322.1098653916162" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="197" data-strategy="4" cx="916.361992497123" cy="323.6974722616841" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="198" data-strategy="4" cx="971.2502998830527" cy="336.6500599875879" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="199" data-strategy="4" cx="908.5945170633543" cy="329.10054392057236" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="201" data-strategy="4" cx="976.9815665401884" cy="311.9629417389354" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="203" data-strategy="4" cx="979.2774598504841" cy="299.0926181791224" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="204" data-strategy="4" cx="916.4523881310938" cy="306.5031176824697" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="205" data-strategy="4" cx="972.5486511040197" cy="306.1823839013954" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="206" data-strategy="4" cx="947.3536883152628" cy="324.8655301521675" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="208" data-strategy="4" cx="919.191789811435" cy="322.39935542247423" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="209" data-strategy="4" cx="896.6623995169737" cy="314.6679719958029" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="210" data-strategy="4" cx="902.9468234996283" cy="330.78201695152325" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="211" data-strategy="4" cx="921.2690836528349" cy="311.6919823524219" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="213" data-strategy="4" cx="928.4960815448051" cy="294.97341943220715" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="214" data-strategy="4" cx="981.5588809049433" cy="294.34463246522563" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="215" data-strategy="4" cx="925.4890305563422" cy="297.7250854670066" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="216" data-strategy="4" cx="955.0114712660381" cy="306.87791251022105" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="218" data-strategy="4" cx="963.8847705391182" cy="297.2065801918232" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="219" data-strategy="4" cx="943.939677153689" cy="297.04776520768866" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="220" data-strategy="4" cx="897.4261626925502" cy="288.18114418503836" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="221" data-strategy="4" cx="952.1358399199825" cy="277.4842330786647" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="223" data-strategy="4" cx="914.2889196804615" cy="318.1407578821344" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="224" data-strategy="4" cx="897.5150586472877" cy="323.21165268206556" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="225" data-strategy="4" cx="979.8038152364859" cy="325.54152149543506" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="226" data-strategy="4" cx="971.651299363255" cy="319.289365596045" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="227" data-strategy="4" cx="944.0874823791887" cy="327.730880554518" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="228" data-strategy="4" cx="961.6929218485117" cy="336.0280720744477" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="229" data-strategy="4" cx="943.693376227999" cy="338.9964117936463" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="230" data-strategy="4" cx="970.9564664211409" cy="333.9186378547221" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="231" data-strategy="4" cx="947.1794498637764" cy="335.4523305606091" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="232" data-strategy="4" cx="950.7294231825974" cy="308.931745509184" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="233" data-strategy="4" cx="960.0765247430222" cy="315.27760817889293" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="234" data-strategy="4" cx="958.0622100811432" cy="306.75459100376526" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="235" data-strategy="4" cx="970.6574660473939" cy="302.8951256888818" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="236" data-strategy="4" cx="966.5393529720297" cy="297.7574130419685" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="237" data-strategy="4" cx="910.4829660282809" cy="329.1548344030103" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="238" data-strategy="4" cx="933.9962827742743" cy="336.89034277664143" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="239" data-strategy="4" cx="929.5151701566333" cy="320.62558359166775" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="240" data-strategy="4" cx="956.1263820210984" cy="317.64758191312364" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="241" data-strategy="4" cx="982.0339382340377" cy="305.2126361838999" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="242" data-strategy="4" cx="931.6952012714551" cy="319.15427559197144" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="243" data-strategy="4" cx="978.8692112877263" cy="320.14085591123154" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="244" data-strategy="4" cx="906.8136771373279" cy="324.46594215965405" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="245" data-strategy="4" cx="925.5045264908252" cy="309.6157841678127" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="246" data-strategy="4" cx="938.1418599802541" cy="325.00750600356884" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="247" data-strategy="4" cx="924.5549597282279" cy="304.8617190662549" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="248" data-strategy="4" cx="953.7167404055087" cy="312.1409863286822" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="249" data-strategy="4" cx="905.7513014526926" cy="308.97750363857756" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="250" data-strategy="4" cx="958.3941999691931" cy="318.7456449332407" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="251" data-strategy="4" cx="953.9740272924693" cy="307.7896567334642" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="252" data-strategy="4" cx="920.0628546088708" cy="290.2598866496619" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="253" data-strategy="4" cx="958.8807087392353" cy="293.7637509286811" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="254" data-strategy="4" cx="907.1671469313817" cy="295.59234325676823" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="255" data-strategy="4" cx="952.5708900736109" cy="319.3901266320995" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="256" data-strategy="4" cx="936.9966140883395" cy="342.23077843130324" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="257" data-strategy="4" cx="905.093067463582" cy="329.5450669905207" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="258" data-strategy="4" cx="963.9892630547369" cy="334.58047451752265" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="259" data-strategy="4" cx="969.2414306048006" cy="341.4617002676253" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="261" data-strategy="4" cx="950.5172406399388" cy="341.3631872914838" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="262" data-strategy="4" cx="920.3201909630998" cy="355.5698699607925" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="263" data-strategy="4" cx="937.3837974772573" cy="363.5733143727064" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="264" data-strategy="4" cx="935.1881263247486" cy="373.4520289054761" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="265" data-strategy="4" cx="939.9998131939776" cy="374.09080528100424" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="266" data-strategy="4" cx="921.6103003593848" cy="367.9949681881639" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="267" data-strategy="4" cx="957.3128041257842" cy="359.18811158873376" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="268" data-strategy="4" cx="945.632054126902" cy="369.2746115481914" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="269" data-strategy="4" cx="972.2303394728145" cy="350.439467126202" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="270" data-strategy="4" cx="943.5968238586471" cy="347.0647546104095" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="271" data-strategy="4" cx="973.9107473868453" cy="352.3874357235832" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="272" data-strategy="4" cx="920.6002953138793" cy="362.62616244192475" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="273" data-strategy="4" cx="906.2772119577146" cy="350.81575433818733" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="274" data-strategy="4" cx="968.6144243115482" cy="366.7376059999991" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="275" data-strategy="4" cx="904.3559843211779" cy="369.6201973873661" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="276" data-strategy="4" cx="955.9473095257429" cy="389.867309262765" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="277" data-strategy="4" cx="955.1410745085257" cy="385.6525575977211" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="278" data-strategy="4" cx="919.6253796792912" cy="372.2250119615852" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="279" data-strategy="4" cx="948.193726176473" cy="413.37609102563715" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="280" data-strategy="4" cx="923.8290747780522" cy="436.2095817812273" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="281" data-strategy="4" cx="972.481098063638" cy="487.0975422776548" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="282" data-strategy="4" cx="941.9515851666621" cy="464.7048360755737" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="283" data-strategy="4" cx="978.4627833761261" cy="449.2364710150089" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="284" data-strategy="4" cx="898.724479687208" cy="423.48638421255777" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="285" data-strategy="4" cx="931.7283495253046" cy="393.5577255298059" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="286" data-strategy="4" cx="949.9687302433298" cy="405.3814981972799" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="287" data-strategy="4" cx="907.3942679014253" cy="397.6160390887288" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="288" data-strategy="4" cx="971.9259326129534" cy="404.31540562183164" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="289" data-strategy="4" cx="898.622426908815" cy="400.49442576388617" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="290" data-strategy="4" cx="920.8937434859118" cy="420.88671207266856" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="291" data-strategy="4" cx="942.411770088744" cy="396.3752327594369" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="292" data-strategy="4" cx="949.919920112057" cy="401.38622802975635" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="293" data-strategy="4" cx="958.2449967771391" cy="398.23033749267796" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="294" data-strategy="4" cx="981.6247328184122" cy="398.3538630293916" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="295" data-strategy="4" cx="919.7737693823266" cy="385.5292254112782" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="296" data-strategy="4" cx="942.7018320267596" cy="393.91288721400105" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="297" data-strategy="4" cx="916.4528047299331" cy="365.47133541135895" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="298" data-strategy="4" cx="936.4814781905509" cy="367.9863970094091" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="299" data-strategy="4" cx="969.3477650915237" cy="346.1952720394505" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="300" data-strategy="4" cx="960.8062414995694" cy="346.0387784129502" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="301" data-strategy="4" cx="964.356860384734" cy="347.14934527927306" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="304" data-strategy="4" cx="915.4978329984266" cy="275.73825862503674" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="305" data-strategy="4" cx="923.8340140145863" cy="301.5728707532899" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="306" data-strategy="4" cx="952.3244272629688" cy="296.4428386831026" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="307" data-strategy="4" cx="969.5639808730575" cy="303.8393113526787" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="308" data-strategy="4" cx="927.3169042298953" cy="305.08825212321983" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="309" data-strategy="4" cx="905.699544178518" cy="301.81720740480137" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="310" data-strategy="4" cx="965.8312555294635" cy="294.52358258880747" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="311" data-strategy="4" cx="976.5562261105373" cy="303.96833412376253" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="312" data-strategy="4" cx="901.3734507304242" cy="292.8572555670498" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="313" data-strategy="4" cx="904.7240474249179" cy="289.4896763487766" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="314" data-strategy="4" cx="917.9645587925021" cy="305.01705041313437" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="315" data-strategy="4" cx="928.5471210412634" cy="295.3574522767584" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="316" data-strategy="4" cx="925.1682500557035" cy="297.3927571622527" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="317" data-strategy="4" cx="942.9352306550403" cy="309.5191420837488" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="318" data-strategy="4" cx="962.943649866496" cy="314.4910161098321" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="319" data-strategy="4" cx="912.5572473792793" cy="306.3848755951111" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="320" data-strategy="4" cx="977.1903008150596" cy="302.3318274309029" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="321" data-strategy="4" cx="962.2436292417015" cy="277.23228411159494" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="322" data-strategy="4" cx="956.4524509997499" cy="285.0778771703897" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="323" data-strategy="4" cx="929.6462044057831" cy="274.9159617990221" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="324" data-strategy="4" cx="921.9571038051423" cy="251.94802208610687" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="325" data-strategy="4" cx="950.6634341911091" cy="261.35160931366005" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="326" data-strategy="4" cx="950.7467010393148" cy="277.4472913104097" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="327" data-strategy="4" cx="946.7106232281886" cy="265.8073071701364" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="328" data-strategy="4" cx="930.253513164441" cy="283.055368135143" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="329" data-strategy="4" cx="948.2792862481276" cy="279.55946463086616" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="330" data-strategy="4" cx="946.6814795981475" cy="283.5339196208794" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="331" data-strategy="4" cx="975.0535126342224" cy="291.08347214623063" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="332" data-strategy="4" cx="947.26318651773" cy="289.5792802920173" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="333" data-strategy="4" cx="978.4773783813728" cy="273.2498180552519" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="334" data-strategy="4" cx="980.0282650601757" cy="269.79483348767275" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="335" data-strategy="4" cx="959.0673749287117" cy="266.94424703395424" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="336" data-strategy="4" cx="960.4222453739683" cy="266.8495959711951" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="337" data-strategy="4" cx="933.2431139354501" cy="258.8324568479994" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="338" data-strategy="4" cx="964.1228781815333" cy="271.3485492841463" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="339" data-strategy="4" cx="922.9083950328345" cy="266.0722356206469" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="340" data-strategy="4" cx="904.8472587646007" cy="254.54012738229144" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="341" data-strategy="4" cx="911.2311630121054" cy="271.79176160517443" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="342" data-strategy="4" cx="922.0605364114957" cy="297.55639410122905" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="343" data-strategy="4" cx="973.4084382395951" cy="274.5238945331138" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="344" data-strategy="4" cx="977.7375263516126" cy="258.75510561517297" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="345" data-strategy="4" cx="904.2877744910917" cy="263.7625998010766" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="346" data-strategy="4" cx="944.8731346665181" cy="273.18297512572553" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="347" data-strategy="4" cx="947.8550552140508" cy="301.9743937633949" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="348" data-strategy="4" cx="900.5473858469808" cy="294.4881797283599" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="349" data-strategy="4" cx="918.1118198708233" cy="274.79914285057" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="395" data-strategy="4" cx="930.9534900508633" cy="316.82209303201864" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="396" data-strategy="4" cx="955.0492787233329" cy="308.15793793139574" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="397" data-strategy="4" cx="926.3921055593878" cy="278.64955971512234" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="398" data-strategy="4" cx="973.6571841194643" cy="286.06639556507156" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="399" data-strategy="4" cx="980.2912456170433" cy="287.01702649914785" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="400" data-strategy="4" cx="963.9557553410613" cy="300.5458264617187" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="401" data-strategy="4" cx="933.8980771417954" cy="317.4464834430854" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="402" data-strategy="4" cx="898.4488184613116" cy="320.1629099746343" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="403" data-strategy="4" cx="955.955345211352" cy="320.1277243395275" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="404" data-strategy="4" cx="901.0734719483169" cy="329.2365596145285" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="405" data-strategy="4" cx="951.2467415398381" cy="314.1408099055091" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="407" data-strategy="4" cx="939.815660377515" cy="286.5742345560251" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="408" data-strategy="4" cx="964.400559712031" cy="284.14115515156266" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="409" data-strategy="4" cx="975.7752514354219" cy="307.07225845758285" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="410" data-strategy="4" cx="965.1277311770763" cy="337.0775303967071" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="412" data-strategy="4" cx="900.2832454321358" cy="300.1283463056516" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="413" data-strategy="4" cx="896.4091332386233" cy="310.0555207499612" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="414" data-strategy="4" cx="944.8042099629024" cy="298.6961092022285" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="415" data-strategy="4" cx="979.5768884343189" cy="252.5464260846136" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="417" data-strategy="4" cx="907.7324385807666" cy="248.06773765500193" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="418" data-strategy="4" cx="912.9106044911797" cy="237.02052179677327" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="419" data-strategy="4" cx="973.8982172000427" cy="237.52649257161855" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="420" data-strategy="4" cx="896.6672944620233" cy="225.7144479019503" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="422" data-strategy="4" cx="913.1937799600166" cy="234.7746327586999" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="423" data-strategy="4" cx="982.5941902948098" cy="223.86489841208223" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="424" data-strategy="4" cx="948.3736956377394" cy="231.1554132458158" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="425" data-strategy="4" cx="913.3950943906489" cy="234.9541053023627" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="427" data-strategy="4" cx="955.1973897528175" cy="254.40995313468883" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="428" data-strategy="4" cx="910.3639489091772" cy="230.7028211296396" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="429" data-strategy="4" cx="934.9280610410285" cy="244.5393907071033" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="430" data-strategy="4" cx="921.8393689841585" cy="244.82511076684438" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="431" data-strategy="4" cx="951.4303593386971" cy="257.86075242791003" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="443" data-strategy="4" cx="922.835020569092" cy="314.9264623576172" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="444" data-strategy="4" cx="940.0692047951428" cy="341.0692017702884" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="445" data-strategy="4" cx="937.3439291093304" cy="328.4322879742491" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="446" data-strategy="4" cx="972.8113770137107" cy="342.5871425877802" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="447" data-strategy="4" cx="957.483765064133" cy="324.13429925023183" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="448" data-strategy="4" cx="908.3018444562292" cy="287.45083908918787" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="449" data-strategy="4" cx="913.6818799738587" cy="301.22959588371776" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="450" data-strategy="4" cx="903.4561410467458" cy="291.10152503553337" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="451" data-strategy="4" cx="924.5878702385517" cy="302.5830364575681" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="452" data-strategy="4" cx="976.5663784267015" cy="308.56634625335676" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="453" data-strategy="4" cx="954.4040861565226" cy="326.43801655459214" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="454" data-strategy="4" cx="912.1379180470127" cy="350.57222852544567" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="455" data-strategy="4" cx="915.9736996348165" cy="329.81506925725006" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="456" data-strategy="4" cx="940.9768445107348" cy="339.1179152798746" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="457" data-strategy="4" cx="966.1669864408119" cy="317.9978063680336" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="458" data-strategy="4" cx="914.188033784844" cy="293.68885585208943" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="459" data-strategy="4" cx="936.4066224230971" cy="303.10625403611016" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="460" data-strategy="4" cx="933.6390578613733" cy="258.48106093893284" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="461" data-strategy="4" cx="976.5158099301307" cy="276.15191795140714" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="462" data-strategy="4" cx="971.6540067952559" cy="264.1039559293067" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="463" data-strategy="4" cx="925.835256815661" cy="254.20382042699333" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="465" data-strategy="4" cx="948.4084806872952" cy="268.87720883560235" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="466" data-strategy="4" cx="943.1498230610575" cy="248.77484184349854" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="467" data-strategy="4" cx="943.4313925196012" cy="235.2376553548858" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="468" data-strategy="4" cx="899.2794050726828" cy="219.37419895036837" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="470" data-strategy="4" cx="936.0967872081421" cy="257.3044709424263" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="471" data-strategy="4" cx="932.375273540278" cy="239.86988630214987" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="472" data-strategy="4" cx="918.0444007402791" cy="250.3228556401134" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="475" data-strategy="4" cx="975.7622221760699" cy="254.4860689021713" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="476" data-strategy="4" cx="920.2762122341092" cy="244.20080990898478" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="477" data-strategy="4" cx="912.0493890202323" cy="217.1184291473489" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="480" data-strategy="4" cx="920.0363718246193" cy="200.3766532277011" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="504" data-strategy="4" cx="921.8404751808347" cy="305.7872670129148" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="505" data-strategy="4" cx="963.0754928025701" cy="280.05964680475665" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="506" data-strategy="4" cx="975.8518268238611" cy="301.2343324753386" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="507" data-strategy="4" cx="899.0820245643541" cy="293.80556068733904" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="508" data-strategy="4" cx="920.9401574960438" cy="293.0547618759224" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="509" data-strategy="4" cx="902.0582278678739" cy="237.85746466641908" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="510" data-strategy="4" cx="924.9572746521656" cy="219.87196018095997" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="511" data-strategy="4" cx="968.8976962205153" cy="207.74174521548923" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="512" data-strategy="4" cx="920.164935751881" cy="187.46019891647805" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="513" data-strategy="4" cx="965.4813441130789" cy="167.85297223807868" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="514" data-strategy="4" cx="936.684142015429" cy="239.99272042361554" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="515" data-strategy="4" cx="948.2376081496718" cy="237.9635714252353" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="516" data-strategy="4" cx="919.2697393613938" cy="219.90527422265316" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="517" data-strategy="4" cx="929.591275293616" cy="234.4773201535003" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="518" data-strategy="4" cx="935.3769173631696" cy="270.8946309454076" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="519" data-strategy="4" cx="978.8569529244509" cy="318.47840410158824" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="520" data-strategy="4" cx="926.1960468625206" cy="293.94972959901224" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="521" data-strategy="4" cx="974.4980670753295" cy="287.73609674867447" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="522" data-strategy="4" cx="947.8708881746551" cy="258.7130759156246" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="523" data-strategy="4" cx="963.2614905632238" cy="168.6041365308315" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="524" data-strategy="4" cx="970.7861648226125" cy="176.04435314588846" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="525" data-strategy="4" cx="949.7255013904122" cy="146.19476680347012" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="527" data-strategy="4" cx="909.2797629397469" cy="120.33579725342925" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="528" data-strategy="4" cx="918.197338606411" cy="117.67434039257824" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="529" data-strategy="4" cx="953.4350599875003" cy="136.27693449793725" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="530" data-strategy="4" cx="967.4094176023074" cy="114.58855108997447" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="531" data-strategy="4" cx="956.1284501394174" cy="96.14043509775338" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="532" data-strategy="4" cx="963.1052268761736" cy="96.30985290996878" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="533" data-strategy="4" cx="904.6846598919278" cy="134.63445137613812" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="535" data-strategy="4" cx="970.7656412734871" cy="207.2269966225409" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="536" data-strategy="4" cx="896.348280485646" cy="179.16248931533485" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="537" data-strategy="4" cx="913.6488415906251" cy="125.21734405683637" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="538" data-strategy="4" cx="958.0640709137567" cy="150.85905630938356" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="539" data-strategy="4" cx="960.2820023295905" cy="148.14875312699854" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="540" data-strategy="4" cx="934.6742723883261" cy="92.02013278054488" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="541" data-strategy="4" cx="976.1247926063336" cy="110.17928913300071" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="542" data-strategy="4" cx="971.6042080591567" cy="80.18381559788631" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="543" data-strategy="4" cx="969.1554890170233" cy="92.43659624909856" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="544" data-strategy="4" cx="936.7582004649527" cy="134.1446221032373" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="545" data-strategy="4" cx="969.7744313246928" cy="126.73035685182398" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="546" data-strategy="4" cx="958.1633576438957" cy="149.91999879204144" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="547" data-strategy="4" cx="906.6088620656261" cy="174.91103087457896" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="548" data-strategy="4" cx="908.4382742271824" cy="188.98685620147958" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="549" data-strategy="4" cx="971.6493420781417" cy="241.33903098979488" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="550" data-strategy="4" cx="978.6235177329354" cy="245.19613448066013" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="551" data-strategy="4" cx="904.6221514102745" cy="266.06753590439826" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="552" data-strategy="4" cx="903.213597086189" cy="287.9370954547467" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="553" data-strategy="4" cx="970.9203188842156" cy="285.4227437657038" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="554" data-strategy="4" cx="921.1202431653768" cy="277.28449343907084" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="555" data-strategy="4" cx="950.418096376259" cy="262.8240240128323" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="556" data-strategy="4" cx="940.1564137018327" cy="257.75250106489835" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="557" data-strategy="4" cx="931.8953930706358" cy="238.20531188764997" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="558" data-strategy="4" cx="918.6200111567887" cy="288.7719418745147" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="559" data-strategy="4" cx="957.0041711108527" cy="268.8782346402387" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="560" data-strategy="4" cx="946.7782752314579" cy="237.39535719980412" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="561" data-strategy="4" cx="937.6062585140564" cy="277.0362623953913" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="562" data-strategy="4" cx="897.556343109936" cy="292.5851861576288" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="563" data-strategy="4" cx="943.9616157643478" cy="332.391752360876" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="564" data-strategy="4" cx="920.558253763832" cy="347.29084714890826" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="565" data-strategy="4" cx="974.3289641362114" cy="340.1576361694482" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="566" data-strategy="4" cx="900.3545799258347" cy="324.4029980908351" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="567" data-strategy="4" cx="902.0512234200827" cy="335.9106714929642" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="568" data-strategy="4" cx="927.1541540760414" cy="356.7066575474245" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="569" data-strategy="4" cx="944.7345577818556" cy="350.3904967281139" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="570" data-strategy="4" cx="976.0124580493198" cy="308.92185800580637" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="571" data-strategy="4" cx="911.7025430219422" cy="273.77006258530105" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="572" data-strategy="4" cx="961.698441777809" cy="242.29016673814945" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="573" data-strategy="4" cx="942.8855327790344" cy="197.08313788158563" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="648" data-strategy="4" cx="950.9962413267232" cy="287.0398706485453" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="649" data-strategy="4" cx="954.4608698498038" cy="280.30385451519476" r="3" fill="#ea580c" fill-opacity=".65"></circle><circle class="point" data-window="650" data-strategy="4" cx="920.9327034980806" cy="277.5743406512744" r="3" fill="#ea580c" fill-opacity=".65"></circle><line x1="895.25" y1="299.0926181791224" x2="983.25" y2="299.0926181791224" stroke="#be185d" stroke-width="3.5" class="median-line"/><text x="939.25" y="572" text-anchor="middle" >AH｜動能持有 24 日</text><text x="939.25" y="596" text-anchor="middle" class="median-label" style="fill:#be185d">中位數 4.65%</text><line x1="1129.75" y1="406.280793189218" x2="1129.75" y2="191.29569317825505" stroke="#0891b2" stroke-width="2" /><rect x="1091.75" y="268.40009165409003" width="76" height="56.524272694888964" fill="#0891b2" fill-opacity=".12" stroke="#0891b2"/><line x1="1091.75" y1="406.280793189218" x2="1167.75" y2="406.280793189218" stroke="#0891b2" stroke-width="2" /><line x1="1091.75" y1="191.29569317825505" x2="1167.75" y2="191.29569317825505" stroke="#0891b2" stroke-width="2" /><circle class="point" data-window="18" data-strategy="5" cx="1128.796377966691" cy="201.92897224315513" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="19" data-strategy="5" cx="1142.4318623553531" cy="199.64832176258926" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="20" data-strategy="5" cx="1151.9924003082779" cy="226.69701697307977" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="21" data-strategy="5" cx="1103.5462327084258" cy="218.67560793537817" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="22" data-strategy="5" cx="1112.7254016754796" cy="213.56222189402342" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="23" data-strategy="5" cx="1159.0433863190017" cy="225.05271079979036" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="24" data-strategy="5" cx="1132.9946094397033" cy="249.3416576434047" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="25" data-strategy="5" cx="1147.7600487762902" cy="240.84934008021065" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="26" data-strategy="5" cx="1163.8589481315548" cy="238.81682263803592" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="27" data-strategy="5" cx="1159.761490104799" cy="256.2864249741401" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="28" data-strategy="5" cx="1161.4308372021092" cy="293.3701977843669" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="29" data-strategy="5" cx="1140.4417948794799" cy="298.86321741439775" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="30" data-strategy="5" cx="1097.4707483805948" cy="274.68838595102295" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="31" data-strategy="5" cx="1095.2436334676872" cy="260.88100931283464" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="32" data-strategy="5" cx="1144.334057851995" cy="292.00786747454464" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="33" data-strategy="5" cx="1110.6204273378758" cy="293.42941569352035" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="34" data-strategy="5" cx="1164.0636498234137" cy="308.9480069962471" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="35" data-strategy="5" cx="1127.144358002714" cy="304.93810940048985" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="36" data-strategy="5" cx="1146.5608383033702" cy="294.2327753744852" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="37" data-strategy="5" cx="1105.701599319349" cy="300.02868198264883" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="38" data-strategy="5" cx="1117.9790710184836" cy="300.93493024572024" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="40" data-strategy="5" cx="1150.610137098575" cy="309.03352598082796" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="41" data-strategy="5" cx="1162.4579021308523" cy="322.5810080559968" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="42" data-strategy="5" cx="1095.9428517712058" cy="323.2983527453257" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="43" data-strategy="5" cx="1154.548864617341" cy="301.3334615176775" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="45" data-strategy="5" cx="1166.0588457094545" cy="353.6864500001722" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="46" data-strategy="5" cx="1135.1981967568886" cy="349.64823443216164" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="47" data-strategy="5" cx="1128.9173429291463" cy="316.1035324664239" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="48" data-strategy="5" cx="1121.176503636208" cy="322.68862659294956" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="50" data-strategy="5" cx="1164.5668705150808" cy="356.6485489538823" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="51" data-strategy="5" cx="1153.038075559625" cy="339.9396775898055" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="52" data-strategy="5" cx="1155.1418592738664" cy="339.3052544642281" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="53" data-strategy="5" cx="1123.6651107314294" cy="328.13375719241077" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="55" data-strategy="5" cx="1161.5534511844348" cy="319.0182357755959" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="56" data-strategy="5" cx="1157.6809095944088" cy="325.3236532472952" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="57" data-strategy="5" cx="1166.2463167875283" cy="327.9490223101486" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="58" data-strategy="5" cx="1151.493467717993" cy="326.2542117876964" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="60" data-strategy="5" cx="1146.5277377985124" cy="322.25456835924876" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="61" data-strategy="5" cx="1103.0416000925084" cy="334.2382891112313" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="62" data-strategy="5" cx="1157.335511053919" cy="337.6027278117726" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="63" data-strategy="5" cx="1127.8508889266993" cy="352.3530679961188" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="64" data-strategy="5" cx="1132.3912927412177" cy="335.8815316167432" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="65" data-strategy="5" cx="1108.3459632014683" cy="327.26331621137143" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="66" data-strategy="5" cx="1149.3919539260335" cy="333.4174737863809" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="68" data-strategy="5" cx="1138.8441077038444" cy="319.0267039678902" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="69" data-strategy="5" cx="1160.7980082884658" cy="328.58874052687673" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="71" data-strategy="5" cx="1157.031378230346" cy="294.2769054981041" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="73" data-strategy="5" cx="1107.773304353499" cy="270.64443035718926" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="74" data-strategy="5" cx="1159.7861990232223" cy="293.04988985041734" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="76" data-strategy="5" cx="1147.7360768164442" cy="298.2427908717222" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="78" data-strategy="5" cx="1164.797956774183" cy="292.25709943954354" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="79" data-strategy="5" cx="1144.9551546243833" cy="287.82139187034494" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="81" data-strategy="5" cx="1146.9904836869687" cy="303.25344247976096" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="84" data-strategy="5" cx="1101.0817998514617" cy="268.3379102962665" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="86" data-strategy="5" cx="1142.0791583615237" cy="284.6024398650989" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="89" data-strategy="5" cx="1094.3203548822542" cy="286.7138320446082" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="91" data-strategy="5" cx="1091.155897412348" cy="277.3215702286045" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="92" data-strategy="5" cx="1158.830372330036" cy="284.09532248483106" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="94" data-strategy="5" cx="1154.6332018482667" cy="293.26554697116126" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="95" data-strategy="5" cx="1094.444815334818" cy="288.8659568488592" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="97" data-strategy="5" cx="1124.9271402336883" cy="296.93742032346427" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="99" data-strategy="5" cx="1157.1944365684624" cy="292.2802515526338" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="100" data-strategy="5" cx="1146.4237852612493" cy="295.6337722982845" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="102" data-strategy="5" cx="1145.0088489622844" cy="290.86427968622445" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="104" data-strategy="5" cx="1103.1529842536274" cy="301.391518080528" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="105" data-strategy="5" cx="1128.0877448605538" cy="299.482510330763" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="107" data-strategy="5" cx="1102.0508146413215" cy="290.9080365410575" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="108" data-strategy="5" cx="1116.1325983552058" cy="302.5323492579117" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="109" data-strategy="5" cx="1157.1679823046898" cy="304.6979804385351" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="110" data-strategy="5" cx="1170.407229547605" cy="320.28600391826916" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="112" data-strategy="5" cx="1153.2292835910068" cy="347.96219765363713" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="113" data-strategy="5" cx="1154.8019355121062" cy="338.72368461127564" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="168" data-strategy="5" cx="1172.6283854160617" cy="272.2837892192565" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="170" data-strategy="5" cx="1106.6630476090545" cy="269.8699977487006" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="171" data-strategy="5" cx="1119.9237221871667" cy="296.0938000723634" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="172" data-strategy="5" cx="1090.472445209467" cy="283.65584407156314" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="173" data-strategy="5" cx="1161.9287953756623" cy="295.9745239836951" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="175" data-strategy="5" cx="1120.4842883501483" cy="304.0418919334455" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="176" data-strategy="5" cx="1130.1365836604268" cy="321.0869660155971" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="177" data-strategy="5" cx="1093.4689120613" cy="328.3926255683661" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="178" data-strategy="5" cx="1110.3438138740098" cy="337.18352033731725" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="179" data-strategy="5" cx="1160.2039254221404" cy="331.393884582928" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="180" data-strategy="5" cx="1099.017774524742" cy="325.30902593482773" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="181" data-strategy="5" cx="1127.6007673652068" cy="318.74662145572626" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="182" data-strategy="5" cx="1112.4708190726901" cy="311.57646918432533" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="183" data-strategy="5" cx="1110.128064239606" cy="309.64191575288925" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="184" data-strategy="5" cx="1122.5005902884661" cy="298.9390713871956" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="186" data-strategy="5" cx="1093.248176786559" cy="309.12983789175104" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="187" data-strategy="5" cx="1171.0545181413631" cy="316.9999134857901" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="188" data-strategy="5" cx="1100.2321224760603" cy="327.752676550216" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="189" data-strategy="5" cx="1142.5451134215073" cy="320.21883519462085" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="191" data-strategy="5" cx="1115.4555192293892" cy="326.4379230201752" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="192" data-strategy="5" cx="1160.0078629552527" cy="316.5050914493609" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="193" data-strategy="5" cx="1119.4477540713422" cy="314.6293190561293" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="194" data-strategy="5" cx="1129.9556382078788" cy="312.3464201975718" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="196" data-strategy="5" cx="1130.1744640382099" cy="320.7808269905145" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="197" data-strategy="5" cx="1130.0887748015343" cy="319.2072289880825" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="198" data-strategy="5" cx="1091.5418695264725" cy="330.2181675150798" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="199" data-strategy="5" cx="1171.1712519919636" cy="328.97298298527136" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="201" data-strategy="5" cx="1100.9690392188675" cy="304.83977223293283" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="203" data-strategy="5" cx="1120.7491694388686" cy="292.50831256889666" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="204" data-strategy="5" cx="1093.1732916614264" cy="298.58218140115105" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="205" data-strategy="5" cx="1096.2701446816461" cy="297.9286758809191" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="206" data-strategy="5" cx="1161.325454887934" cy="326.3417356473751" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="208" data-strategy="5" cx="1113.9649140774923" cy="309.8864854848535" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="209" data-strategy="5" cx="1157.8898364050679" cy="305.7233203027373" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="210" data-strategy="5" cx="1170.2119760040482" cy="322.80085787500855" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="211" data-strategy="5" cx="1103.3158470571645" cy="312.16616957232145" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="213" data-strategy="5" cx="1102.7796032546173" cy="292.20169672814967" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="214" data-strategy="5" cx="1095.0566486315897" cy="298.7463501744929" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="215" data-strategy="5" cx="1165.9920315107186" cy="304.16765981915404" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="216" data-strategy="5" cx="1129.1005388816081" cy="309.044346309897" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="218" data-strategy="5" cx="1108.3136127846356" cy="297.5379653299384" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="219" data-strategy="5" cx="1117.6155123552196" cy="295.1042451214649" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="220" data-strategy="5" cx="1151.0030622501752" cy="300.2660717181379" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="221" data-strategy="5" cx="1096.8030393530175" cy="287.84045611313235" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="223" data-strategy="5" cx="1160.4524455762808" cy="320.67127888783557" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="224" data-strategy="5" cx="1102.5207221628953" cy="324.346992441933" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="225" data-strategy="5" cx="1170.8402102577033" cy="330.61621842278004" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="226" data-strategy="5" cx="1147.549944078036" cy="328.5699788530555" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="227" data-strategy="5" cx="1130.1032188118588" cy="335.0440033501381" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="228" data-strategy="5" cx="1094.615292638817" cy="346.7702885715497" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="229" data-strategy="5" cx="1111.6231561249383" cy="345.21335825036573" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="230" data-strategy="5" cx="1095.2652266001057" cy="334.0097383885669" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="231" data-strategy="5" cx="1086.587910969863" cy="346.27852317552794" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="232" data-strategy="5" cx="1129.2300581035051" cy="313.145948336753" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="233" data-strategy="5" cx="1134.090932838295" cy="325.2170124791602" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="234" data-strategy="5" cx="1130.727513213649" cy="306.9567941975781" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="235" data-strategy="5" cx="1124.9351753801832" cy="304.5864324469716" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="236" data-strategy="5" cx="1172.547797117271" cy="308.268336220887" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="237" data-strategy="5" cx="1090.206777475866" cy="329.31357637660034" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="238" data-strategy="5" cx="1171.834358447269" cy="334.6496814641816" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="239" data-strategy="5" cx="1113.5055704474994" cy="322.81807229698603" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="240" data-strategy="5" cx="1167.200020180422" cy="321.1890152105228" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="241" data-strategy="5" cx="1115.5861919924555" cy="308.7754992667738" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="242" data-strategy="5" cx="1115.743650516139" cy="320.0708796146239" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="243" data-strategy="5" cx="1093.782495752424" cy="313.81842792612196" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="244" data-strategy="5" cx="1155.5165396979605" cy="317.4814726164144" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="245" data-strategy="5" cx="1106.8805770506683" cy="305.8970005964073" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="246" data-strategy="5" cx="1146.039590362078" cy="311.9928409858516" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="247" data-strategy="5" cx="1089.5142308577956" cy="307.9388096272744" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="248" data-strategy="5" cx="1127.2549897479212" cy="306.83944249428373" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="249" data-strategy="5" cx="1153.1646896353413" cy="310.3422866436088" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="250" data-strategy="5" cx="1114.9707195598605" cy="320.67363662132965" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="251" data-strategy="5" cx="1167.3610537432758" cy="306.4591989359064" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="252" data-strategy="5" cx="1163.1325265382502" cy="293.9176858349961" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="253" data-strategy="5" cx="1126.887890098615" cy="296.6231042715334" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="254" data-strategy="5" cx="1167.5938342270333" cy="303.9906422245143" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="255" data-strategy="5" cx="1101.6636735690406" cy="324.91177841196475" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="256" data-strategy="5" cx="1106.3750705787706" cy="347.3099533542029" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="257" data-strategy="5" cx="1154.008311759126" cy="330.18421965059053" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="258" data-strategy="5" cx="1115.0090357582787" cy="344.5032259011511" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="259" data-strategy="5" cx="1108.4240510680768" cy="342.2057922202993" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="261" data-strategy="5" cx="1166.8525591517055" cy="338.76640650283645" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="262" data-strategy="5" cx="1150.5443895837125" cy="354.3718609354087" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="263" data-strategy="5" cx="1091.3991310065478" cy="358.88685456732463" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="264" data-strategy="5" cx="1146.3468782662244" cy="365.489863990426" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="265" data-strategy="5" cx="1088.000169245385" cy="364.17434517320874" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="266" data-strategy="5" cx="1106.6944597007355" cy="363.83167821654496" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="267" data-strategy="5" cx="1100.1498794907723" cy="357.2555822691286" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="268" data-strategy="5" cx="1140.7750611295955" cy="368.15066452384485" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="269" data-strategy="5" cx="1119.4626008868543" cy="354.2393190105842" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="270" data-strategy="5" cx="1133.8977842406603" cy="348.1739378559474" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="271" data-strategy="5" cx="1165.0832777860535" cy="361.44287218993503" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="272" data-strategy="5" cx="1170.4326696479518" cy="357.5009566171685" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="273" data-strategy="5" cx="1101.605161379493" cy="347.0884259989617" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="274" data-strategy="5" cx="1116.448200143434" cy="368.4434874338621" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="275" data-strategy="5" cx="1137.3582719310373" cy="373.7340190418509" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="276" data-strategy="5" cx="1159.602561117353" cy="391.405081400118" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="277" data-strategy="5" cx="1162.3708969625154" cy="385.89933112204955" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="278" data-strategy="5" cx="1140.2042221924169" cy="364.79363963880985" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="279" data-strategy="5" cx="1116.6727278910232" cy="406.280793189218" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="280" data-strategy="5" cx="1149.352028720319" cy="425.1295697562892" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="281" data-strategy="5" cx="1113.7278053357704" cy="483.621721885625" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="282" data-strategy="5" cx="1154.2423882296841" cy="456.50049491793226" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="283" data-strategy="5" cx="1136.5848109244537" cy="438.931639441439" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="284" data-strategy="5" cx="1149.3892010554284" cy="414.0629967401381" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="285" data-strategy="5" cx="1155.7674236201465" cy="390.9543594551677" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="286" data-strategy="5" cx="1141.733431759047" cy="395.6140817917908" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="287" data-strategy="5" cx="1163.6920081029816" cy="392.9785922428204" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="288" data-strategy="5" cx="1110.4063993746756" cy="389.5420423541086" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="289" data-strategy="5" cx="1137.44052482087" cy="399.3895659668077" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="290" data-strategy="5" cx="1139.396236460497" cy="402.7850191385856" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="291" data-strategy="5" cx="1169.261333124843" cy="393.2836878271041" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="292" data-strategy="5" cx="1089.2827536315476" cy="400.34973446865723" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="293" data-strategy="5" cx="1106.5923066946154" cy="394.5996432294344" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="294" data-strategy="5" cx="1164.583448938066" cy="385.49887315296974" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="295" data-strategy="5" cx="1099.7510467071588" cy="386.0345257154391" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="296" data-strategy="5" cx="1162.2047588314267" cy="382.3035540220031" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="297" data-strategy="5" cx="1118.3727056564" cy="369.83474642810575" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="298" data-strategy="5" cx="1115.8314833013624" cy="369.6243632051206" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="299" data-strategy="5" cx="1146.6328405233498" cy="344.7924120701745" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="300" data-strategy="5" cx="1165.3842548350924" cy="347.6205972018588" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="301" data-strategy="5" cx="1122.2552602877631" cy="348.1163938620968" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="304" data-strategy="5" cx="1105.924858946894" cy="264.2599919768003" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="305" data-strategy="5" cx="1166.3144875546445" cy="294.9090084645674" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="306" data-strategy="5" cx="1092.7002060239456" cy="286.0685599265846" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="307" data-strategy="5" cx="1088.4827338595187" cy="298.4874286434822" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="308" data-strategy="5" cx="1154.2484706371886" cy="299.823277986181" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="309" data-strategy="5" cx="1163.3690172954978" cy="294.216414691639" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="310" data-strategy="5" cx="1170.650923135581" cy="289.68279738835207" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="311" data-strategy="5" cx="1172.2314414686757" cy="289.69346635563545" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="312" data-strategy="5" cx="1117.19716424992" cy="292.955850459997" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="313" data-strategy="5" cx="1128.4518142131537" cy="289.4295204376956" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="314" data-strategy="5" cx="1130.1871454418788" cy="303.84267467691666" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="315" data-strategy="5" cx="1127.534979439515" cy="303.590867423076" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="316" data-strategy="5" cx="1127.2059620107375" cy="309.93517885625937" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="317" data-strategy="5" cx="1121.4636121492667" cy="315.62421001294587" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="318" data-strategy="5" cx="1116.0985453152134" cy="318.97237895976934" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="319" data-strategy="5" cx="1115.9154151562586" cy="315.3028337115409" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="320" data-strategy="5" cx="1167.1746836280035" cy="311.51955301589845" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="321" data-strategy="5" cx="1160.382184217449" cy="276.8622401579564" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="322" data-strategy="5" cx="1167.233686494461" cy="291.22246653330217" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="323" data-strategy="5" cx="1170.7214908526046" cy="283.35631489777074" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="324" data-strategy="5" cx="1116.8939895720132" cy="262.4549984537824" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="325" data-strategy="5" cx="1139.6887004193209" cy="265.34516555747064" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="326" data-strategy="5" cx="1127.396909535881" cy="281.5887422317062" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="327" data-strategy="5" cx="1103.0598625589994" cy="267.9754464718259" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="328" data-strategy="5" cx="1089.0584163154306" cy="288.19502653582447" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="329" data-strategy="5" cx="1163.6400376904467" cy="296.16053368526235" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="330" data-strategy="5" cx="1125.1393398549203" cy="289.21216877482914" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="331" data-strategy="5" cx="1148.0809230209575" cy="293.6013461033475" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="332" data-strategy="5" cx="1089.1662555571654" cy="297.7903510740117" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="333" data-strategy="5" cx="1092.4564031200184" cy="278.2239762225522" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="334" data-strategy="5" cx="1165.3244602304705" cy="280.5748940856966" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="335" data-strategy="5" cx="1135.164630800885" cy="282.03682211271075" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="336" data-strategy="5" cx="1103.4589393337612" cy="290.0849438321276" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="337" data-strategy="5" cx="1125.5706711256955" cy="285.775440383778" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="338" data-strategy="5" cx="1133.7248563039154" cy="273.3526396780986" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="339" data-strategy="5" cx="1092.7523563284867" cy="275.1730540943923" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="340" data-strategy="5" cx="1168.2583742944535" cy="268.8502482767993" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="341" data-strategy="5" cx="1137.6273770881792" cy="283.62779198688463" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="342" data-strategy="5" cx="1168.2840793657972" cy="299.1964770887057" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="343" data-strategy="5" cx="1098.6938168322583" cy="283.69416044537695" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="344" data-strategy="5" cx="1134.1277318164011" cy="266.0397237644663" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="345" data-strategy="5" cx="1106.2576103071015" cy="268.1998938404064" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="346" data-strategy="5" cx="1167.1557820151918" cy="285.5392782784809" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="347" data-strategy="5" cx="1153.164271402023" cy="303.09945246873" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="348" data-strategy="5" cx="1124.398971636574" cy="290.0687632208335" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="349" data-strategy="5" cx="1113.4632169897643" cy="283.24968142470595" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="395" data-strategy="5" cx="1160.8103180985968" cy="311.8140996451671" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="396" data-strategy="5" cx="1109.804465070261" cy="306.68079059835486" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="397" data-strategy="5" cx="1168.4106957318356" cy="264.925960915239" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="398" data-strategy="5" cx="1173.2191338308405" cy="282.05061502011074" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="399" data-strategy="5" cx="1160.6193617342058" cy="284.98240815555505" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="400" data-strategy="5" cx="1090.79593252866" cy="300.52938021535016" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="401" data-strategy="5" cx="1086.5250911356165" cy="317.4173965422107" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="402" data-strategy="5" cx="1133.2169915472168" cy="319.0742700290871" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="403" data-strategy="5" cx="1169.1670707325113" cy="312.56825357845526" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="404" data-strategy="5" cx="1173.014356273641" cy="328.61789230581377" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="405" data-strategy="5" cx="1095.4611824337258" cy="322.56263565357756" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="407" data-strategy="5" cx="1095.6225130774462" cy="292.3971008771563" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="408" data-strategy="5" cx="1134.40538668824" cy="284.24427550649546" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="409" data-strategy="5" cx="1094.4732102739922" cy="292.2405881615547" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="410" data-strategy="5" cx="1142.523823008565" cy="292.6771023603567" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="412" data-strategy="5" cx="1133.678774817439" cy="279.4751221217559" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="413" data-strategy="5" cx="1122.476193458692" cy="279.3027739532373" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="414" data-strategy="5" cx="1128.567219564828" cy="279.38364247948505" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="415" data-strategy="5" cx="1088.2391156020956" cy="231.9573916284287" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="417" data-strategy="5" cx="1120.2783085780788" cy="241.71305733735434" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="418" data-strategy="5" cx="1099.3702046500925" cy="221.02638997286886" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="419" data-strategy="5" cx="1161.2468868290289" cy="230.08773268911378" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="420" data-strategy="5" cx="1131.4494572016888" cy="214.89650063939155" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="422" data-strategy="5" cx="1161.7756103495376" cy="226.75765512346513" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="423" data-strategy="5" cx="1156.0164650331815" cy="222.70610389644784" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="424" data-strategy="5" cx="1172.6908850999912" cy="220.74480455685372" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="425" data-strategy="5" cx="1116.807507661912" cy="242.58044483574486" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="427" data-strategy="5" cx="1143.4998369273437" cy="256.55532492758516" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="428" data-strategy="5" cx="1137.0366941669565" cy="235.88853523706774" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="429" data-strategy="5" cx="1169.323028587536" cy="254.83182469389314" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="430" data-strategy="5" cx="1148.4472884042436" cy="258.8111524412832" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="431" data-strategy="5" cx="1102.4768915456307" cy="265.1820756072864" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="443" data-strategy="5" cx="1124.1258246863385" cy="324.93695028599336" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="444" data-strategy="5" cx="1130.2573219235344" cy="336.66715649978494" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="445" data-strategy="5" cx="1095.6550478210193" cy="334.4784758382717" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="446" data-strategy="5" cx="1132.4320097715556" cy="343.62829832609106" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="447" data-strategy="5" cx="1093.7048284646576" cy="319.5058869171842" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="448" data-strategy="5" cx="1163.1210110534844" cy="292.4047050979236" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="449" data-strategy="5" cx="1141.0426970489386" cy="303.59932617979524" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="450" data-strategy="5" cx="1101.5512425174643" cy="300.5372826493498" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="451" data-strategy="5" cx="1150.7709627642553" cy="306.48468528675704" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="452" data-strategy="5" cx="1165.8911487665523" cy="313.6791421838605" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="453" data-strategy="5" cx="1097.7062849899626" cy="327.8284551846437" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="454" data-strategy="5" cx="1100.6751851417125" cy="350.9910518388154" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="455" data-strategy="5" cx="1123.7170358272751" cy="325.997618070496" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="456" data-strategy="5" cx="1118.827006142821" cy="341.45592666407174" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="457" data-strategy="5" cx="1107.2023749402304" cy="318.97366731296586" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="458" data-strategy="5" cx="1167.153444565748" cy="292.46868652220246" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="459" data-strategy="5" cx="1156.7014121824366" cy="301.7615876602797" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="460" data-strategy="5" cx="1102.9119457182853" cy="259.26418854541714" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="461" data-strategy="5" cx="1135.9248040619889" cy="257.18624371816304" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="462" data-strategy="5" cx="1165.785450084294" cy="265.62673204865683" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="463" data-strategy="5" cx="1109.3691457993657" cy="253.3230162702683" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="465" data-strategy="5" cx="1162.997464700029" cy="279.2757684524029" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="466" data-strategy="5" cx="1135.18597700605" cy="245.58091704534453" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="467" data-strategy="5" cx="1147.059220888556" cy="252.2617571374476" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="468" data-strategy="5" cx="1109.8190956331177" cy="224.20389773179883" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="470" data-strategy="5" cx="1134.3136926064913" cy="266.74336443077976" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="471" data-strategy="5" cx="1128.606028033206" cy="258.90120741789076" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="472" data-strategy="5" cx="1105.8236516206834" cy="253.20766607642355" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="475" data-strategy="5" cx="1135.6494143831208" cy="258.95823422089484" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="476" data-strategy="5" cx="1104.2413302912344" cy="250.577423752912" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="477" data-strategy="5" cx="1147.9763745735443" cy="228.7574562173582" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="480" data-strategy="5" cx="1147.8263672308728" cy="196.6754664561284" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="504" data-strategy="5" cx="1139.416455264594" cy="320.73353591963945" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="505" data-strategy="5" cx="1162.5024894461853" cy="284.1111204233108" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="506" data-strategy="5" cx="1120.4293497741132" cy="289.5417482921512" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="507" data-strategy="5" cx="1144.758368590575" cy="309.02236728972605" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="508" data-strategy="5" cx="1137.1019869662719" cy="276.8625206827071" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="509" data-strategy="5" cx="1155.8946302893403" cy="240.05111239809276" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="510" data-strategy="5" cx="1169.7388460338952" cy="218.81486232616456" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="511" data-strategy="5" cx="1135.6060902536567" cy="191.52325673406963" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="512" data-strategy="5" cx="1129.4401075989138" cy="199.7411180689523" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="513" data-strategy="5" cx="1126.4544641279795" cy="192.6281463551986" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="514" data-strategy="5" cx="1092.086876354602" cy="263.25181850283207" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="515" data-strategy="5" cx="1165.898811020756" cy="217.41222766899713" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="516" data-strategy="5" cx="1171.4383628542803" cy="226.47786919185359" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="517" data-strategy="5" cx="1167.6297830376373" cy="219.11695339290142" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="518" data-strategy="5" cx="1156.5554291479357" cy="262.80609947881237" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="519" data-strategy="5" cx="1088.8191273245648" cy="302.98169262431765" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="520" data-strategy="5" cx="1091.029972347574" cy="282.21353279410175" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="521" data-strategy="5" cx="1158.779860955723" cy="284.03911077783306" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="522" data-strategy="5" cx="1147.0567758450736" cy="247.09930571679902" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="523" data-strategy="5" cx="1096.521495174535" cy="182.26269738099748" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="524" data-strategy="5" cx="1138.8965582961757" cy="191.29569317825505" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="525" data-strategy="5" cx="1130.5141032834838" cy="159.51778405538062" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="527" data-strategy="5" cx="1106.437987028588" cy="155.2777441729658" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="528" data-strategy="5" cx="1139.8068571651927" cy="127.31796521622539" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="529" data-strategy="5" cx="1109.6919965258696" cy="120.42247709437791" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="530" data-strategy="5" cx="1117.5911417875357" cy="109.43549430175807" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="531" data-strategy="5" cx="1139.7508344500643" cy="75.38434115133592" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="532" data-strategy="5" cx="1166.758600111104" cy="104.13779995090565" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="533" data-strategy="5" cx="1129.2338313615005" cy="148.38884539356704" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="535" data-strategy="5" cx="1116.5028935049902" cy="191.8913282417017" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="536" data-strategy="5" cx="1133.78436586994" cy="173.17378963767766" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="537" data-strategy="5" cx="1132.6167201657886" cy="137.1922287293831" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="538" data-strategy="5" cx="1107.9765279268875" cy="155.8937013109894" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="539" data-strategy="5" cx="1145.0752908393101" cy="153.34654963277092" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="540" data-strategy="5" cx="1127.9925496175665" cy="103.92084892214937" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="541" data-strategy="5" cx="1103.7987213176887" cy="140.30566492935685" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="542" data-strategy="5" cx="1159.5820073759623" cy="105.84934950367295" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="543" data-strategy="5" cx="1151.5322377306682" cy="106.98844526501763" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="544" data-strategy="5" cx="1148.2410200159368" cy="142.03886027146888" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="545" data-strategy="5" cx="1172.9494243764148" cy="143.2555542658784" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="546" data-strategy="5" cx="1132.336206416189" cy="162.15142459569574" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="547" data-strategy="5" cx="1165.7220677836021" cy="199.66273004694054" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="548" data-strategy="5" cx="1160.4762427394226" cy="202.3821466320043" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="549" data-strategy="5" cx="1123.6173508915895" cy="239.75359747371309" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="550" data-strategy="5" cx="1091.5384561804185" cy="262.60213382923206" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="551" data-strategy="5" cx="1099.9602549192934" cy="279.25005465854315" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="552" data-strategy="5" cx="1115.3388805475997" cy="272.24785364964816" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="553" data-strategy="5" cx="1104.8509850583605" cy="268.46227301191357" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="554" data-strategy="5" cx="1100.5085418842891" cy="267.73450812423965" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="555" data-strategy="5" cx="1148.327052960368" cy="247.9592652373106" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="556" data-strategy="5" cx="1114.80453198762" cy="264.76705817763474" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="557" data-strategy="5" cx="1148.8892138823294" cy="256.17832483465276" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="558" data-strategy="5" cx="1133.979507829516" cy="271.59933602305756" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="559" data-strategy="5" cx="1149.6668720190153" cy="254.4532587595381" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="560" data-strategy="5" cx="1086.8616110865357" cy="235.7773962196218" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="561" data-strategy="5" cx="1110.1716245890689" cy="248.38044262204002" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="562" data-strategy="5" cx="1086.3259615822355" cy="259.13356081470477" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="563" data-strategy="5" cx="1136.3577199663598" cy="312.02259393116805" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="564" data-strategy="5" cx="1088.3901613932396" cy="320.5349695033591" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="565" data-strategy="5" cx="1147.4865898119263" cy="324.60367251428414" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="566" data-strategy="5" cx="1141.1157168203604" cy="329.67324055514615" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="567" data-strategy="5" cx="1163.9071230745171" cy="332.61678863205765" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="568" data-strategy="5" cx="1164.7663031598208" cy="322.71231836408623" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="569" data-strategy="5" cx="1088.6478986866045" cy="338.89382813446156" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="570" data-strategy="5" cx="1093.6540554224446" cy="309.4856371221092" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="571" data-strategy="5" cx="1120.8851270032678" cy="282.17366131665295" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="572" data-strategy="5" cx="1111.0510867584508" cy="248.09228356731853" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="573" data-strategy="5" cx="1106.4134517721127" cy="162.12655062400736" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="648" data-strategy="5" cx="1132.0779332148375" cy="294.91373032349514" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="649" data-strategy="5" cx="1172.94511240751" cy="294.8626658850401" r="3" fill="#0891b2" fill-opacity=".65"></circle><circle class="point" data-window="650" data-strategy="5" cx="1117.3473171432763" cy="288.21271746695277" r="3" fill="#0891b2" fill-opacity=".65"></circle><line x1="1085.75" y1="298.9390713871956" x2="1173.75" y2="298.9390713871956" stroke="#be185d" stroke-width="3.5" class="median-line"/><text x="1129.75" y="572" text-anchor="middle" >BH｜動能法人持有 24 日</text><text x="1129.75" y="596" text-anchor="middle" class="median-label" style="fill:#be185d">中位數 4.68%</text></svg></div></section>\n<section><details><summary>查看排除窗口 <span id="excludedHeading" class="secondary"></span></summary><p class="note">灰點為資料不完整的暫算值，不納入箱形圖與績效比較。所有排除窗口也完整保留。</p><div class="chart" id="excludedChart"><svg viewBox="0 0 1250 620" role="img" aria-label="排除窗口散點圖"><line x1="82" y1="500.88235294117646" x2="1225" y2="500.88235294117646" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="505.88235294117646" text-anchor="middle" >-30.0%</text><line x1="82" y1="442.6470588235294" x2="1225" y2="442.6470588235294" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="447.6470588235294" text-anchor="middle" >-20.0%</text><line x1="82" y1="384.4117647058824" x2="1225" y2="384.4117647058824" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="389.4117647058824" text-anchor="middle" >-10.0%</text><line x1="82" y1="267.94117647058823" x2="1225" y2="267.94117647058823" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="272.94117647058823" text-anchor="middle" >10.0%</text><line x1="82" y1="209.7058823529412" x2="1225" y2="209.7058823529412" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="214.7058823529412" text-anchor="middle" >20.0%</text><line x1="82" y1="151.47058823529412" x2="1225" y2="151.47058823529412" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="156.47058823529412" text-anchor="middle" >30.0%</text><line x1="82" y1="93.23529411764706" x2="1225" y2="93.23529411764706" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="98.23529411764706" text-anchor="middle" >40.0%</text><line x1="82" y1="35" x2="1225" y2="35" stroke="#e2e8f0" stroke-width="1" /><text x="50" y="40" text-anchor="middle" >50.0%</text><line x1="82" y1="326.1764705882353" x2="1225" y2="326.1764705882353" stroke="#111827" stroke-width="2" class="zero-baseline" stroke-dasharray="7 5"/><text x="50" y="331.1764705882353" text-anchor="middle" font-weight="bold">0%</text><circle class="point" data-window="1" data-strategy="0" cx="164.9991433762719" cy="301.6264658309051" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="2" data-strategy="0" cx="154.21072501079126" cy="300.94257216581263" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="3" data-strategy="0" cx="193.08239918082137" cy="293.1388962750466" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="4" data-strategy="0" cx="143.28353086831984" cy="289.43217318968107" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="5" data-strategy="0" cx="200.93374783961445" cy="273.113496170256" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="6" data-strategy="0" cx="167.137931329574" cy="274.1235552640642" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="7" data-strategy="0" cx="157.4090810198154" cy="275.2192585759939" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="8" data-strategy="0" cx="140.82874882073608" cy="268.60717317703904" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="9" data-strategy="0" cx="141.6364362307783" cy="275.1267850232364" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="10" data-strategy="0" cx="174.42998453198325" cy="272.47918393445775" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="11" data-strategy="0" cx="149.55498549147916" cy="279.56778758042947" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="12" data-strategy="0" cx="149.32194230070235" cy="283.0455064733417" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="13" data-strategy="0" cx="204.9932942601891" cy="284.48959214657515" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="14" data-strategy="0" cx="142.26078082538766" cy="270.21627300055445" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="15" data-strategy="0" cx="188.6498573097518" cy="264.1204117733117" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="16" data-strategy="0" cx="209.68328617789643" cy="272.0025043864558" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="17" data-strategy="0" cx="162.14153648833033" cy="278.69778780962224" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="39" data-strategy="0" cx="184.37165123639116" cy="296.3163955648502" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="44" data-strategy="0" cx="140.57421254632573" cy="316.1214784742865" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="49" data-strategy="0" cx="165.55354664625307" cy="364.88930717952724" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="54" data-strategy="0" cx="165.14330962541504" cy="334.16013968868174" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="59" data-strategy="0" cx="172.90950149135045" cy="325.58712548779266" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="67" data-strategy="0" cx="163.03802658296036" cy="316.40100880863224" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="70" data-strategy="0" cx="202.0446123408089" cy="305.98135465660374" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="72" data-strategy="0" cx="194.50554140936796" cy="275.23495194100616" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="75" data-strategy="0" cx="135.1195272760495" cy="279.7612190156635" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="77" data-strategy="0" cx="219.8490200278129" cy="313.8017810264716" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="80" data-strategy="0" cx="179.3446844989162" cy="303.7696862167942" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="82" data-strategy="0" cx="176.8293873536128" cy="302.5993316949669" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="83" data-strategy="0" cx="186.15527177237936" cy="312.92000336672936" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="85" data-strategy="0" cx="155.5677936266041" cy="288.28294732702557" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="87" data-strategy="0" cx="217.14132965022264" cy="289.2627064394967" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="88" data-strategy="0" cx="219.3806495638269" cy="291.105411106586" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="90" data-strategy="0" cx="188.17134124929257" cy="275.579664123834" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="93" data-strategy="0" cx="154.0706849304387" cy="291.38430108003445" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="96" data-strategy="0" cx="180.6814622821848" cy="313.07374896088635" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="98" data-strategy="0" cx="141.72922274531578" cy="320.0000255105447" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="101" data-strategy="0" cx="206.87395059532827" cy="299.1949987259698" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="103" data-strategy="0" cx="171.28390095322894" cy="292.2416300048989" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="106" data-strategy="0" cx="185.41556492986473" cy="305.4765515453389" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="111" data-strategy="0" cx="217.84706221819465" cy="348.7470382847191" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="114" data-strategy="0" cx="135.32789813652232" cy="382.6190252744656" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="115" data-strategy="0" cx="216.03413894882215" cy="385.8253533732908" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="116" data-strategy="0" cx="140.91682926988256" cy="369.04422122007554" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="117" data-strategy="0" cx="207.681983955239" cy="373.90856512346625" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="118" data-strategy="0" cx="184.31364729934916" cy="359.7043746661666" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="119" data-strategy="0" cx="175.82338536368187" cy="387.62516102208923" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="120" data-strategy="0" cx="136.17731290690614" cy="423.4740792615706" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="121" data-strategy="0" cx="137.58806178769802" cy="405.83420359332735" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="122" data-strategy="0" cx="140.25638836077331" cy="391.0216132521151" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="123" data-strategy="0" cx="156.17602083589242" cy="407.96191179762127" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="124" data-strategy="0" cx="205.50646000292278" cy="398.14573336300026" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="125" data-strategy="0" cx="194.7298844923702" cy="383.851259015732" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="126" data-strategy="0" cx="220.16164089403873" cy="388.2696058977607" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="127" data-strategy="0" cx="139.2662855909708" cy="385.36147964368377" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="128" data-strategy="0" cx="180.72216392418696" cy="393.1992068966891" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="129" data-strategy="0" cx="173.0974302770621" cy="371.38708561676384" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="130" data-strategy="0" cx="146.53655194847664" cy="369.42421086376135" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="131" data-strategy="0" cx="182.91638759144917" cy="368.64234367470505" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="132" data-strategy="0" cx="181.72795959113972" cy="367.299592790302" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="133" data-strategy="0" cx="182.62572501662393" cy="359.30108084064057" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="134" data-strategy="0" cx="191.02872819635795" cy="350.39895917693264" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="135" data-strategy="0" cx="186.56990473145885" cy="338.6461165390753" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="136" data-strategy="0" cx="211.74313590656436" cy="349.79528897281017" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="137" data-strategy="0" cx="162.58023621903087" cy="325.2850974184988" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="138" data-strategy="0" cx="144.87400964379128" cy="316.15921670531947" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="139" data-strategy="0" cx="210.26011465553313" cy="312.3903531030861" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="140" data-strategy="0" cx="199.61127799781693" cy="326.1830967340298" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="141" data-strategy="0" cx="202.39409699171568" cy="345.0433370961163" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="142" data-strategy="0" cx="158.86105008156485" cy="346.6510363814186" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="143" data-strategy="0" cx="193.72143613091043" cy="302.14111235424303" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="144" data-strategy="0" cx="164.4761933417752" cy="289.751509434911" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="145" data-strategy="0" cx="136.98794719298422" cy="314.3130111444628" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="146" data-strategy="0" cx="165.8131545177116" cy="320.14713491460697" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="147" data-strategy="0" cx="181.34799850576385" cy="344.08210323739564" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="148" data-strategy="0" cx="178.95379366989553" cy="329.5242674992664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="149" data-strategy="0" cx="190.77071685135257" cy="330.86950873312935" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="150" data-strategy="0" cx="196.3640728035" cy="334.6340899415051" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="151" data-strategy="0" cx="215.67987430245094" cy="341.83109428405805" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="152" data-strategy="0" cx="207.8495876911297" cy="342.96592359068313" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="153" data-strategy="0" cx="198.7246104825855" cy="345.2719429147587" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="154" data-strategy="0" cx="194.14945294979134" cy="345.3998829285665" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="155" data-strategy="0" cx="201.78023457923527" cy="337.8664376430883" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="156" data-strategy="0" cx="154.1449733137791" cy="325.1805685329236" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="157" data-strategy="0" cx="209.4435796636439" cy="320.17569111133355" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="158" data-strategy="0" cx="157.9291347698241" cy="324.67122230364265" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="159" data-strategy="0" cx="139.384390735005" cy="339.58177318883656" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="160" data-strategy="0" cx="138.02421412168536" cy="337.46922726821424" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="161" data-strategy="0" cx="177.37451604648024" cy="345.4468247978512" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="162" data-strategy="0" cx="202.3253831539669" cy="342.164758135744" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="163" data-strategy="0" cx="194.017475918447" cy="337.2436772330234" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="164" data-strategy="0" cx="206.09133533084255" cy="334.6691969054198" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="165" data-strategy="0" cx="215.0532952815228" cy="296.18542349159287" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="166" data-strategy="0" cx="206.78863458828954" cy="290.43003018675915" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="167" data-strategy="0" cx="148.4063033016761" cy="284.66096810476995" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="169" data-strategy="0" cx="140.36233891636067" cy="278.32942434746576" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="174" data-strategy="0" cx="135.55467800965354" cy="308.2636775710439" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="185" data-strategy="0" cx="168.31272707837988" cy="293.2107998442244" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="190" data-strategy="0" cx="142.79862298495752" cy="347.9222650908678" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="195" data-strategy="0" cx="206.6779967353908" cy="322.45731241983157" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="200" data-strategy="0" cx="135.51637659922687" cy="309.66696911270776" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="202" data-strategy="0" cx="182.94841351539992" cy="286.8106780472676" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="207" data-strategy="0" cx="212.44083536439416" cy="339.63996231073094" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="212" data-strategy="0" cx="200.58667892451965" cy="310.4675387918226" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="217" data-strategy="0" cx="199.19808387585886" cy="316.4389956200283" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="222" data-strategy="0" cx="213.01665777277157" cy="298.27713440122403" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="260" data-strategy="0" cx="145.9656379640548" cy="322.102057821984" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="302" data-strategy="0" cx="142.31245465652694" cy="275.38888960489305" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="303" data-strategy="0" cx="200.14395990765996" cy="223.02772079593277" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="350" data-strategy="0" cx="173.33330185285848" cy="285.3525857313536" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="351" data-strategy="0" cx="149.49077466178775" cy="281.1676189439008" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="352" data-strategy="0" cx="167.69030978965384" cy="287.3776033746188" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="353" data-strategy="0" cx="210.57046590090917" cy="300.7452361728111" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="354" data-strategy="0" cx="200.5687757034877" cy="286.5586669676718" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="355" data-strategy="0" cx="215.49544643071346" cy="277.95655650327046" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="356" data-strategy="0" cx="138.1088792613162" cy="289.77022596251766" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="357" data-strategy="0" cx="191.3202191064327" cy="286.7342193247655" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="358" data-strategy="0" cx="194.43328423907482" cy="288.4349120060984" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="359" data-strategy="0" cx="215.10717315523326" cy="283.45677356744136" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="360" data-strategy="0" cx="134.99755114247182" cy="265.16639714200977" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="361" data-strategy="0" cx="190.1726512624612" cy="270.20039485404857" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="362" data-strategy="0" cx="178.52946677743154" cy="275.91749734138546" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="363" data-strategy="0" cx="201.89637334862195" cy="274.25365140572507" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="364" data-strategy="0" cx="161.85427187925328" cy="278.7468541392849" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="365" data-strategy="0" cx="217.10063212537952" cy="262.5178168990731" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="366" data-strategy="0" cx="190.59861167130293" cy="250.25848079063246" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="367" data-strategy="0" cx="156.0360699742699" cy="252.4135440944837" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="368" data-strategy="0" cx="152.66926471862962" cy="262.5048297386776" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="369" data-strategy="0" cx="162.6722969507968" cy="258.8650166256393" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="370" data-strategy="0" cx="138.83094081152387" cy="251.04959983115253" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="371" data-strategy="0" cx="152.08431701664907" cy="249.79947585566475" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="372" data-strategy="0" cx="137.14674246122996" cy="247.9891809091863" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="373" data-strategy="0" cx="182.66043867351584" cy="259.79301485946405" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="374" data-strategy="0" cx="176.88309239361226" cy="278.8620414479889" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="375" data-strategy="0" cx="140.24990154905396" cy="268.9848132463262" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="376" data-strategy="0" cx="201.85741805932477" cy="264.3559540889558" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="377" data-strategy="0" cx="189.2139840843215" cy="222.16602503952845" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="378" data-strategy="0" cx="192.47572006901464" cy="227.36361770599274" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="379" data-strategy="0" cx="181.68227578103955" cy="216.07608493820055" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="380" data-strategy="0" cx="199.44772327563658" cy="229.3978146932704" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="381" data-strategy="0" cx="215.35021137275334" cy="219.92830119149258" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="382" data-strategy="0" cx="147.63460817147683" cy="250.81884615285384" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="383" data-strategy="0" cx="215.83984617478308" cy="262.66444826056284" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="384" data-strategy="0" cx="148.12945856193514" cy="269.8670068106982" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="385" data-strategy="0" cx="139.2984331272555" cy="265.7028235628325" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="386" data-strategy="0" cx="145.75273140099873" cy="253.13905917525452" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="387" data-strategy="0" cx="179.18259420994292" cy="246.42685634035527" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="388" data-strategy="0" cx="180.04859042317796" cy="255.5733069661705" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="389" data-strategy="0" cx="185.93539950461803" cy="251.14151159255496" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="390" data-strategy="0" cx="144.78068876900002" cy="259.4226888379419" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="391" data-strategy="0" cx="214.5762457152422" cy="274.86171786667137" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="392" data-strategy="0" cx="136.91418013496332" cy="295.0175161287836" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="393" data-strategy="0" cx="187.43470458529256" cy="293.5725164590513" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="394" data-strategy="0" cx="184.5130875673227" cy="301.1886172843159" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="406" data-strategy="0" cx="197.61331811332923" cy="262.12343524413325" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="411" data-strategy="0" cx="188.66184016546632" cy="294.249549564672" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="416" data-strategy="0" cx="147.22588865917285" cy="272.9390071584726" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="421" data-strategy="0" cx="194.95106705940924" cy="277.1942572989241" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="426" data-strategy="0" cx="207.16756553011996" cy="246.43652785320708" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="432" data-strategy="0" cx="212.13711139211293" cy="256.1243512566336" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="433" data-strategy="0" cx="158.32891036717842" cy="244.34373859004793" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="434" data-strategy="0" cx="143.64376754049158" cy="263.45589593665864" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="435" data-strategy="0" cx="191.87963625231978" cy="289.29003567935086" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="436" data-strategy="0" cx="217.26564926508064" cy="276.2494995427496" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="437" data-strategy="0" cx="186.26194908189933" cy="299.0138654042305" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="438" data-strategy="0" cx="135.47595011851462" cy="300.8576698115721" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="439" data-strategy="0" cx="183.45593668630536" cy="281.2132710974408" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="440" data-strategy="0" cx="200.57134045546414" cy="274.8282564717622" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="441" data-strategy="0" cx="194.40770219637673" cy="279.6327130214405" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="442" data-strategy="0" cx="192.83899875133415" cy="282.725256662971" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="464" data-strategy="0" cx="181.4588392092975" cy="267.00675164863486" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="469" data-strategy="0" cx="181.64984616856813" cy="226.840400576909" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="473" data-strategy="0" cx="187.00005732517278" cy="264.71031431782876" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="474" data-strategy="0" cx="200.11430466852897" cy="262.7951812961826" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="478" data-strategy="0" cx="164.81304827635302" cy="225.44227414956433" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="479" data-strategy="0" cx="196.57214441645934" cy="209.81479819460333" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="481" data-strategy="0" cx="182.55543758668594" cy="217.13966570997178" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="482" data-strategy="0" cx="219.22400847564376" cy="219.03610938933758" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="483" data-strategy="0" cx="182.14960238743038" cy="204.40671124895772" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="484" data-strategy="0" cx="160.30959349214623" cy="220.26802719482382" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="485" data-strategy="0" cx="177.09970039104275" cy="233.8619825310778" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="486" data-strategy="0" cx="147.09016275254032" cy="256.0286505304374" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="487" data-strategy="0" cx="143.85344376471335" cy="256.4715233097824" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="488" data-strategy="0" cx="192.8501724161529" cy="229.40432185059052" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="489" data-strategy="0" cx="160.58188436757501" cy="266.60025694061767" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="490" data-strategy="0" cx="157.6193669312918" cy="293.9730194479507" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="491" data-strategy="0" cx="170.04266814389055" cy="273.693945264545" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="492" data-strategy="0" cx="164.6246373170506" cy="264.820705977156" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="493" data-strategy="0" cx="210.62746383129547" cy="241.36049563777385" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="494" data-strategy="0" cx="150.06108720502823" cy="230.88404862174835" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="495" data-strategy="0" cx="160.35725436195747" cy="203.47976628425644" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="496" data-strategy="0" cx="168.79066466958088" cy="206.52565489421667" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="497" data-strategy="0" cx="181.9274131377793" cy="203.03168623690215" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="498" data-strategy="0" cx="174.83130055660877" cy="228.58268956508024" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="499" data-strategy="0" cx="215.71586601400753" cy="270.20362002047625" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="500" data-strategy="0" cx="178.45529356414062" cy="306.4315394450393" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="501" data-strategy="0" cx="168.23858974482366" cy="277.7322853149035" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="502" data-strategy="0" cx="169.66808027851303" cy="289.577934910028" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="503" data-strategy="0" cx="182.61610193704803" cy="324.66170570193214" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="526" data-strategy="0" cx="198.60978301682206" cy="230.67063632028368" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="534" data-strategy="0" cx="192.4682187823279" cy="268.6099172218062" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="574" data-strategy="0" cx="213.79820155442604" cy="176.90404535336307" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="575" data-strategy="0" cx="159.8568750933587" cy="184.5365784167763" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="576" data-strategy="0" cx="205.50658389310186" cy="214.64781440428968" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="577" data-strategy="0" cx="185.36846484771212" cy="233.51548441876585" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="578" data-strategy="0" cx="143.1051902842274" cy="295.5683324512477" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="579" data-strategy="0" cx="162.6969578903053" cy="299.67857065013703" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="580" data-strategy="0" cx="134.7845254781825" cy="289.6738330373345" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="581" data-strategy="0" cx="183.93867464533028" cy="292.1168303280535" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="582" data-strategy="0" cx="199.77299644491828" cy="300.7795158553745" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="583" data-strategy="0" cx="176.5909299055081" cy="319.07502683199544" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="584" data-strategy="0" cx="193.4668791086755" cy="324.5818579625751" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="585" data-strategy="0" cx="141.64879217326936" cy="364.5214005040657" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="586" data-strategy="0" cx="150.52918753255693" cy="343.61201151884813" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="587" data-strategy="0" cx="176.42898863922096" cy="312.04800512101326" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="588" data-strategy="0" cx="165.7955366883589" cy="287.91317239757007" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="589" data-strategy="0" cx="179.81454711341547" cy="325.86459125413586" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="590" data-strategy="0" cx="136.97378696067926" cy="296.30900593004355" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="591" data-strategy="0" cx="146.60222660278663" cy="291.86933201227527" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="592" data-strategy="0" cx="213.91581526807298" cy="369.4753627103855" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="593" data-strategy="0" cx="167.2039103915362" cy="403.0061938856326" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="594" data-strategy="0" cx="211.54300149039847" cy="383.6316910499623" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="595" data-strategy="0" cx="155.2365927342071" cy="357.23602151994976" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="596" data-strategy="0" cx="137.08487066075824" cy="378.9980304643559" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="597" data-strategy="0" cx="177.3957353336527" cy="423.3819262541628" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="598" data-strategy="0" cx="156.53527109240588" cy="412.80590937524875" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="599" data-strategy="0" cx="219.37242965652797" cy="434.33991124176697" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="600" data-strategy="0" cx="145.55864719091878" cy="478.67993904936105" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="601" data-strategy="0" cx="152.13948185972868" cy="471.8240655464358" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="602" data-strategy="0" cx="212.51169784481465" cy="428.72525720884397" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="603" data-strategy="0" cx="144.13837456697286" cy="430.8933928988819" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="604" data-strategy="0" cx="176.4880462603924" cy="410.1025592673431" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="605" data-strategy="0" cx="157.42222802697657" cy="384.7222108795846" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="606" data-strategy="0" cx="208.12338786212484" cy="375.7370249870088" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="607" data-strategy="0" cx="173.97441946360235" cy="390.1044125562117" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="608" data-strategy="0" cx="193.00778349663887" cy="349.2267205540495" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="609" data-strategy="0" cx="185.622562845659" cy="334.8849816357073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="610" data-strategy="0" cx="219.22691072628768" cy="352.1203369966647" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="611" data-strategy="0" cx="134.1286994960276" cy="347.7616955020778" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="612" data-strategy="0" cx="197.7278330396575" cy="316.756402849348" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="613" data-strategy="0" cx="159.4852816206037" cy="337.8149205809082" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="614" data-strategy="0" cx="201.6198780819789" cy="341.7308750085009" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="615" data-strategy="0" cx="134.54655391541385" cy="314.48705448264616" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="616" data-strategy="0" cx="217.47580059886755" cy="273.84734337675746" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="617" data-strategy="0" cx="188.03175574765328" cy="292.8749007571832" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="618" data-strategy="0" cx="155.2660721956815" cy="343.31017318378764" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="619" data-strategy="0" cx="135.16092485916653" cy="330.71086489247995" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="620" data-strategy="0" cx="202.6788191721163" cy="287.30934215205355" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="621" data-strategy="0" cx="194.71508905839022" cy="256.8773891796348" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="622" data-strategy="0" cx="168.4425798640067" cy="224.79440654854935" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="623" data-strategy="0" cx="145.75275799081723" cy="182.74960612337793" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="624" data-strategy="0" cx="208.06241807989022" cy="140.23658810299844" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="625" data-strategy="0" cx="218.45501026309813" cy="203.7061851332917" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="626" data-strategy="0" cx="150.4711550199496" cy="271.3326675552575" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="627" data-strategy="0" cx="169.45199312849212" cy="271.9278734746151" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="628" data-strategy="0" cx="146.31206471964708" cy="289.45377479636306" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="629" data-strategy="0" cx="137.08303928198248" cy="297.84005946519613" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="630" data-strategy="0" cx="181.36071775803566" cy="289.3386095438977" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="631" data-strategy="0" cx="140.3182314071019" cy="305.2489722641564" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="632" data-strategy="0" cx="200.45366807385727" cy="316.63293568398507" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="633" data-strategy="0" cx="145.37398640220198" cy="342.34326197596283" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="634" data-strategy="0" cx="219.89345039959704" cy="357.9638288739628" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="635" data-strategy="0" cx="169.45209196562828" cy="355.2205650514456" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="636" data-strategy="0" cx="133.90472606964408" cy="357.245630321493" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="637" data-strategy="0" cx="153.51625919691816" cy="329.5340892591001" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="638" data-strategy="0" cx="142.45006598036434" cy="302.3003378448568" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="639" data-strategy="0" cx="200.34248535031855" cy="293.6852324786018" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="640" data-strategy="0" cx="143.56960631863018" cy="287.45040232948435" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="641" data-strategy="0" cx="152.98099040821216" cy="288.8504041375106" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="642" data-strategy="0" cx="162.5467497726588" cy="281.76989427532374" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="643" data-strategy="0" cx="171.37137138344667" cy="292.0621672473586" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="644" data-strategy="0" cx="151.40928720156379" cy="304.0686255157788" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="645" data-strategy="0" cx="196.43509864046945" cy="301.53139655702955" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="646" data-strategy="0" cx="168.10821774751102" cy="281.73431569511934" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="647" data-strategy="0" cx="206.05213872831263" cy="277.6068837522979" r="3" fill="#94a3b8" fill-opacity=".65"></circle><text x="177.25" y="572" text-anchor="middle" >REF｜流動性等權</text><circle class="point" data-window="1" data-strategy="1" cx="324.74953973423897" cy="321.35149628260405" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="2" data-strategy="1" cx="361.31063192910807" cy="314.65708155433" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="3" data-strategy="1" cx="334.6116079567268" cy="288.66803483111653" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="4" data-strategy="1" cx="367.5413550560375" cy="268.2741300380166" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="5" data-strategy="1" cx="388.81228289396523" cy="234.4685336901594" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="6" data-strategy="1" cx="409.4870769679999" cy="244.0336377070695" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="7" data-strategy="1" cx="364.3494881879398" cy="262.33584145589816" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="8" data-strategy="1" cx="391.3425186994773" cy="252.7218804319248" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="9" data-strategy="1" cx="342.9776133835102" cy="248.4326200238727" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="10" data-strategy="1" cx="400.30848408447656" cy="226.3248815385319" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="11" data-strategy="1" cx="389.33836034657287" cy="229.30161345670376" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="12" data-strategy="1" cx="356.70269072854416" cy="233.65150689653044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="13" data-strategy="1" cx="395.1714711972865" cy="229.83017420593958" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="14" data-strategy="1" cx="366.7993537806495" cy="213.56207243570185" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="15" data-strategy="1" cx="398.06112527661924" cy="195.6095834218011" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="16" data-strategy="1" cx="346.56966031238267" cy="214.37077644902513" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="17" data-strategy="1" cx="344.4122685499242" cy="222.61528139781933" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="39" data-strategy="1" cx="332.41329614223406" cy="318.15863243934797" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="44" data-strategy="1" cx="332.63711365035107" cy="309.5927558629432" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="49" data-strategy="1" cx="398.0802945470745" cy="365.93057372467456" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="54" data-strategy="1" cx="409.17672128410453" cy="334.2593024596147" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="59" data-strategy="1" cx="411.1549200168957" cy="326.37639867390794" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="67" data-strategy="1" cx="336.6756987642147" cy="344.1412809745739" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="70" data-strategy="1" cx="403.35374298889565" cy="321.40751832436786" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="72" data-strategy="1" cx="401.4109407441761" cy="287.60038272067453" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="75" data-strategy="1" cx="329.85257547591755" cy="276.26113181555627" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="77" data-strategy="1" cx="328.72139094575635" cy="325.66121505915504" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="80" data-strategy="1" cx="328.28349386466454" cy="297.47583140551893" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="82" data-strategy="1" cx="376.41541369496986" cy="309.34669442168814" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="83" data-strategy="1" cx="391.45534184030737" cy="288.1135690052122" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="85" data-strategy="1" cx="335.84066776362624" cy="284.50646827818315" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="87" data-strategy="1" cx="353.8854412084679" cy="288.28059263665614" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="88" data-strategy="1" cx="409.9597344179715" cy="265.45361697622513" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="90" data-strategy="1" cx="343.45132422207246" cy="284.0989201410209" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="93" data-strategy="1" cx="371.89830903580855" cy="281.72450423272574" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="96" data-strategy="1" cx="362.51978306274" cy="300.92537847384403" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="98" data-strategy="1" cx="336.5247735041682" cy="313.7359536462737" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="101" data-strategy="1" cx="329.0917556534977" cy="280.188089438417" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="103" data-strategy="1" cx="402.95253965320126" cy="282.89358068614706" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="106" data-strategy="1" cx="379.760832446435" cy="295.86325580522583" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="111" data-strategy="1" cx="348.7355263051176" cy="333.8764549048268" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="114" data-strategy="1" cx="387.79959185213505" cy="360.0926228861866" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="115" data-strategy="1" cx="410.6862705334223" cy="349.35147249235087" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="116" data-strategy="1" cx="391.93467600161205" cy="334.4822373847468" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="117" data-strategy="1" cx="350.5430707066441" cy="357.3943093536139" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="118" data-strategy="1" cx="409.13731167737296" cy="338.8921710307605" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="119" data-strategy="1" cx="345.87811954153096" cy="365.9790457401278" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="120" data-strategy="1" cx="329.97301383117633" cy="398.81537677551677" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="121" data-strategy="1" cx="409.61045207580503" cy="397.5178750981654" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="122" data-strategy="1" cx="329.21725248925725" cy="386.26273115705806" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="123" data-strategy="1" cx="348.8148428899339" cy="385.3930933885729" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="124" data-strategy="1" cx="386.05255919111977" cy="385.6472887222381" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="125" data-strategy="1" cx="349.3541457574297" cy="388.1237616654774" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="126" data-strategy="1" cx="374.26276427499585" cy="379.3866194096511" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="127" data-strategy="1" cx="355.26527937402665" cy="380.50867361674307" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="128" data-strategy="1" cx="409.8947379896459" cy="386.69554396200925" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="129" data-strategy="1" cx="334.5175606405108" cy="378.6516436358465" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="130" data-strategy="1" cx="359.70834319341316" cy="374.7322358009821" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="131" data-strategy="1" cx="365.4058627548849" cy="361.4128043735557" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="132" data-strategy="1" cx="329.39709643896435" cy="364.1459328550746" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="133" data-strategy="1" cx="377.87600873214745" cy="349.089544987357" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="134" data-strategy="1" cx="361.36406122213975" cy="345.91968586009915" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="135" data-strategy="1" cx="342.08907685597575" cy="329.18137187542015" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="136" data-strategy="1" cx="385.9886072330136" cy="326.647850943583" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="137" data-strategy="1" cx="333.3134786485635" cy="307.5198352292127" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="138" data-strategy="1" cx="405.8460569419152" cy="308.92780689346944" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="139" data-strategy="1" cx="387.0613712343054" cy="309.1633624144938" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="140" data-strategy="1" cx="383.9783883238853" cy="326.62643817190695" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="141" data-strategy="1" cx="407.34162881980654" cy="329.95234875265805" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="142" data-strategy="1" cx="379.25669827089814" cy="327.99606495870523" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="143" data-strategy="1" cx="411.0831915094615" cy="299.1243553293983" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="144" data-strategy="1" cx="411.2376866756991" cy="294.1934422059217" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="145" data-strategy="1" cx="363.5971701508731" cy="306.5386774140952" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="146" data-strategy="1" cx="356.0241437399048" cy="320.5857906967723" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="147" data-strategy="1" cx="349.70055869650656" cy="341.93340982772975" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="148" data-strategy="1" cx="393.33695490713467" cy="326.53412642654683" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="149" data-strategy="1" cx="336.22970499834105" cy="321.07650905683334" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="150" data-strategy="1" cx="334.13102589027585" cy="301.09818145177053" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="151" data-strategy="1" cx="368.8593767126205" cy="315.98195116255437" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="152" data-strategy="1" cx="410.1491220130596" cy="318.63091229979375" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="153" data-strategy="1" cx="375.9364628244731" cy="317.6272236008022" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="154" data-strategy="1" cx="407.1656410125579" cy="325.20094469540714" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="155" data-strategy="1" cx="345.8873346116183" cy="306.158967677718" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="156" data-strategy="1" cx="383.2879267615253" cy="311.35311648976705" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="157" data-strategy="1" cx="341.66381076887654" cy="322.9505869395788" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="158" data-strategy="1" cx="364.10488150556904" cy="337.0091685208325" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="159" data-strategy="1" cx="360.5092053050299" cy="344.5449543387434" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="160" data-strategy="1" cx="397.78091734839" cy="321.99455651522453" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="161" data-strategy="1" cx="406.51850100732554" cy="325.4584251584379" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="162" data-strategy="1" cx="358.81692482172974" cy="326.98626144830456" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="163" data-strategy="1" cx="347.03920503200476" cy="329.185099116527" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="164" data-strategy="1" cx="361.45047098743925" cy="323.81313207070184" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="165" data-strategy="1" cx="371.3162145811199" cy="295.602220382909" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="166" data-strategy="1" cx="383.37784869441083" cy="289.9571262817011" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="167" data-strategy="1" cx="408.0944731245418" cy="295.4541010910643" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="169" data-strategy="1" cx="359.7046641853432" cy="292.6189734868354" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="174" data-strategy="1" cx="406.5602081149077" cy="319.75336265152924" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="185" data-strategy="1" cx="357.0408828352182" cy="308.71015650912136" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="190" data-strategy="1" cx="327.4572696902504" cy="345.14097466741777" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="195" data-strategy="1" cx="396.0751990801356" cy="316.47737652650693" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="200" data-strategy="1" cx="363.4765616980361" cy="285.84824621567316" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="202" data-strategy="1" cx="357.551205899057" cy="268.785065477107" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="207" data-strategy="1" cx="408.53125997138017" cy="319.4370106769101" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="212" data-strategy="1" cx="378.9135039978045" cy="286.573854499549" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="217" data-strategy="1" cx="407.39747799374476" cy="296.7771689046753" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="222" data-strategy="1" cx="409.6681847264458" cy="295.9588291365508" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="260" data-strategy="1" cx="397.9370778734677" cy="337.17008392803456" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="302" data-strategy="1" cx="402.7548053779679" cy="303.7787890428877" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="303" data-strategy="1" cx="401.4758436771192" cy="276.4182680583625" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="350" data-strategy="1" cx="363.8345805675148" cy="273.4091631096051" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="351" data-strategy="1" cx="406.590076265173" cy="266.15293242875475" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="352" data-strategy="1" cx="374.73536354394236" cy="283.0770906995684" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="353" data-strategy="1" cx="344.48936156594857" cy="309.1415525772781" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="354" data-strategy="1" cx="354.26033059670556" cy="287.39584927950966" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="355" data-strategy="1" cx="380.80806793136554" cy="285.70847578889" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="356" data-strategy="1" cx="393.7138729738367" cy="296.65225881937846" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="357" data-strategy="1" cx="405.30800937833345" cy="293.07391604018545" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="358" data-strategy="1" cx="382.47877841004225" cy="299.77259905636475" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="359" data-strategy="1" cx="393.7848212389597" cy="283.99101173866785" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="360" data-strategy="1" cx="326.9837435449687" cy="284.64375763792395" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="361" data-strategy="1" cx="335.68221898157304" cy="276.1687271260992" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="362" data-strategy="1" cx="399.66997700045914" cy="283.14803968628496" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="363" data-strategy="1" cx="408.7449916708546" cy="292.9537149043071" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="364" data-strategy="1" cx="353.133070532259" cy="283.02702273880584" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="365" data-strategy="1" cx="326.5840455840577" cy="272.1966511746839" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="366" data-strategy="1" cx="344.868612305459" cy="249.57790445290001" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="367" data-strategy="1" cx="388.8287083668169" cy="244.98697898337568" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="368" data-strategy="1" cx="338.31801041567707" cy="248.1400349312081" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="369" data-strategy="1" cx="353.3094564061968" cy="231.29476961474523" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="370" data-strategy="1" cx="405.2395140315259" cy="199.5708090229557" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="371" data-strategy="1" cx="335.6958292348238" cy="205.23028623913757" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="372" data-strategy="1" cx="340.34478683449026" cy="191.31122797258394" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="373" data-strategy="1" cx="404.7582561961752" cy="210.73765871201044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="374" data-strategy="1" cx="330.0228935423773" cy="242.64224479482115" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="375" data-strategy="1" cx="354.92138059638825" cy="219.08038037383133" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="376" data-strategy="1" cx="367.4293913440796" cy="234.03993368808167" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="377" data-strategy="1" cx="409.6444608085874" cy="215.29209620629064" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="378" data-strategy="1" cx="354.0808790735192" cy="204.06362248875698" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="379" data-strategy="1" cx="384.69973819574466" cy="189.9902702770485" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="380" data-strategy="1" cx="353.12397768476336" cy="197.1904469631212" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="381" data-strategy="1" cx="405.26066726635906" cy="179.14507128805047" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="382" data-strategy="1" cx="407.75435855633975" cy="207.33144112105268" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="383" data-strategy="1" cx="363.6992795659145" cy="235.76928634624667" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="384" data-strategy="1" cx="376.4779377542336" cy="251.85752113959754" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="385" data-strategy="1" cx="361.8029459604213" cy="226.0220110606896" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="386" data-strategy="1" cx="346.19712655177557" cy="215.71573091048995" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="387" data-strategy="1" cx="354.387473980466" cy="221.7456599321246" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="388" data-strategy="1" cx="346.33599317002245" cy="225.67803835786808" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="389" data-strategy="1" cx="376.2207804507165" cy="235.48012571173976" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="390" data-strategy="1" cx="404.2972762972342" cy="255.21916065120553" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="391" data-strategy="1" cx="370.191917897451" cy="269.97154785443325" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="392" data-strategy="1" cx="408.9128071984634" cy="305.57099240932473" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="393" data-strategy="1" cx="346.34952168360064" cy="302.9297673979818" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="394" data-strategy="1" cx="342.84339140922407" cy="316.4986599089826" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="406" data-strategy="1" cx="367.6630923318705" cy="255.94956635515695" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="411" data-strategy="1" cx="334.36746692513" cy="288.2189060173814" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="416" data-strategy="1" cx="372.0005048821751" cy="248.1859136867247" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="421" data-strategy="1" cx="403.1820446760465" cy="247.51704922957936" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="426" data-strategy="1" cx="387.483018059274" cy="237.42412783459682" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="432" data-strategy="1" cx="405.8622766707862" cy="248.96899321772815" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="433" data-strategy="1" cx="331.13141203887847" cy="244.057083509556" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="434" data-strategy="1" cx="396.26899611283494" cy="268.6211310019344" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="435" data-strategy="1" cx="378.53766434711906" cy="286.79782228050624" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="436" data-strategy="1" cx="329.0288374660411" cy="267.1678166070252" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="437" data-strategy="1" cx="324.7954995559685" cy="297.59619128500503" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="438" data-strategy="1" cx="382.3626014556905" cy="299.1948569805584" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="439" data-strategy="1" cx="394.72152674316266" cy="299.7044770766834" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="440" data-strategy="1" cx="383.92604675231814" cy="288.3168217168117" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="441" data-strategy="1" cx="357.0217301226166" cy="306.68065288075377" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="442" data-strategy="1" cx="379.6220635636217" cy="298.7880489131205" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="464" data-strategy="1" cx="398.2701016683018" cy="277.3019284492139" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="469" data-strategy="1" cx="343.07708629250027" cy="246.30372121910352" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="473" data-strategy="1" cx="372.1663206034106" cy="251.02598501896372" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="474" data-strategy="1" cx="328.3365854674125" cy="243.76755873808133" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="478" data-strategy="1" cx="361.0263181017303" cy="211.5477254584955" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="479" data-strategy="1" cx="367.5711216073085" cy="183.97142819494488" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="481" data-strategy="1" cx="325.005833962198" cy="190.11674845627013" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="482" data-strategy="1" cx="397.65061852577185" cy="180.41503507630983" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="483" data-strategy="1" cx="395.7320738959209" cy="172.23978826497998" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="484" data-strategy="1" cx="377.5241994109895" cy="191.91064622891827" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="485" data-strategy="1" cx="402.2381539609989" cy="198.18604929009413" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="486" data-strategy="1" cx="396.7568497031574" cy="220.26034420872946" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="487" data-strategy="1" cx="402.74364054422495" cy="206.49352169755477" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="488" data-strategy="1" cx="390.2213130218789" cy="201.9263769013287" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="489" data-strategy="1" cx="396.0207318223975" cy="221.17033520856344" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="490" data-strategy="1" cx="390.59147425810227" cy="267.1190613995526" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="491" data-strategy="1" cx="370.47690148042904" cy="241.83998208939474" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="492" data-strategy="1" cx="342.0330407965628" cy="238.7550114971827" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="493" data-strategy="1" cx="328.47533523455553" cy="215.13077349108195" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="494" data-strategy="1" cx="368.69567829191925" cy="210.6304851485625" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="495" data-strategy="1" cx="389.41198064572995" cy="224.18120667435628" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="496" data-strategy="1" cx="329.6009493721552" cy="186.61393447007552" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="497" data-strategy="1" cx="396.04930388569727" cy="171.74988701165273" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="498" data-strategy="1" cx="373.61850152088977" cy="190.76452702607475" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="499" data-strategy="1" cx="345.3119452788393" cy="224.603676504411" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="500" data-strategy="1" cx="361.90369119567146" cy="305.79692774704216" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="501" data-strategy="1" cx="389.6025994006081" cy="254.5940962546501" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="502" data-strategy="1" cx="382.93036043877214" cy="257.5856739927633" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="503" data-strategy="1" cx="395.362808828907" cy="306.7203482825769" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="526" data-strategy="1" cx="328.4644387966921" cy="88.64351992091534" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="534" data-strategy="1" cx="334.4166292745055" cy="178.55059260206184" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="574" data-strategy="1" cx="340.56811590584437" cy="188.16391668209377" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="575" data-strategy="1" cx="400.49375168804556" cy="197.3018823412031" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="576" data-strategy="1" cx="410.81658523526346" cy="226.94841299509406" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="577" data-strategy="1" cx="409.6328563060187" cy="240.70727552245677" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="578" data-strategy="1" cx="362.087648246906" cy="310.30974056951504" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="579" data-strategy="1" cx="383.3135843929486" cy="324.61369934493104" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="580" data-strategy="1" cx="329.3167735182386" cy="313.20689137435085" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="581" data-strategy="1" cx="333.02691284729184" cy="298.2046369014537" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="582" data-strategy="1" cx="336.5406312207552" cy="311.3357662126072" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="583" data-strategy="1" cx="392.5233285582172" cy="314.36043201525086" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="584" data-strategy="1" cx="395.28889696453854" cy="299.3623750664687" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="585" data-strategy="1" cx="386.4208482017468" cy="339.3257503253278" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="586" data-strategy="1" cx="374.25961320631234" cy="307.36602138692433" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="587" data-strategy="1" cx="401.91017177096865" cy="293.4255118806723" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="588" data-strategy="1" cx="357.09765001437216" cy="241.70433378562257" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="589" data-strategy="1" cx="396.6460913464388" cy="264.64471677519293" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="590" data-strategy="1" cx="366.3476910460455" cy="233.01299251366478" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="591" data-strategy="1" cx="373.1007400780739" cy="230.87621504839075" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="592" data-strategy="1" cx="398.95613977161054" cy="325.7066820840883" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="593" data-strategy="1" cx="395.53112530357674" cy="332.0852242110152" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="594" data-strategy="1" cx="324.265373420904" cy="328.1726646067595" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="595" data-strategy="1" cx="363.4479023059295" cy="317.98611294081803" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="596" data-strategy="1" cx="360.5011173898319" cy="363.8112211438054" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="597" data-strategy="1" cx="337.09221067510487" cy="418.7618842202054" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="598" data-strategy="1" cx="333.9908005546638" cy="372.8190021522661" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="599" data-strategy="1" cx="405.28293237035126" cy="391.88745079508806" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="600" data-strategy="1" cx="362.09630916591993" cy="433.2040047982375" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="601" data-strategy="1" cx="381.4454039496158" cy="434.61947400212983" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="602" data-strategy="1" cx="385.7122695287451" cy="409.99155624074024" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="603" data-strategy="1" cx="334.5182226928899" cy="381.4830370734599" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="604" data-strategy="1" cx="387.1915982392598" cy="403.0193269674708" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="605" data-strategy="1" cx="376.1788916795319" cy="412.5129904379545" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="606" data-strategy="1" cx="326.60866153971216" cy="410.04224445177044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="607" data-strategy="1" cx="371.14314829976644" cy="405.71415826437965" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="608" data-strategy="1" cx="339.6472221610153" cy="356.6291693887781" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="609" data-strategy="1" cx="357.60528931798274" cy="377.1510331023646" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="610" data-strategy="1" cx="362.29693364846025" cy="397.28633069562073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="611" data-strategy="1" cx="377.0331072086275" cy="399.3784134964464" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="612" data-strategy="1" cx="399.3417474535468" cy="360.5946957945271" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="613" data-strategy="1" cx="333.6971503728213" cy="364.1252877425548" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="614" data-strategy="1" cx="391.3534102934337" cy="391.6778197234068" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="615" data-strategy="1" cx="405.0193101776948" cy="367.45043640564086" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="616" data-strategy="1" cx="390.9315553662828" cy="348.2223282943421" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="617" data-strategy="1" cx="346.3886116691494" cy="347.93949922277585" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="618" data-strategy="1" cx="354.54902641398826" cy="356.93438704946266" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="619" data-strategy="1" cx="392.4636925752467" cy="368.6938237749635" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="620" data-strategy="1" cx="398.2080473990567" cy="344.92086259775186" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="621" data-strategy="1" cx="374.9926966225173" cy="315.4790032830979" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="622" data-strategy="1" cx="349.14140003916566" cy="277.3747387161294" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="623" data-strategy="1" cx="357.0106650283714" cy="237.77211389343753" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="624" data-strategy="1" cx="362.8979239549336" cy="257.4604091480338" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="625" data-strategy="1" cx="334.2312615366984" cy="304.34323808086845" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="626" data-strategy="1" cx="402.10640860200976" cy="338.5178547814147" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="627" data-strategy="1" cx="363.0667383666272" cy="305.029877526833" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="628" data-strategy="1" cx="331.02030801149033" cy="298.6500790677671" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="629" data-strategy="1" cx="341.24389991123866" cy="326.04569651741735" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="630" data-strategy="1" cx="395.41064894732335" cy="323.64300659261" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="631" data-strategy="1" cx="340.80472086351983" cy="337.2180769478683" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="632" data-strategy="1" cx="385.52129713642955" cy="330.00013899562737" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="633" data-strategy="1" cx="401.9265931151249" cy="326.46811969999067" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="634" data-strategy="1" cx="335.70495640233435" cy="367.69013385891213" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="635" data-strategy="1" cx="379.45891447155935" cy="362.88556560745815" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="636" data-strategy="1" cx="341.71653654575437" cy="356.65448480980405" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="637" data-strategy="1" cx="344.25706618190003" cy="335.45785350518554" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="638" data-strategy="1" cx="352.6160023668083" cy="314.9113964437104" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="639" data-strategy="1" cx="365.9125260266355" cy="320.88078173521944" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="640" data-strategy="1" cx="374.257779636714" cy="325.9311959388122" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="641" data-strategy="1" cx="326.513713465343" cy="320.34050784209364" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="642" data-strategy="1" cx="345.2376783961763" cy="318.80134111972194" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="643" data-strategy="1" cx="332.08914855044367" cy="334.7499473266163" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="644" data-strategy="1" cx="381.62618073108774" cy="352.7609287690523" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="645" data-strategy="1" cx="396.20830045633215" cy="338.98899256495065" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="646" data-strategy="1" cx="331.8026022846383" cy="331.9898623674496" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="647" data-strategy="1" cx="352.64679446924407" cy="346.31066069928977" r="3" fill="#94a3b8" fill-opacity=".65"></circle><text x="367.75" y="572" text-anchor="middle" >A｜每週動能</text><circle class="point" data-window="1" data-strategy="2" cx="539.1677459221409" cy="325.5446538954254" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="2" data-strategy="2" cx="567.5033193521155" cy="320.95681377254925" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="3" data-strategy="2" cx="549.4360851533936" cy="294.3271743170081" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="4" data-strategy="2" cx="544.3634792926005" cy="281.0225764607026" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="5" data-strategy="2" cx="537.5674024829768" cy="243.30926644775548" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="6" data-strategy="2" cx="592.9204290941973" cy="253.83496580934832" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="7" data-strategy="2" cx="517.8922697368937" cy="269.2476845636821" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="8" data-strategy="2" cx="591.8728612404616" cy="263.8938338639888" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="9" data-strategy="2" cx="584.672073085886" cy="256.83080309360366" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="10" data-strategy="2" cx="555.9559976980149" cy="234.14947167531534" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="11" data-strategy="2" cx="560.3716920432394" cy="217.05503256108906" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="12" data-strategy="2" cx="561.1689173773309" cy="242.1330356869906" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="13" data-strategy="2" cx="566.3979237291601" cy="239.85102003007685" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="14" data-strategy="2" cx="541.2547567246658" cy="222.11228891083655" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="15" data-strategy="2" cx="539.9981686061747" cy="208.09995977459664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="16" data-strategy="2" cx="544.7537502769021" cy="198.1529574423669" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="17" data-strategy="2" cx="548.0384821530224" cy="225.60586044408225" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="39" data-strategy="2" cx="569.4714298207934" cy="325.5459595523927" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="44" data-strategy="2" cx="529.750171372029" cy="322.7869536349619" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="49" data-strategy="2" cx="600.3078718674642" cy="378.06954579830114" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="54" data-strategy="2" cx="582.0217255613939" cy="345.20792716305516" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="59" data-strategy="2" cx="525.2137418637931" cy="334.38401437461624" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="67" data-strategy="2" cx="597.0882356510467" cy="336.5930170304213" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="70" data-strategy="2" cx="594.136197926164" cy="316.1936425972108" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="72" data-strategy="2" cx="566.8659166827885" cy="285.5239601258772" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="75" data-strategy="2" cx="589.181973164145" cy="283.6422405552951" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="77" data-strategy="2" cx="548.0094580286174" cy="327.3335966404409" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="80" data-strategy="2" cx="580.7504340848772" cy="302.2380940727409" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="82" data-strategy="2" cx="575.2697083628702" cy="305.24759598162586" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="83" data-strategy="2" cx="562.027162997248" cy="292.2580363025425" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="85" data-strategy="2" cx="564.8267736625112" cy="297.2388440243075" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="87" data-strategy="2" cx="592.6212592334157" cy="293.23849623163557" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="88" data-strategy="2" cx="537.600785223843" cy="275.46150180480527" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="90" data-strategy="2" cx="540.4364705928247" cy="294.8365703308996" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="93" data-strategy="2" cx="599.2549363856538" cy="279.0523784500067" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="96" data-strategy="2" cx="586.1988301172304" cy="291.7644518379211" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="98" data-strategy="2" cx="524.2719177100138" cy="303.95547240727154" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="101" data-strategy="2" cx="526.3557482034628" cy="269.74514788266896" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="103" data-strategy="2" cx="583.7540618080204" cy="271.6508040687121" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="106" data-strategy="2" cx="534.0497518880144" cy="285.2144126882562" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="111" data-strategy="2" cx="518.6417725805863" cy="331.33440779001415" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="114" data-strategy="2" cx="568.6560599527752" cy="361.16784014446694" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="115" data-strategy="2" cx="536.4240584058027" cy="339.0481373089239" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="116" data-strategy="2" cx="527.3235418675625" cy="339.16315124178686" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="117" data-strategy="2" cx="595.5272863548641" cy="350.8960130605071" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="118" data-strategy="2" cx="564.5863669653146" cy="343.61794158792407" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="119" data-strategy="2" cx="578.4398186743608" cy="366.164024493905" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="120" data-strategy="2" cx="545.6843657152276" cy="384.1236712228547" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="121" data-strategy="2" cx="589.842190263122" cy="393.0486540225506" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="122" data-strategy="2" cx="571.9039434103786" cy="382.55850882388035" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="123" data-strategy="2" cx="543.138018143538" cy="389.49113694253884" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="124" data-strategy="2" cx="580.7329027602866" cy="388.88649358801274" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="125" data-strategy="2" cx="572.4317169524469" cy="377.7076542366228" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="126" data-strategy="2" cx="574.1860110435222" cy="379.68915235181197" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="127" data-strategy="2" cx="537.4179118872125" cy="379.6078976676214" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="128" data-strategy="2" cx="594.7839914063309" cy="390.8627962339033" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="129" data-strategy="2" cx="569.7258397623384" cy="388.02633060479155" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="130" data-strategy="2" cx="529.1104509417919" cy="371.07390984646173" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="131" data-strategy="2" cx="595.8756310080065" cy="366.63132467567056" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="132" data-strategy="2" cx="577.9503771033085" cy="366.0139883856044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="133" data-strategy="2" cx="583.1586574494133" cy="355.16141611211293" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="134" data-strategy="2" cx="578.6583601490095" cy="358.0378344018883" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="135" data-strategy="2" cx="581.6089947158898" cy="327.58675649768975" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="136" data-strategy="2" cx="536.665119944588" cy="331.4022545142112" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="137" data-strategy="2" cx="519.3366467265633" cy="313.260165667661" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="138" data-strategy="2" cx="566.73764151152" cy="315.31133508678755" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="139" data-strategy="2" cx="563.8242801347265" cy="324.39084651084744" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="140" data-strategy="2" cx="554.0297837860219" cy="328.80817862755464" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="141" data-strategy="2" cx="540.2690173612839" cy="339.59978838810696" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="142" data-strategy="2" cx="576.0620875788653" cy="333.52491821626677" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="143" data-strategy="2" cx="583.5391972427818" cy="301.0125534790073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="144" data-strategy="2" cx="530.7261060367495" cy="306.66006853042563" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="145" data-strategy="2" cx="582.2018365540923" cy="312.40540026900976" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="146" data-strategy="2" cx="534.507454171911" cy="325.078247340236" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="147" data-strategy="2" cx="600.7162708378969" cy="336.50100240845825" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="148" data-strategy="2" cx="551.9699825064763" cy="326.422431776476" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="149" data-strategy="2" cx="573.5640427503442" cy="323.3515391790749" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="150" data-strategy="2" cx="556.7038701480257" cy="308.23743141441497" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="151" data-strategy="2" cx="522.0431600147781" cy="321.5849470062719" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="152" data-strategy="2" cx="570.2521624995153" cy="315.41435410692236" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="153" data-strategy="2" cx="564.6615319899564" cy="316.18723520080187" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="154" data-strategy="2" cx="539.5228398601231" cy="320.46866796655473" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="155" data-strategy="2" cx="579.2436860507185" cy="309.0623480376358" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="156" data-strategy="2" cx="534.636369410157" cy="312.86315628944266" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="157" data-strategy="2" cx="601.5064280539066" cy="315.5770693741165" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="158" data-strategy="2" cx="544.4888855695273" cy="331.2394191804966" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="159" data-strategy="2" cx="579.384598178041" cy="339.77035921445525" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="160" data-strategy="2" cx="551.2852472922928" cy="319.80950700698475" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="161" data-strategy="2" cx="592.5140319724517" cy="323.68488125133644" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="162" data-strategy="2" cx="536.2569477296313" cy="325.1238882496353" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="163" data-strategy="2" cx="531.6522106985944" cy="327.704142283197" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="164" data-strategy="2" cx="544.1531392497944" cy="326.1090034226025" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="165" data-strategy="2" cx="538.2239850865378" cy="296.5654719124083" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="166" data-strategy="2" cx="557.7492365732262" cy="286.57511671798017" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="167" data-strategy="2" cx="577.2305602219922" cy="301.4011111964025" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="169" data-strategy="2" cx="534.7295137147138" cy="302.92505009400185" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="174" data-strategy="2" cx="525.6788548292141" cy="331.28705695253547" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="185" data-strategy="2" cx="578.8526041312239" cy="307.95908008707664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="190" data-strategy="2" cx="572.7219043331122" cy="337.0172527579785" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="195" data-strategy="2" cx="581.8800287298486" cy="311.1609164313002" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="200" data-strategy="2" cx="516.9806093795025" cy="284.7793217554707" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="202" data-strategy="2" cx="544.7600063188587" cy="264.1155595204298" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="207" data-strategy="2" cx="577.1355484875377" cy="311.83925695156415" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="212" data-strategy="2" cx="535.1357035409459" cy="289.42927692366504" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="217" data-strategy="2" cx="515.9172590199997" cy="304.76880399935067" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="222" data-strategy="2" cx="530.5308378797981" cy="304.6133025659893" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="260" data-strategy="2" cx="570.5047788713805" cy="340.60346437009446" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="302" data-strategy="2" cx="558.7291612227855" cy="302.30074877822784" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="303" data-strategy="2" cx="595.8450617739045" cy="277.7590995356784" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="350" data-strategy="2" cx="556.5658292318143" cy="278.80056031165094" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="351" data-strategy="2" cx="560.2743910024324" cy="273.75681406364197" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="352" data-strategy="2" cx="558.0266404220945" cy="290.6726146816874" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="353" data-strategy="2" cx="563.389395802889" cy="313.50832591823996" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="354" data-strategy="2" cx="593.2519714345515" cy="299.46087167138097" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="355" data-strategy="2" cx="555.7042533043335" cy="290.2772066426376" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="356" data-strategy="2" cx="523.1042830700658" cy="299.4042352172795" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="357" data-strategy="2" cx="584.9451009397416" cy="302.78980262096223" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="358" data-strategy="2" cx="551.7250691517378" cy="312.1218919149372" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="359" data-strategy="2" cx="540.97287143197" cy="297.03837006980405" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="360" data-strategy="2" cx="521.86497068463" cy="282.4261370929278" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="361" data-strategy="2" cx="517.9154202059459" cy="277.19113119404216" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="362" data-strategy="2" cx="536.8518479133038" cy="295.3982559228954" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="363" data-strategy="2" cx="567.2557781487618" cy="303.0579157614585" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="364" data-strategy="2" cx="548.1849611760524" cy="293.9960103447819" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="365" data-strategy="2" cx="598.8806441317096" cy="276.13189279456526" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="366" data-strategy="2" cx="595.7113456029698" cy="248.80888323117242" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="367" data-strategy="2" cx="532.9711851461844" cy="250.6249726274472" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="368" data-strategy="2" cx="546.0659450173163" cy="250.4914940812388" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="369" data-strategy="2" cx="519.8957795965899" cy="226.59946222903557" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="370" data-strategy="2" cx="539.978470778262" cy="200.2256318931888" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="371" data-strategy="2" cx="519.6547079787608" cy="197.8663451890171" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="372" data-strategy="2" cx="516.1428839385334" cy="193.97951482568715" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="373" data-strategy="2" cx="580.373914126269" cy="219.3742356282765" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="374" data-strategy="2" cx="533.9854908574064" cy="238.63160018291103" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="375" data-strategy="2" cx="546.8168043417836" cy="216.95535948369488" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="376" data-strategy="2" cx="530.5122328474456" cy="219.34828278092326" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="377" data-strategy="2" cx="594.4561533327953" cy="216.47577081807117" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="378" data-strategy="2" cx="595.6558894393444" cy="209.293735099615" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="379" data-strategy="2" cx="519.8638553043555" cy="180.5383882309863" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="380" data-strategy="2" cx="601.3935407696135" cy="191.299364140074" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="381" data-strategy="2" cx="527.591193726224" cy="169.29669596465544" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="382" data-strategy="2" cx="517.0245948986269" cy="205.66859960755394" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="383" data-strategy="2" cx="555.9372099376002" cy="234.0091913432323" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="384" data-strategy="2" cx="572.8728441158164" cy="235.10024057698513" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="385" data-strategy="2" cx="595.6390098285727" cy="225.9609911861981" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="386" data-strategy="2" cx="534.6445777684587" cy="210.13690164394953" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="387" data-strategy="2" cx="533.2061954113502" cy="225.17024716611192" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="388" data-strategy="2" cx="535.9490923924184" cy="222.98861423189553" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="389" data-strategy="2" cx="540.8894657785515" cy="225.99050129038594" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="390" data-strategy="2" cx="570.9169206801074" cy="256.2716278311529" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="391" data-strategy="2" cx="574.4654546286015" cy="267.0333158912185" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="392" data-strategy="2" cx="559.1401198801633" cy="300.2885509486537" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="393" data-strategy="2" cx="516.5702923177442" cy="299.2837517177629" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="394" data-strategy="2" cx="593.5301080223998" cy="310.1408770716615" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="406" data-strategy="2" cx="543.6236508895671" cy="250.99216987326818" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="411" data-strategy="2" cx="550.9928179019608" cy="280.423807333519" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="416" data-strategy="2" cx="583.0651955619596" cy="246.90899341452595" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="421" data-strategy="2" cx="562.8774848282338" cy="244.1887988831182" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="426" data-strategy="2" cx="519.9320827672555" cy="232.6693095917019" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="432" data-strategy="2" cx="567.0370210315523" cy="244.00064597425444" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="433" data-strategy="2" cx="544.8918860426629" cy="255.69000748769858" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="434" data-strategy="2" cx="530.6738787194095" cy="277.26694295925245" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="435" data-strategy="2" cx="580.5463551624516" cy="303.3843170517346" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="436" data-strategy="2" cx="537.7601225511386" cy="271.87402112722566" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="437" data-strategy="2" cx="585.0141686537041" cy="295.27787357699407" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="438" data-strategy="2" cx="568.9530271422207" cy="309.8237527355806" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="439" data-strategy="2" cx="580.6497926176155" cy="304.17625113145374" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="440" data-strategy="2" cx="573.4063797833986" cy="309.16288520893016" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="441" data-strategy="2" cx="540.0109361913362" cy="302.91907917077583" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="442" data-strategy="2" cx="549.1913807800887" cy="294.7877806962694" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="464" data-strategy="2" cx="580.1683536133983" cy="290.44108563913466" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="469" data-strategy="2" cx="555.1908740952788" cy="261.42444459670867" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="473" data-strategy="2" cx="529.7699146659314" cy="274.7098198201179" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="474" data-strategy="2" cx="559.7570433247051" cy="265.40406873739835" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="478" data-strategy="2" cx="539.5016570431776" cy="239.66710438256737" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="479" data-strategy="2" cx="540.3751319213434" cy="206.82731488973147" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="481" data-strategy="2" cx="570.6321842847768" cy="193.49112173809107" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="482" data-strategy="2" cx="565.1355392036779" cy="189.34596772692893" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="483" data-strategy="2" cx="545.2868426733132" cy="193.9330914816423" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="484" data-strategy="2" cx="599.5296801990356" cy="213.56407055373273" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="485" data-strategy="2" cx="589.3462649691799" cy="186.10617263483658" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="486" data-strategy="2" cx="569.027622690975" cy="217.2499131459535" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="487" data-strategy="2" cx="578.3717990088629" cy="221.79901429884524" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="488" data-strategy="2" cx="550.3104551584589" cy="215.65763640853132" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="489" data-strategy="2" cx="587.8734645937028" cy="249.56084613248493" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="490" data-strategy="2" cx="566.212357548295" cy="256.5265636275403" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="491" data-strategy="2" cx="551.2251637636778" cy="229.20487161402863" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="492" data-strategy="2" cx="580.4654818203792" cy="235.18323906146443" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="493" data-strategy="2" cx="582.8290757024675" cy="226.1334633597425" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="494" data-strategy="2" cx="584.2885557533286" cy="232.80535679790523" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="495" data-strategy="2" cx="565.2663503451438" cy="196.7569897926528" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="496" data-strategy="2" cx="567.5374662805225" cy="166.77552650023472" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="497" data-strategy="2" cx="576.5392380931931" cy="168.14360422100296" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="498" data-strategy="2" cx="555.7409283273943" cy="194.95815472696754" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="499" data-strategy="2" cx="560.7981304148686" cy="225.31521045593215" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="500" data-strategy="2" cx="580.0586473087485" cy="283.52302126159316" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="501" data-strategy="2" cx="518.9923304934497" cy="250.00033659531053" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="502" data-strategy="2" cx="557.929505127057" cy="262.7348503577905" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="503" data-strategy="2" cx="589.4824306280724" cy="314.08591237450963" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="526" data-strategy="2" cx="593.5452301188266" cy="123.71559005934338" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="534" data-strategy="2" cx="580.8321127488963" cy="192.54022876779862" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="574" data-strategy="2" cx="525.7197548911764" cy="176.41666524056774" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="575" data-strategy="2" cx="529.6593428390537" cy="201.0668106857564" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="576" data-strategy="2" cx="594.8079025806128" cy="240.62659456694146" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="577" data-strategy="2" cx="590.0359392761538" cy="215.19811305921476" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="578" data-strategy="2" cx="546.1900750939201" cy="298.54642668188694" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="579" data-strategy="2" cx="584.7642639148412" cy="313.3216076391722" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="580" data-strategy="2" cx="554.755738520288" cy="325.20139162473856" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="581" data-strategy="2" cx="588.7083260521556" cy="293.270795603321" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="582" data-strategy="2" cx="533.5316619193077" cy="279.4640019883892" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="583" data-strategy="2" cx="521.3243763038853" cy="309.8215247755286" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="584" data-strategy="2" cx="567.3025573906361" cy="294.91480520299876" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="585" data-strategy="2" cx="541.8765252452204" cy="346.7231032621415" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="586" data-strategy="2" cx="585.255447504761" cy="303.3329525429416" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="587" data-strategy="2" cx="583.0750640708858" cy="271.70546982129616" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="588" data-strategy="2" cx="595.5763279086425" cy="245.08724242053793" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="589" data-strategy="2" cx="549.8837435422525" cy="257.4254243627594" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="590" data-strategy="2" cx="551.1955443429817" cy="247.8136643284941" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="591" data-strategy="2" cx="569.7094405516989" cy="229.6760146958227" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="592" data-strategy="2" cx="548.1070412359039" cy="287.311442707181" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="593" data-strategy="2" cx="592.5540970993413" cy="324.2976475077229" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="594" data-strategy="2" cx="566.4376566625755" cy="311.79123098461224" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="595" data-strategy="2" cx="516.4137224449602" cy="319.8258772481469" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="596" data-strategy="2" cx="539.1505732408167" cy="358.40886420424533" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="597" data-strategy="2" cx="570.6735909299916" cy="382.4253565753085" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="598" data-strategy="2" cx="570.8733476492272" cy="362.8418705283941" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="599" data-strategy="2" cx="518.7065754705909" cy="368.30684819341894" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="600" data-strategy="2" cx="600.2090923619578" cy="418.6966313315435" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="601" data-strategy="2" cx="538.1543778397767" cy="408.02751083314746" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="602" data-strategy="2" cx="578.1140077413431" cy="365.5892130362137" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="603" data-strategy="2" cx="517.0610457137446" cy="364.44806870022205" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="604" data-strategy="2" cx="568.3664025062457" cy="380.27328418701103" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="605" data-strategy="2" cx="526.4057591569008" cy="404.8575091396104" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="606" data-strategy="2" cx="533.8194477604645" cy="401.29128952162876" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="607" data-strategy="2" cx="532.8651567438656" cy="380.3803862713625" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="608" data-strategy="2" cx="550.9761819597111" cy="346.03545925581903" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="609" data-strategy="2" cx="580.116812069346" cy="362.6431435361257" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="610" data-strategy="2" cx="601.4241179362845" cy="397.8531823890654" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="611" data-strategy="2" cx="519.4126642258155" cy="391.54478513960106" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="612" data-strategy="2" cx="573.3000374809237" cy="344.3734973950681" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="613" data-strategy="2" cx="529.4740764311229" cy="357.1908001467926" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="614" data-strategy="2" cx="531.5686656159288" cy="377.9993281080501" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="615" data-strategy="2" cx="518.8011404341387" cy="377.6917370666718" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="616" data-strategy="2" cx="557.107448195696" cy="339.73084469217736" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="617" data-strategy="2" cx="590.1528637325097" cy="331.62801773002167" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="618" data-strategy="2" cx="539.5211192819271" cy="343.35265058501267" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="619" data-strategy="2" cx="563.0947434257122" cy="352.0793342708657" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="620" data-strategy="2" cx="589.1219527447628" cy="354.75015193294587" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="621" data-strategy="2" cx="600.7114336204926" cy="314.41328644427097" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="622" data-strategy="2" cx="591.3196814097554" cy="283.79389411989325" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="623" data-strategy="2" cx="600.5538091848906" cy="247.312607904912" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="624" data-strategy="2" cx="579.2565628703607" cy="266.33549278564647" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="625" data-strategy="2" cx="560.9862648128989" cy="327.1103678649128" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="626" data-strategy="2" cx="524.2404742995823" cy="338.53985152157856" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="627" data-strategy="2" cx="582.279214345045" cy="299.50841956951837" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="628" data-strategy="2" cx="556.166996451457" cy="304.0245966400857" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="629" data-strategy="2" cx="563.3477639789528" cy="320.90200695652925" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="630" data-strategy="2" cx="562.3942688243158" cy="333.7949892818741" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="631" data-strategy="2" cx="541.8781885083263" cy="339.2438488880125" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="632" data-strategy="2" cx="559.6385105032334" cy="314.2754160425462" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="633" data-strategy="2" cx="534.3930012342836" cy="324.66003389153883" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="634" data-strategy="2" cx="589.8164454715862" cy="359.33035848392615" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="635" data-strategy="2" cx="530.3312760888375" cy="363.00134732936255" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="636" data-strategy="2" cx="600.4518697131653" cy="354.8002089379624" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="637" data-strategy="2" cx="595.4654955952969" cy="324.04458941975724" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="638" data-strategy="2" cx="584.9332688016996" cy="313.0396042976033" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="639" data-strategy="2" cx="525.9851525308954" cy="319.8821284376974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="640" data-strategy="2" cx="579.1226995710224" cy="322.30620860032263" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="641" data-strategy="2" cx="594.1932349027695" cy="324.80690593893246" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="642" data-strategy="2" cx="551.4562264815212" cy="316.43160126448436" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="643" data-strategy="2" cx="590.5452501090833" cy="329.81153132492096" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="644" data-strategy="2" cx="576.0745854461209" cy="344.81518142977876" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="645" data-strategy="2" cx="533.3196552207871" cy="333.68136593841865" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="646" data-strategy="2" cx="531.8062551079926" cy="321.8260948327057" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="647" data-strategy="2" cx="581.382812428011" cy="325.72674101650307" r="3" fill="#94a3b8" fill-opacity=".65"></circle><text x="558.25" y="572" text-anchor="middle" >B｜動能＋法人</text><circle class="point" data-window="1" data-strategy="3" cx="739.7901789215447" cy="326.74840773420567" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="2" data-strategy="3" cx="773.9553037140992" cy="330.78609730594076" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="3" data-strategy="3" cx="775.8640804144613" cy="308.55533445815433" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="4" data-strategy="3" cx="759.8091784000331" cy="283.25274559053935" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="5" data-strategy="3" cx="705.4140684125724" cy="266.97996477247045" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="6" data-strategy="3" cx="729.8475739886823" cy="261.9034421708826" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="7" data-strategy="3" cx="782.8699618112687" cy="285.3309894040907" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="8" data-strategy="3" cx="755.4041267720451" cy="278.64223274006514" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="9" data-strategy="3" cx="756.4591617614408" cy="264.3270049179541" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="10" data-strategy="3" cx="725.6931220095862" cy="251.1229980875078" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="11" data-strategy="3" cx="757.2895305600069" cy="227.13769621801177" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="12" data-strategy="3" cx="789.3827114176479" cy="249.09496861745163" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="13" data-strategy="3" cx="741.7147190433325" cy="247.4977657551382" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="14" data-strategy="3" cx="747.6649965423632" cy="224.54853464855418" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="15" data-strategy="3" cx="776.1729216894214" cy="216.98355334324324" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="16" data-strategy="3" cx="760.4212269013503" cy="211.0694361829901" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="17" data-strategy="3" cx="725.546229979118" cy="226.6892285307418" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="39" data-strategy="3" cx="705.8242032772122" cy="301.7187904726023" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="44" data-strategy="3" cx="784.4610296526816" cy="305.90466684609106" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="49" data-strategy="3" cx="712.1821945401466" cy="361.07179351805127" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="54" data-strategy="3" cx="713.3026284832906" cy="340.9501029476747" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="59" data-strategy="3" cx="758.0754620645805" cy="333.342523399663" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="67" data-strategy="3" cx="741.7929977984177" cy="339.71942762118226" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="70" data-strategy="3" cx="773.6310028500573" cy="304.645888726235" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="72" data-strategy="3" cx="764.3524372059993" cy="289.890318990883" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="75" data-strategy="3" cx="771.7683735897747" cy="271.5819527755991" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="77" data-strategy="3" cx="721.4097450222148" cy="314.3261157558992" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="80" data-strategy="3" cx="743.7485641515234" cy="293.16895509649675" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="82" data-strategy="3" cx="791.9999402296238" cy="294.0155151916513" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="83" data-strategy="3" cx="721.1767498549568" cy="286.80089824109353" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="85" data-strategy="3" cx="791.1409646203704" cy="289.51387459879163" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="87" data-strategy="3" cx="761.3141146470166" cy="283.50505727131224" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="88" data-strategy="3" cx="715.4176817197867" cy="277.43919036285627" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="90" data-strategy="3" cx="752.3897879195008" cy="285.0606926325015" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="93" data-strategy="3" cx="771.7094792674021" cy="287.71731711901066" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="96" data-strategy="3" cx="720.7970600367274" cy="293.34859763316683" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="98" data-strategy="3" cx="735.8246315876605" cy="312.6647400309198" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="101" data-strategy="3" cx="778.3936941000389" cy="267.1602357228712" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="103" data-strategy="3" cx="735.9845303940144" cy="288.6907083696013" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="106" data-strategy="3" cx="768.9171377137671" cy="280.4409693971142" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="111" data-strategy="3" cx="712.8885486291942" cy="319.8578219197074" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="114" data-strategy="3" cx="791.9338744088858" cy="352.0094294162466" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="115" data-strategy="3" cx="769.812472007062" cy="342.75277154927346" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="116" data-strategy="3" cx="757.7906306100298" cy="336.18638412786055" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="117" data-strategy="3" cx="751.1889176455218" cy="345.351235631588" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="118" data-strategy="3" cx="731.7702491160227" cy="339.08839674000006" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="119" data-strategy="3" cx="767.1265758158796" cy="356.3250464782884" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="120" data-strategy="3" cx="754.4350327412964" cy="383.2893858808264" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="121" data-strategy="3" cx="714.5485347560782" cy="385.3844091624211" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="122" data-strategy="3" cx="780.4939489221126" cy="378.61796450915153" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="123" data-strategy="3" cx="764.2365707813454" cy="388.96835697338804" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="124" data-strategy="3" cx="707.9323639712919" cy="385.30575787736467" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="125" data-strategy="3" cx="712.8064047715169" cy="378.38371260778706" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="126" data-strategy="3" cx="729.7707538629335" cy="379.1418678053195" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="127" data-strategy="3" cx="735.4820094944105" cy="383.62365570367854" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="128" data-strategy="3" cx="783.8647745742468" cy="391.3095375495602" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="129" data-strategy="3" cx="761.4136462458326" cy="384.2862699718241" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="130" data-strategy="3" cx="782.5000304173828" cy="376.00623376022116" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="131" data-strategy="3" cx="705.4709237031298" cy="373.85321758432855" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="132" data-strategy="3" cx="751.2953777171406" cy="375.21554005522836" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="133" data-strategy="3" cx="778.3497911649621" cy="363.0188929123998" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="134" data-strategy="3" cx="777.1800612901525" cy="362.7308983285744" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="135" data-strategy="3" cx="775.442697195504" cy="343.7452064011208" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="136" data-strategy="3" cx="739.8019802458365" cy="341.7068830258078" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="137" data-strategy="3" cx="765.7833241376558" cy="323.8259926319987" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="138" data-strategy="3" cx="791.6716877268677" cy="329.15649491827753" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="139" data-strategy="3" cx="759.6611910717126" cy="327.9459288153327" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="140" data-strategy="3" cx="733.4657457507368" cy="341.6955692733687" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="141" data-strategy="3" cx="707.2096253674448" cy="345.5632396443061" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="142" data-strategy="3" cx="729.4776194764884" cy="343.0682257828376" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="143" data-strategy="3" cx="741.4383463126363" cy="317.70595271390476" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="144" data-strategy="3" cx="718.6374166444876" cy="319.4802449927333" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="145" data-strategy="3" cx="731.0490988300529" cy="338.28065004377396" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="146" data-strategy="3" cx="778.5747959934366" cy="331.75539342099023" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="147" data-strategy="3" cx="714.592929471957" cy="337.795680287047" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="148" data-strategy="3" cx="731.1732122472604" cy="331.41951641414005" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="149" data-strategy="3" cx="778.2263416744247" cy="333.1546281028899" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="150" data-strategy="3" cx="709.0507872322235" cy="333.869737776408" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="151" data-strategy="3" cx="775.2005556301378" cy="328.4520504836427" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="152" data-strategy="3" cx="734.5571148164771" cy="323.2178989403551" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="153" data-strategy="3" cx="779.3587445778892" cy="326.0787964918493" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="154" data-strategy="3" cx="749.5834332126894" cy="332.69542340353416" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="155" data-strategy="3" cx="769.9336049994397" cy="331.28440415491974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="156" data-strategy="3" cx="752.2966473781205" cy="317.98201919811754" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="157" data-strategy="3" cx="710.8720096320912" cy="320.2765793618824" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="158" data-strategy="3" cx="733.4116823481893" cy="335.48859922789796" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="159" data-strategy="3" cx="713.0185747241283" cy="341.9339973802962" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="160" data-strategy="3" cx="781.6441261121323" cy="341.68069593186317" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="161" data-strategy="3" cx="790.4527857430608" cy="329.38116271459506" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="162" data-strategy="3" cx="738.9348488958426" cy="322.4754101769983" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="163" data-strategy="3" cx="744.9669722590421" cy="326.62455863984144" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="164" data-strategy="3" cx="757.9780041998667" cy="328.8486292920175" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="165" data-strategy="3" cx="717.2070806618905" cy="307.2593653530454" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="166" data-strategy="3" cx="717.0959376280558" cy="302.0333644175118" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="167" data-strategy="3" cx="770.8822532227668" cy="302.6659746591918" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="169" data-strategy="3" cx="792.0140091520824" cy="303.8095177725823" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="174" data-strategy="3" cx="781.5641066705502" cy="327.6214186087968" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="185" data-strategy="3" cx="769.9482392892014" cy="306.82838904492843" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="190" data-strategy="3" cx="771.44698273208" cy="329.5187690455005" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="195" data-strategy="3" cx="726.1077008403595" cy="307.37219211535825" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="200" data-strategy="3" cx="726.3425810951063" cy="274.63796705688105" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="202" data-strategy="3" cx="737.8571334652245" cy="268.60370274386605" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="207" data-strategy="3" cx="740.4503792745054" cy="310.40061414070334" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="212" data-strategy="3" cx="732.1751542692648" cy="303.3084034583326" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="217" data-strategy="3" cx="767.5504784580222" cy="315.0154816512028" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="222" data-strategy="3" cx="751.6197867470271" cy="316.47787954772497" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="260" data-strategy="3" cx="725.319652755343" cy="343.15335027881974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="302" data-strategy="3" cx="748.6170856173517" cy="293.9236910330671" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="303" data-strategy="3" cx="741.4118454369801" cy="278.02028184799894" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="350" data-strategy="3" cx="758.5980117389427" cy="277.9279804319237" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="351" data-strategy="3" cx="716.5002768778218" cy="268.56222242925384" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="352" data-strategy="3" cx="732.576307842874" cy="289.1510493749553" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="353" data-strategy="3" cx="784.2437283908793" cy="307.8882631172413" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="354" data-strategy="3" cx="744.973553654643" cy="286.3341510638082" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="355" data-strategy="3" cx="757.752287067237" cy="292.77291443555026" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="356" data-strategy="3" cx="761.4186190690585" cy="300.1396419220861" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="357" data-strategy="3" cx="722.8747749337057" cy="301.9097584694323" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="358" data-strategy="3" cx="749.6452578874405" cy="310.5738000699199" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="359" data-strategy="3" cx="713.9414753694167" cy="289.61090979792533" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="360" data-strategy="3" cx="716.2405602890578" cy="294.6030379428081" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="361" data-strategy="3" cx="769.4272942252848" cy="287.1029773098802" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="362" data-strategy="3" cx="732.3240362555309" cy="295.8859101235922" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="363" data-strategy="3" cx="712.5509001076514" cy="307.22361334214617" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="364" data-strategy="3" cx="765.5067914816026" cy="292.66526989805084" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="365" data-strategy="3" cx="708.9224972266713" cy="286.3782915530129" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="366" data-strategy="3" cx="773.4624206151384" cy="266.3990863052095" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="367" data-strategy="3" cx="744.9122961832095" cy="267.4050425988132" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="368" data-strategy="3" cx="710.7681561255076" cy="271.5819758444008" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="369" data-strategy="3" cx="785.8037736545992" cy="233.85514447570662" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="370" data-strategy="3" cx="732.9242391109016" cy="218.0747656207288" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="371" data-strategy="3" cx="761.351872733139" cy="218.80181253956272" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="372" data-strategy="3" cx="716.9943033017826" cy="212.98582899968443" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="373" data-strategy="3" cx="719.3379613600765" cy="233.3491289975702" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="374" data-strategy="3" cx="742.383178513629" cy="259.63977396556254" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="375" data-strategy="3" cx="764.9384256767482" cy="231.15730642896315" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="376" data-strategy="3" cx="752.0803815412747" cy="244.26035610366884" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="377" data-strategy="3" cx="753.4011786946218" cy="249.94172796140248" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="378" data-strategy="3" cx="790.4405634918137" cy="252.48671903141226" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="379" data-strategy="3" cx="791.0294671744963" cy="232.8583516322696" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="380" data-strategy="3" cx="775.3970921136898" cy="222.9368863553162" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="381" data-strategy="3" cx="768.0220556368482" cy="215.10229274927048" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="382" data-strategy="3" cx="733.9418522170437" cy="247.6757982582672" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="383" data-strategy="3" cx="769.446486374373" cy="280.62942771314647" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="384" data-strategy="3" cx="787.8897080771907" cy="286.4668835054304" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="385" data-strategy="3" cx="783.4198075672957" cy="253.13771487626184" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="386" data-strategy="3" cx="715.4967799603023" cy="252.00259534844028" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="387" data-strategy="3" cx="776.6706148717558" cy="246.78926372367664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="388" data-strategy="3" cx="721.8286858988719" cy="269.55183323368146" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="389" data-strategy="3" cx="785.8490384149735" cy="273.6163050085278" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="390" data-strategy="3" cx="739.9402409489965" cy="274.2474011520112" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="391" data-strategy="3" cx="749.0747549829985" cy="290.75673502635004" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="392" data-strategy="3" cx="706.1007650253672" cy="310.63545893711876" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="393" data-strategy="3" cx="783.7497378554511" cy="309.03687987299065" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="394" data-strategy="3" cx="755.1970280825551" cy="328.45562598195244" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="406" data-strategy="3" cx="707.834394890034" cy="240.77608123719395" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="411" data-strategy="3" cx="734.3626784987654" cy="271.45169366566347" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="416" data-strategy="3" cx="708.23233128504" cy="245.26654055569438" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="421" data-strategy="3" cx="790.4908014309478" cy="247.80917333958996" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="426" data-strategy="3" cx="727.6134427409968" cy="237.60868642579064" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="432" data-strategy="3" cx="766.8438975726418" cy="251.72339364092946" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="433" data-strategy="3" cx="751.8106030175115" cy="266.0346985474387" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="434" data-strategy="3" cx="743.4278093098964" cy="276.34600279554627" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="435" data-strategy="3" cx="752.8840067150755" cy="294.5114590134683" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="436" data-strategy="3" cx="707.9369979331364" cy="279.3723686736075" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="437" data-strategy="3" cx="783.0962828529364" cy="302.7750502130546" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="438" data-strategy="3" cx="728.4392485070657" cy="316.9313949745153" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="439" data-strategy="3" cx="744.426682428007" cy="305.0615643072756" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="440" data-strategy="3" cx="730.743532693073" cy="302.35688210293927" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="441" data-strategy="3" cx="760.7752177798783" cy="313.20624773152775" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="442" data-strategy="3" cx="750.5871139208932" cy="301.2350845797058" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="464" data-strategy="3" cx="739.7529319883389" cy="297.00132951662147" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="469" data-strategy="3" cx="717.0461099270104" cy="252.16275856403348" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="473" data-strategy="3" cx="710.7629964907493" cy="259.4871756685144" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="474" data-strategy="3" cx="775.0410093979302" cy="243.92090048393865" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="478" data-strategy="3" cx="732.0085790828983" cy="232.00520179751547" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="479" data-strategy="3" cx="783.1829919290394" cy="197.19796205616976" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="481" data-strategy="3" cx="760.2821346071764" cy="216.64633981746564" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="482" data-strategy="3" cx="736.9893821607744" cy="208.5708264124664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="483" data-strategy="3" cx="764.7492834589292" cy="194.07490327785965" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="484" data-strategy="3" cx="772.4685424379282" cy="201.45385472282956" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="485" data-strategy="3" cx="762.9597251506538" cy="208.27241700770676" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="486" data-strategy="3" cx="745.0561595200833" cy="233.12647584307368" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="487" data-strategy="3" cx="729.9171079536163" cy="223.44320800371403" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="488" data-strategy="3" cx="734.915598429557" cy="215.37470912816042" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="489" data-strategy="3" cx="748.7930919782017" cy="232.23302012573214" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="490" data-strategy="3" cx="733.9624381030841" cy="266.3931198931347" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="491" data-strategy="3" cx="764.4689544012457" cy="239.89570395551218" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="492" data-strategy="3" cx="765.7976412182257" cy="242.30467344451245" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="493" data-strategy="3" cx="732.0744069319244" cy="235.62771115418877" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="494" data-strategy="3" cx="757.0942099794635" cy="218.2089597822022" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="495" data-strategy="3" cx="728.1849573042809" cy="215.88707160684842" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="496" data-strategy="3" cx="725.0178168525651" cy="194.20481006560766" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="497" data-strategy="3" cx="763.3968941614802" cy="188.52572319159182" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="498" data-strategy="3" cx="780.4223296719947" cy="202.02285716062758" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="499" data-strategy="3" cx="713.8244137037568" cy="221.47937818305724" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="500" data-strategy="3" cx="753.0752660805449" cy="284.56986320025896" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="501" data-strategy="3" cx="742.5865736497573" cy="254.29023545867437" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="502" data-strategy="3" cx="722.3860416821265" cy="272.03698564798015" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="503" data-strategy="3" cx="773.5042589606459" cy="309.7075146487288" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="526" data-strategy="3" cx="784.2397098289276" cy="121.57361941905073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="534" data-strategy="3" cx="722.198179564858" cy="203.66101219394585" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="574" data-strategy="3" cx="754.1225067204817" cy="188.63673421570266" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="575" data-strategy="3" cx="778.837384439166" cy="217.21506099039556" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="576" data-strategy="3" cx="771.4974262214275" cy="250.7614990733567" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="577" data-strategy="3" cx="777.4915556846983" cy="252.43641018197775" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="578" data-strategy="3" cx="760.7542816355532" cy="320.6260885784269" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="579" data-strategy="3" cx="733.4719203402113" cy="319.8789100815567" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="580" data-strategy="3" cx="718.9145171785333" cy="332.2629119840479" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="581" data-strategy="3" cx="762.4515247452356" cy="324.2300225235982" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="582" data-strategy="3" cx="731.0285937718954" cy="322.74440501680147" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="583" data-strategy="3" cx="766.6956408588986" cy="336.7847984789792" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="584" data-strategy="3" cx="714.3648881322702" cy="312.0070707878349" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="585" data-strategy="3" cx="736.8986112776657" cy="352.0499973524642" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="586" data-strategy="3" cx="721.8562395914953" cy="330.22715110457165" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="587" data-strategy="3" cx="784.5046099903523" cy="298.31058963016113" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="588" data-strategy="3" cx="727.4728423587152" cy="272.71199776950476" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="589" data-strategy="3" cx="754.5594658730007" cy="279.38473046035307" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="590" data-strategy="3" cx="727.1204500579798" cy="262.1867197891853" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="591" data-strategy="3" cx="718.3510170702137" cy="264.09894638477044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="592" data-strategy="3" cx="774.9822741660104" cy="313.849644230602" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="593" data-strategy="3" cx="717.1444782424505" cy="346.3797829538584" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="594" data-strategy="3" cx="716.8168561838164" cy="327.0446911512316" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="595" data-strategy="3" cx="761.1954759924192" cy="320.39780587145276" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="596" data-strategy="3" cx="728.7140900669685" cy="367.3003434126793" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="597" data-strategy="3" cx="729.0909681986533" cy="390.32201210628324" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="598" data-strategy="3" cx="723.654518670791" cy="368.76836493813875" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="599" data-strategy="3" cx="781.289508848758" cy="369.56002887251236" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="600" data-strategy="3" cx="742.5636912794905" cy="393.5799899981197" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="601" data-strategy="3" cx="705.5738823016503" cy="399.37199807650217" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="602" data-strategy="3" cx="722.8458068094134" cy="362.53728954356046" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="603" data-strategy="3" cx="720.0449697957854" cy="361.9709874609552" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="604" data-strategy="3" cx="778.8304655606676" cy="385.82387266328806" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="605" data-strategy="3" cx="754.2667541454133" cy="375.45339312366673" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="606" data-strategy="3" cx="784.2799970673099" cy="387.2427959834456" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="607" data-strategy="3" cx="756.6110435116983" cy="362.23354222529633" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="608" data-strategy="3" cx="732.7566659039685" cy="348.6933359117471" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="609" data-strategy="3" cx="740.2625420170468" cy="373.2929318911167" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="610" data-strategy="3" cx="790.5717276614428" cy="367.5172292911953" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="611" data-strategy="3" cx="739.4003187275559" cy="380.471932097649" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="612" data-strategy="3" cx="707.9701338353336" cy="332.8737417526346" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="613" data-strategy="3" cx="788.5631766971222" cy="364.2518168210729" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="614" data-strategy="3" cx="710.9818323433835" cy="383.0664747140585" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="615" data-strategy="3" cx="715.0263387636688" cy="354.5217718921401" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="616" data-strategy="3" cx="780.6338903012107" cy="333.8034908934366" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="617" data-strategy="3" cx="768.7702060876934" cy="321.3599951134716" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="618" data-strategy="3" cx="725.3787562788971" cy="347.8550295882423" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="619" data-strategy="3" cx="778.9028414840359" cy="354.7122484229653" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="620" data-strategy="3" cx="788.7011702729769" cy="344.27233100880153" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="621" data-strategy="3" cx="760.8752255536418" cy="320.43802216901406" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="622" data-strategy="3" cx="761.3890843179879" cy="295.75066392834617" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="623" data-strategy="3" cx="718.3893807480563" cy="269.94275878589787" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="624" data-strategy="3" cx="758.9969783397551" cy="292.55380191516923" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="625" data-strategy="3" cx="719.2433529540082" cy="340.588036946822" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="626" data-strategy="3" cx="717.8132395704123" cy="339.01075933736905" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="627" data-strategy="3" cx="777.8251159000506" cy="324.5886730248141" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="628" data-strategy="3" cx="705.4787355502904" cy="303.12873870080256" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="629" data-strategy="3" cx="759.5509584286548" cy="327.23920845648684" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="630" data-strategy="3" cx="728.3234016128107" cy="341.71503009132556" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="631" data-strategy="3" cx="753.1217121659749" cy="330.91307468999275" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="632" data-strategy="3" cx="767.1603324417821" cy="334.33938836686025" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="633" data-strategy="3" cx="771.9432568847419" cy="315.19002366506413" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="634" data-strategy="3" cx="784.6908329027581" cy="354.6585946012973" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="635" data-strategy="3" cx="735.0507209578906" cy="366.1560181347495" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="636" data-strategy="3" cx="717.535834579783" cy="343.7119054731766" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="637" data-strategy="3" cx="783.9695719840092" cy="340.7247528100884" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="638" data-strategy="3" cx="754.6282718266652" cy="300.0149004847256" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="639" data-strategy="3" cx="769.3377134155174" cy="315.5858351844123" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="640" data-strategy="3" cx="764.9867362194916" cy="336.41856858058145" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="641" data-strategy="3" cx="748.8016099382093" cy="328.34498796006767" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="642" data-strategy="3" cx="780.0652043200117" cy="335.8416276872517" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="643" data-strategy="3" cx="710.9863514025855" cy="316.6815209865136" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="644" data-strategy="3" cx="707.7491015055292" cy="340.644548726843" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="645" data-strategy="3" cx="737.6351869030481" cy="339.5950500485429" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="646" data-strategy="3" cx="709.0932048726579" cy="333.1001239472022" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="647" data-strategy="3" cx="714.1530240033954" cy="347.56510877995703" r="3" fill="#94a3b8" fill-opacity=".65"></circle><text x="748.75" y="572" text-anchor="middle" >C｜動能＋低波動</text><circle class="point" data-window="1" data-strategy="4" cx="977.8400513526237" cy="343.0872672263618" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="2" data-strategy="4" cx="958.1919851014601" cy="330.9238285468134" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="3" data-strategy="4" cx="916.2219470992078" cy="325.23411146427037" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="4" data-strategy="4" cx="975.1021384549979" cy="309.8512888194477" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="5" data-strategy="4" cx="908.6823063818613" cy="318.085589239779" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="6" data-strategy="4" cx="935.9447798799627" cy="314.0404004134927" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="7" data-strategy="4" cx="980.7080563733948" cy="327.22178490633974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="8" data-strategy="4" cx="902.5016163775014" cy="282.5471443259628" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="9" data-strategy="4" cx="940.055517642515" cy="266.5757824394327" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="10" data-strategy="4" cx="949.3702085694863" cy="254.651346964412" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="11" data-strategy="4" cx="908.4550606413992" cy="263.46278665901684" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="12" data-strategy="4" cx="908.9412225262005" cy="271.62422873888937" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="13" data-strategy="4" cx="964.6139628025032" cy="255.64881778800478" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="14" data-strategy="4" cx="956.0150101699304" cy="232.7713221611973" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="15" data-strategy="4" cx="940.6487316957055" cy="217.60745150962296" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="16" data-strategy="4" cx="898.7681093239275" cy="224.11969340527486" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="17" data-strategy="4" cx="941.1672670831889" cy="220.17772096140453" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="39" data-strategy="4" cx="915.9390025568458" cy="305.65983278364695" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="44" data-strategy="4" cx="956.9755849619418" cy="302.1713463349818" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="49" data-strategy="4" cx="912.4333771899375" cy="352.8342570126849" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="54" data-strategy="4" cx="910.0676101107167" cy="326.7964736539013" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="59" data-strategy="4" cx="958.6645508294233" cy="331.21405516704664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="67" data-strategy="4" cx="934.5429931482985" cy="343.8621649385897" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="70" data-strategy="4" cx="905.2778174017523" cy="326.52228059487453" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="72" data-strategy="4" cx="953.625415831966" cy="260.3258318019578" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="75" data-strategy="4" cx="939.3868813777049" cy="290.22106136909093" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="77" data-strategy="4" cx="932.6362601354107" cy="305.69133159139255" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="80" data-strategy="4" cx="936.7410583214522" cy="307.977351171098" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="82" data-strategy="4" cx="974.7912818226094" cy="299.0329486322378" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="83" data-strategy="4" cx="928.9461386223779" cy="297.4659068945254" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="85" data-strategy="4" cx="953.4812531056049" cy="259.5683122133704" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="87" data-strategy="4" cx="979.8275332572475" cy="278.26604111164465" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="88" data-strategy="4" cx="968.5101772674544" cy="271.6062564230945" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="90" data-strategy="4" cx="926.7543892884647" cy="266.65408687655827" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="93" data-strategy="4" cx="955.0301784051863" cy="282.0924376367444" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="96" data-strategy="4" cx="944.5490982954936" cy="296.5379538884409" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="98" data-strategy="4" cx="953.7457457973871" cy="305.10068097879315" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="101" data-strategy="4" cx="912.7045013724041" cy="288.2551603617748" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="103" data-strategy="4" cx="949.6298215079162" cy="305.5511528180609" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="106" data-strategy="4" cx="947.2034332168957" cy="302.0448389980568" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="111" data-strategy="4" cx="939.9149519940239" cy="331.1212080693849" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="114" data-strategy="4" cx="959.2573379119117" cy="355.2721467001464" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="115" data-strategy="4" cx="961.5779153161643" cy="362.29834058505367" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="116" data-strategy="4" cx="975.9903592290115" cy="346.25419673173377" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="117" data-strategy="4" cx="971.272456353304" cy="349.72995698233314" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="118" data-strategy="4" cx="908.9127495435705" cy="333.71722068913823" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="119" data-strategy="4" cx="942.2980642404075" cy="367.17971360458824" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="120" data-strategy="4" cx="959.9602071057325" cy="409.0600610022471" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="121" data-strategy="4" cx="912.6038397717211" cy="399.4136617322882" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="122" data-strategy="4" cx="934.4545385944334" cy="371.35019628614356" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="123" data-strategy="4" cx="957.064324595542" cy="385.59488310357773" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="124" data-strategy="4" cx="900.9708973373032" cy="379.01492292271905" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="125" data-strategy="4" cx="943.2729293690361" cy="379.2615424513466" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="126" data-strategy="4" cx="936.3267562031474" cy="373.10872825368466" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="127" data-strategy="4" cx="919.1565411090369" cy="374.4757064982336" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="128" data-strategy="4" cx="939.0904872564589" cy="387.84309158746146" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="129" data-strategy="4" cx="958.196493477165" cy="366.72416299511997" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="130" data-strategy="4" cx="897.043687834757" cy="359.7673182499924" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="131" data-strategy="4" cx="938.7998804321078" cy="339.96065096284696" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="132" data-strategy="4" cx="939.8496702081029" cy="338.9606156192965" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="133" data-strategy="4" cx="919.2776688240059" cy="330.280576243304" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="134" data-strategy="4" cx="926.8329956251094" cy="325.9593689856351" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="135" data-strategy="4" cx="981.1667771698199" cy="290.49250462639117" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="136" data-strategy="4" cx="926.0868936030445" cy="285.8739273793261" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="137" data-strategy="4" cx="933.4285570784228" cy="291.36882309014834" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="138" data-strategy="4" cx="916.9604594387797" cy="278.71797672484894" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="139" data-strategy="4" cx="975.1166052266279" cy="298.10552127198304" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="140" data-strategy="4" cx="974.8603738962635" cy="290.81573769666284" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="141" data-strategy="4" cx="936.8839036737111" cy="321.3417523145169" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="142" data-strategy="4" cx="907.9825792235906" cy="311.5247014994781" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="143" data-strategy="4" cx="934.0117388969716" cy="281.3908316218768" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="144" data-strategy="4" cx="919.3578197013117" cy="282.48847143292954" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="145" data-strategy="4" cx="937.4080376242234" cy="324.5033292287899" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="146" data-strategy="4" cx="978.2716694680614" cy="321.4718596593852" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="147" data-strategy="4" cx="896.0239414758122" cy="334.893291270793" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="148" data-strategy="4" cx="964.8484567192331" cy="321.1066011835029" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="149" data-strategy="4" cx="946.0444721036002" cy="309.98509769494933" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="150" data-strategy="4" cx="937.596492988697" cy="296.8572731720332" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="151" data-strategy="4" cx="958.04869599013" cy="310.6517313812875" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="152" data-strategy="4" cx="968.8669854529501" cy="305.8414819881014" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="153" data-strategy="4" cx="967.604127964329" cy="302.96445740894904" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="154" data-strategy="4" cx="913.4607477390816" cy="314.4830508967778" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="155" data-strategy="4" cx="979.7252289093467" cy="315.8606843609769" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="156" data-strategy="4" cx="907.4397676689462" cy="313.0252902709387" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="157" data-strategy="4" cx="908.062370221299" cy="317.05682672002393" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="158" data-strategy="4" cx="907.3054019559626" cy="328.610218820779" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="159" data-strategy="4" cx="921.4780670719783" cy="336.29804839040963" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="160" data-strategy="4" cx="939.5825467069254" cy="318.9494959968848" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="161" data-strategy="4" cx="934.4871491997183" cy="297.03068769205856" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="162" data-strategy="4" cx="970.4377233839524" cy="303.5705143552729" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="163" data-strategy="4" cx="979.1504960223582" cy="294.6813312302374" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="164" data-strategy="4" cx="970.6486704564808" cy="294.78975188132176" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="165" data-strategy="4" cx="955.3132843650608" cy="265.1068551013923" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="166" data-strategy="4" cx="926.9755659935569" cy="257.60017122334693" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="167" data-strategy="4" cx="922.0301822435279" cy="264.8903825143734" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="169" data-strategy="4" cx="971.2374338466514" cy="275.813620988166" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="174" data-strategy="4" cx="958.0932150935755" cy="310.39985543299576" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="185" data-strategy="4" cx="933.684360813181" cy="313.563461024637" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="190" data-strategy="4" cx="968.8625106075121" cy="353.435044913801" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="195" data-strategy="4" cx="965.7461226411991" cy="309.6527736478556" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="200" data-strategy="4" cx="908.1025693189353" cy="297.3646007052797" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="202" data-strategy="4" cx="912.5677619971342" cy="281.99432131792844" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="207" data-strategy="4" cx="944.3235568142792" cy="332.7637753532387" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="212" data-strategy="4" cx="957.6204751904759" cy="308.93787498339174" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="217" data-strategy="4" cx="911.5085747452125" cy="301.66755859805073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="222" data-strategy="4" cx="918.376229523514" cy="298.6927834639206" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="260" data-strategy="4" cx="905.3278626876123" cy="326.06638426261856" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="302" data-strategy="4" cx="946.8253202542721" cy="300.2371413313963" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="303" data-strategy="4" cx="958.3598254386574" cy="268.61209378616775" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="350" data-strategy="4" cx="915.6559027667737" cy="274.75133368077695" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="351" data-strategy="4" cx="945.9435305228343" cy="281.3536603307431" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="352" data-strategy="4" cx="979.990337866114" cy="309.2395277566892" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="353" data-strategy="4" cx="897.3019371062496" cy="316.4676053206311" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="354" data-strategy="4" cx="930.4428891375783" cy="299.885688814356" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="355" data-strategy="4" cx="901.0796642308565" cy="296.0653096115528" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="356" data-strategy="4" cx="940.4179223951984" cy="311.77597301651974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="357" data-strategy="4" cx="978.1897722935859" cy="311.39174513233115" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="358" data-strategy="4" cx="921.4590055773069" cy="315.1129525454974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="359" data-strategy="4" cx="976.2521408810026" cy="315.5697469172806" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="360" data-strategy="4" cx="939.5617361825741" cy="307.5521208861654" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="361" data-strategy="4" cx="978.7434264165515" cy="301.9437006010281" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="362" data-strategy="4" cx="916.0315403298912" cy="289.9936824220259" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="363" data-strategy="4" cx="900.0525061179869" cy="297.53937228208093" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="364" data-strategy="4" cx="906.4446298107805" cy="298.4150184474043" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="365" data-strategy="4" cx="970.5093455621892" cy="279.4872365911782" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="366" data-strategy="4" cx="908.6344909809243" cy="254.73125711822118" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="367" data-strategy="4" cx="914.6845862775026" cy="261.56342655441654" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="368" data-strategy="4" cx="950.3814257390607" cy="269.1316693394299" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="369" data-strategy="4" cx="961.8221999225934" cy="241.6015667372679" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="370" data-strategy="4" cx="952.788272870188" cy="212.8912128059742" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="371" data-strategy="4" cx="898.5029174490387" cy="204.46841867515062" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="372" data-strategy="4" cx="973.5256611962647" cy="199.78584806749308" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="373" data-strategy="4" cx="943.3714100238557" cy="193.56244639622003" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="374" data-strategy="4" cx="919.3100576815036" cy="230.659643617509" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="375" data-strategy="4" cx="912.8186430354032" cy="217.25339833367977" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="376" data-strategy="4" cx="948.6139991160951" cy="235.71017832967283" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="377" data-strategy="4" cx="940.5871579635437" cy="190.24735750150782" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="378" data-strategy="4" cx="922.830995686232" cy="171.13679996240126" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="379" data-strategy="4" cx="896.8520832553793" cy="147.55616015121524" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="380" data-strategy="4" cx="944.8741714983943" cy="170.60452818887995" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="381" data-strategy="4" cx="955.6372480712953" cy="162.65978162296665" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="382" data-strategy="4" cx="974.3418426976486" cy="174.79735622362978" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="383" data-strategy="4" cx="976.5245219221331" cy="210.2597612567384" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="384" data-strategy="4" cx="974.4412479559321" cy="238.65228603274667" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="385" data-strategy="4" cx="959.8495656412402" cy="227.42959406203894" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="386" data-strategy="4" cx="906.8839824666898" cy="219.621804446176" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="387" data-strategy="4" cx="932.2830237970566" cy="236.10088210293293" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="388" data-strategy="4" cx="905.1808075811532" cy="232.99576008252353" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="389" data-strategy="4" cx="946.7938230357885" cy="221.44534706203112" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="390" data-strategy="4" cx="907.6349607003217" cy="231.8895895306978" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="391" data-strategy="4" cx="945.4473876598628" cy="268.44914165195786" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="392" data-strategy="4" cx="908.8050896542096" cy="297.84423196637005" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="393" data-strategy="4" cx="942.9579206464751" cy="295.0166117935707" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="394" data-strategy="4" cx="978.5769138882523" cy="311.4968290724699" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="406" data-strategy="4" cx="960.0079421503731" cy="283.9038526506772" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="411" data-strategy="4" cx="975.3278883921523" cy="308.1460725797913" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="416" data-strategy="4" cx="969.1095977153891" cy="250.93626805703514" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="421" data-strategy="4" cx="959.4254504585016" cy="249.89350617991815" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="426" data-strategy="4" cx="913.4475746739407" cy="252.2278836620671" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="432" data-strategy="4" cx="966.3267452369405" cy="265.6717280027977" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="433" data-strategy="4" cx="946.6237747123241" cy="254.2653214813689" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="434" data-strategy="4" cx="914.4614575148937" cy="257.71709061135584" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="435" data-strategy="4" cx="925.1767008723123" cy="296.72539276438863" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="436" data-strategy="4" cx="935.3298382976874" cy="266.2228586748407" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="437" data-strategy="4" cx="976.2479682793819" cy="279.6351421073655" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="438" data-strategy="4" cx="929.8600394845009" cy="297.81462172103795" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="439" data-strategy="4" cx="954.2617134515356" cy="279.60252754925983" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="440" data-strategy="4" cx="981.0485702634213" cy="276.9660003846569" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="441" data-strategy="4" cx="909.9187498617931" cy="304.1363243717362" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="442" data-strategy="4" cx="974.7113103185542" cy="287.7588110453556" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="464" data-strategy="4" cx="966.5755382511744" cy="262.1194727646275" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="469" data-strategy="4" cx="906.3793609519555" cy="224.94077400452812" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="473" data-strategy="4" cx="908.1487189388537" cy="244.89379060491098" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="474" data-strategy="4" cx="958.6346217031986" cy="253.43125206768678" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="478" data-strategy="4" cx="901.3618119634928" cy="222.40612258533184" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="479" data-strategy="4" cx="976.8776667937952" cy="195.87750395641572" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="481" data-strategy="4" cx="974.4431959926187" cy="184.28477060472244" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="482" data-strategy="4" cx="970.2506395022382" cy="171.4617953443254" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="483" data-strategy="4" cx="911.4978742856923" cy="146.93309333366847" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="484" data-strategy="4" cx="904.00081853549" cy="162.44787715498134" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="485" data-strategy="4" cx="916.0449864531231" cy="157.19420201048808" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="486" data-strategy="4" cx="958.1230618585796" cy="180.1292678964992" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="487" data-strategy="4" cx="974.4296260940663" cy="182.61411111781942" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="488" data-strategy="4" cx="937.1782051565175" cy="151.75863703642045" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="489" data-strategy="4" cx="979.9385334640267" cy="197.01087564713728" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="490" data-strategy="4" cx="948.2246454133851" cy="222.32058595382787" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="491" data-strategy="4" cx="972.9960253075045" cy="215.44218751793633" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="492" data-strategy="4" cx="935.9207802522195" cy="211.61040333260772" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="493" data-strategy="4" cx="926.1411675624366" cy="230.26148786053693" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="494" data-strategy="4" cx="971.0821311243277" cy="208.04775443987583" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="495" data-strategy="4" cx="935.3246685655027" cy="166.2643150407653" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="496" data-strategy="4" cx="972.010012112974" cy="177.88238051109775" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="497" data-strategy="4" cx="978.8556216249444" cy="198.2620484046783" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="498" data-strategy="4" cx="903.2864183873473" cy="206.04335097116723" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="499" data-strategy="4" cx="972.0373979781757" cy="251.0706938256385" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="500" data-strategy="4" cx="970.971834454299" cy="301.58222350963234" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="501" data-strategy="4" cx="934.535286324214" cy="276.958110576605" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="502" data-strategy="4" cx="965.0796026963811" cy="276.5182317817804" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="503" data-strategy="4" cx="935.4030082239329" cy="315.1263323159364" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="526" data-strategy="4" cx="930.8955246474404" cy="98.54009272864405" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="534" data-strategy="4" cx="961.5030332896534" cy="182.72981493660828" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="574" data-strategy="4" cx="954.6233847458172" cy="161.5659770889905" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="575" data-strategy="4" cx="944.5288923827694" cy="187.45751950700915" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="576" data-strategy="4" cx="907.4039005303623" cy="213.09637445525058" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="577" data-strategy="4" cx="978.4778422407012" cy="226.5796452911911" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="578" data-strategy="4" cx="936.2764880982957" cy="293.23207255694797" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="579" data-strategy="4" cx="952.7544194580769" cy="318.06781608213146" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="580" data-strategy="4" cx="980.4250829267237" cy="320.84654105900523" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="581" data-strategy="4" cx="903.9945143364727" cy="304.03580777508233" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="582" data-strategy="4" cx="910.0137063810016" cy="299.1283016975574" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="583" data-strategy="4" cx="964.0898285271346" cy="311.43996120762847" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="584" data-strategy="4" cx="921.022810415871" cy="312.0713577739163" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="585" data-strategy="4" cx="949.5435643745207" cy="346.0645564753637" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="586" data-strategy="4" cx="950.1805710239637" cy="321.1579369500674" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="587" data-strategy="4" cx="898.4211139811517" cy="298.4303331911722" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="588" data-strategy="4" cx="963.5820116735744" cy="269.32114129554793" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="589" data-strategy="4" cx="923.5000289360898" cy="299.1785232988662" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="590" data-strategy="4" cx="904.6917338108672" cy="284.35121290469436" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="591" data-strategy="4" cx="909.9590569642907" cy="270.6889460124305" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="592" data-strategy="4" cx="971.1641091100762" cy="335.6798653499042" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="593" data-strategy="4" cx="928.3400262567936" cy="382.1470776069833" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="594" data-strategy="4" cx="897.5143221893996" cy="369.5829541067885" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="595" data-strategy="4" cx="927.6177110731684" cy="351.1514545896938" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="596" data-strategy="4" cx="951.2267819890387" cy="365.10528632696503" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="597" data-strategy="4" cx="919.6841073791145" cy="419.61150819024755" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="598" data-strategy="4" cx="903.1424650219133" cy="414.27125458749794" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="599" data-strategy="4" cx="950.1232027799916" cy="426.6701082588333" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="600" data-strategy="4" cx="912.061449464961" cy="460.3700641056637" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="601" data-strategy="4" cx="914.0004502613999" cy="453.49588959393054" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="602" data-strategy="4" cx="903.5356568561933" cy="416.31559314377057" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="603" data-strategy="4" cx="901.6752259453292" cy="410.1768448395074" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="604" data-strategy="4" cx="935.7556775560546" cy="401.45478839092766" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="605" data-strategy="4" cx="920.1718649166212" cy="390.87771001060975" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="606" data-strategy="4" cx="939.4891174727898" cy="393.5444150164991" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="607" data-strategy="4" cx="924.8433533215405" cy="402.5459611553851" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="608" data-strategy="4" cx="935.7379220765292" cy="362.52780152606346" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="609" data-strategy="4" cx="900.6845062230301" cy="360.76316947721585" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="610" data-strategy="4" cx="903.2983572420895" cy="356.67299651178945" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="611" data-strategy="4" cx="965.9500908152054" cy="365.11127859171313" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="612" data-strategy="4" cx="913.5235970274363" cy="329.99796030364024" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="613" data-strategy="4" cx="925.8608851028209" cy="351.71909328472384" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="614" data-strategy="4" cx="903.824081002308" cy="355.7675290137699" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="615" data-strategy="4" cx="943.9428433619374" cy="334.68270516255916" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="616" data-strategy="4" cx="922.3249213474039" cy="321.91271009266774" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="617" data-strategy="4" cx="904.0933024419902" cy="335.00941415025244" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="618" data-strategy="4" cx="905.5858906999106" cy="333.243010332275" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="619" data-strategy="4" cx="959.5616524282266" cy="320.40866705288096" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="620" data-strategy="4" cx="920.5236939461282" cy="296.7641487886974" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="621" data-strategy="4" cx="898.6322431238973" cy="282.1595736812774" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="622" data-strategy="4" cx="918.6886272467297" cy="264.36944334988664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="623" data-strategy="4" cx="965.6401665207283" cy="239.71150195658035" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="624" data-strategy="4" cx="911.1350151563996" cy="216.35819981413331" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="625" data-strategy="4" cx="947.9231573131428" cy="258.79009030409406" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="626" data-strategy="4" cx="898.7199429133534" cy="267.5166910223962" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="627" data-strategy="4" cx="933.0021903524488" cy="254.55868979234168" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="628" data-strategy="4" cx="957.7348604820792" cy="272.8841937715711" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="629" data-strategy="4" cx="976.9272229689474" cy="286.1420260002166" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="630" data-strategy="4" cx="931.3702258976402" cy="283.6849580317214" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="631" data-strategy="4" cx="902.3344082421448" cy="286.1982292077191" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="632" data-strategy="4" cx="965.4325005728103" cy="303.57150974106565" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="633" data-strategy="4" cx="950.0559373545242" cy="321.13967425442627" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="634" data-strategy="4" cx="978.0102630481593" cy="351.2925267030093" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="635" data-strategy="4" cx="955.7532238924005" cy="345.5142332305336" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="636" data-strategy="4" cx="937.5322300854671" cy="359.6112236327568" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="637" data-strategy="4" cx="963.7688836445002" cy="336.13921397774584" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="638" data-strategy="4" cx="963.6671838779918" cy="309.7674149541461" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="639" data-strategy="4" cx="908.3488381803802" cy="312.30250790041595" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="640" data-strategy="4" cx="904.9094622784552" cy="315.7888749435294" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="641" data-strategy="4" cx="959.1173109621602" cy="313.42908259018037" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="642" data-strategy="4" cx="905.3806452417741" cy="297.9875110661046" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="643" data-strategy="4" cx="957.367640543667" cy="318.9844204032491" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="644" data-strategy="4" cx="918.6254670123295" cy="332.57386140944016" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="645" data-strategy="4" cx="918.4677231102905" cy="325.521270181141" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="646" data-strategy="4" cx="944.2504742128468" cy="301.5558274031712" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="647" data-strategy="4" cx="966.4061810863645" cy="311.6173572323922" r="3" fill="#94a3b8" fill-opacity=".65"></circle><text x="939.25" y="572" text-anchor="middle" >AH｜動能持有 24 日</text><circle class="point" data-window="1" data-strategy="5" cx="1120.9303709781655" cy="341.44973061607817" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="2" data-strategy="5" cx="1158.4104399538323" cy="330.11320112090465" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="3" data-strategy="5" cx="1091.8617034838567" cy="326.5790629303891" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="4" data-strategy="5" cx="1168.995104333523" cy="314.22477510304054" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="5" data-strategy="5" cx="1110.2760891382786" cy="322.71453937151136" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="6" data-strategy="5" cx="1113.7300728602745" cy="328.89634909698833" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="7" data-strategy="5" cx="1142.7217621665395" cy="331.9522322721365" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="8" data-strategy="5" cx="1153.544311795849" cy="320.6278312065244" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="9" data-strategy="5" cx="1088.8335105019348" cy="298.438518663531" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="10" data-strategy="5" cx="1160.994010399793" cy="263.0469576433105" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="11" data-strategy="5" cx="1122.951945385309" cy="272.688268088786" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="12" data-strategy="5" cx="1154.9503437932262" cy="277.00915670700385" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="13" data-strategy="5" cx="1132.1244198999975" cy="246.99304256784367" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="14" data-strategy="5" cx="1130.2701747762171" cy="246.2624345715658" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="15" data-strategy="5" cx="1126.3441844841254" cy="212.2127078711896" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="16" data-strategy="5" cx="1149.854801920067" cy="225.05210064739273" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="17" data-strategy="5" cx="1105.002179910364" cy="216.59017619076937" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="39" data-strategy="5" cx="1107.164764789202" cy="309.4806805566512" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="44" data-strategy="5" cx="1101.8948564151372" cy="305.0169876630525" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="49" data-strategy="5" cx="1168.3566991146836" cy="347.6171214229628" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="54" data-strategy="5" cx="1165.7806726601245" cy="327.32818780541044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="59" data-strategy="5" cx="1145.211863463022" cy="327.93941109358286" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="67" data-strategy="5" cx="1122.5145625610628" cy="334.297839148121" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="70" data-strategy="5" cx="1168.5878890958468" cy="313.1137254947332" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="72" data-strategy="5" cx="1135.0995794205858" cy="263.8267700163621" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="75" data-strategy="5" cx="1154.7449524764875" cy="284.41778094037215" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="77" data-strategy="5" cx="1097.0751038141339" cy="312.9293176042396" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="80" data-strategy="5" cx="1112.9998792392191" cy="305.3742000100635" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="82" data-strategy="5" cx="1104.5381042116098" cy="286.9784188456973" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="83" data-strategy="5" cx="1101.3233996352287" cy="291.55353683039385" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="85" data-strategy="5" cx="1145.5223153274087" cy="272.2999083982625" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="87" data-strategy="5" cx="1153.2345951612356" cy="288.65504036208915" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="88" data-strategy="5" cx="1167.927335294929" cy="278.3944962033368" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="90" data-strategy="5" cx="1129.050957905047" cy="278.0995315643419" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="93" data-strategy="5" cx="1101.2629810835251" cy="286.1301002764717" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="96" data-strategy="5" cx="1165.971233328739" cy="297.0912899291606" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="98" data-strategy="5" cx="1144.4193896652591" cy="300.50125267804367" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="101" data-strategy="5" cx="1121.0951169246023" cy="282.4904046816882" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="103" data-strategy="5" cx="1093.5290544889142" cy="302.85214119483214" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="106" data-strategy="5" cx="1128.569584332039" cy="296.97355145422415" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="111" data-strategy="5" cx="1164.486418214238" cy="324.0596285377153" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="114" data-strategy="5" cx="1126.7037918299404" cy="359.26524707253276" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="115" data-strategy="5" cx="1096.1687519053437" cy="352.44019043097956" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="116" data-strategy="5" cx="1122.7626156122444" cy="342.11149772825644" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="117" data-strategy="5" cx="1122.7051019003945" cy="344.25576033364956" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="118" data-strategy="5" cx="1090.3053793912895" cy="330.0412744893909" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="119" data-strategy="5" cx="1143.2386834879198" cy="354.34550257649175" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="120" data-strategy="5" cx="1149.6454601658925" cy="395.82311796555035" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="121" data-strategy="5" cx="1113.2050510653492" cy="395.7148309918407" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="122" data-strategy="5" cx="1122.4073715535694" cy="370.8804227953332" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="123" data-strategy="5" cx="1126.0103474072448" cy="373.2847806682911" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="124" data-strategy="5" cx="1111.9225097998599" cy="359.71737124428756" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="125" data-strategy="5" cx="1104.379247060183" cy="349.3092916758551" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="126" data-strategy="5" cx="1101.976233541946" cy="365.17113652344955" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="127" data-strategy="5" cx="1086.6120978093522" cy="365.33732607841335" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="128" data-strategy="5" cx="1154.0831383026127" cy="383.0230904430655" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="129" data-strategy="5" cx="1106.6922747400563" cy="360.9767389499777" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="130" data-strategy="5" cx="1102.6039930133832" cy="357.18808083159763" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="131" data-strategy="5" cx="1145.5006176237507" cy="343.10540518418804" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="132" data-strategy="5" cx="1144.7899491763144" cy="338.9447348613668" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="133" data-strategy="5" cx="1133.7064544652033" cy="332.9413553484311" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="134" data-strategy="5" cx="1114.341355713309" cy="321.5358341449549" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="135" data-strategy="5" cx="1117.8100626237372" cy="293.9921722198069" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="136" data-strategy="5" cx="1114.9074631163328" cy="296.6135717783674" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="137" data-strategy="5" cx="1167.0085905504284" cy="289.83190799028273" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="138" data-strategy="5" cx="1125.0504596022051" cy="287.23601197398193" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="139" data-strategy="5" cx="1165.0996542313103" cy="301.1385070462751" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="140" data-strategy="5" cx="1152.6215402196667" cy="301.68237884691627" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="141" data-strategy="5" cx="1170.1067870017048" cy="334.09575023225153" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="142" data-strategy="5" cx="1141.8906175938923" cy="320.92922154766853" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="143" data-strategy="5" cx="1107.6798391014058" cy="291.90592165566073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="144" data-strategy="5" cx="1132.5027552084873" cy="287.9078131275858" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="145" data-strategy="5" cx="1133.4950001728234" cy="323.6962728194418" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="146" data-strategy="5" cx="1160.3493049222097" cy="310.24806929311444" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="147" data-strategy="5" cx="1159.1531177880738" cy="322.153410702362" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="148" data-strategy="5" cx="1121.4136288447278" cy="312.18277389164655" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="149" data-strategy="5" cx="1098.8984579384896" cy="327.94873020810655" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="150" data-strategy="5" cx="1100.948467373678" cy="306.02717824504685" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="151" data-strategy="5" cx="1132.59612329671" cy="307.8335583503406" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="152" data-strategy="5" cx="1130.6517975421878" cy="292.71690043230484" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="153" data-strategy="5" cx="1160.3975916701718" cy="304.376175242541" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="154" data-strategy="5" cx="1158.854406354814" cy="307.7641879200044" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="155" data-strategy="5" cx="1156.5962381639472" cy="312.7862584365589" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="156" data-strategy="5" cx="1150.9757973928083" cy="304.06336933768915" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="157" data-strategy="5" cx="1128.7838759331053" cy="316.32108876210935" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="158" data-strategy="5" cx="1115.8769712400176" cy="325.10037553198026" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="159" data-strategy="5" cx="1110.543889675029" cy="333.4524739098471" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="160" data-strategy="5" cx="1089.0929814132671" cy="320.8542556590456" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="161" data-strategy="5" cx="1106.5479238867317" cy="321.6202748075045" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="162" data-strategy="5" cx="1087.9365464106697" cy="313.774968678377" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="163" data-strategy="5" cx="1107.662786287868" cy="296.3375468838018" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="164" data-strategy="5" cx="1145.5220156993073" cy="287.0553292431168" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="165" data-strategy="5" cx="1137.6548573789398" cy="263.74761494153245" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="166" data-strategy="5" cx="1167.493303405452" cy="255.89186965269184" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="167" data-strategy="5" cx="1135.6723270253449" cy="269.96803844018143" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="169" data-strategy="5" cx="1132.216018965826" cy="271.48810378800414" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="174" data-strategy="5" cx="1163.339425271813" cy="313.9230001773144" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="185" data-strategy="5" cx="1156.0461504954944" cy="321.1628104359403" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="190" data-strategy="5" cx="1118.8373021748391" cy="345.46782512664583" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="195" data-strategy="5" cx="1122.8563466454427" cy="305.4333634725064" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="200" data-strategy="5" cx="1145.9535610625255" cy="296.4492940764705" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="202" data-strategy="5" cx="1107.6985961225528" cy="276.2871411861354" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="207" data-strategy="5" cx="1118.7202638332208" cy="326.0346691834441" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="212" data-strategy="5" cx="1119.816263149797" cy="305.06527425983757" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="217" data-strategy="5" cx="1111.315862059115" cy="308.2574533516567" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="222" data-strategy="5" cx="1152.1326616106537" cy="303.94386861721836" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="260" data-strategy="5" cx="1172.0900117987126" cy="321.72162361908875" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="302" data-strategy="5" cx="1157.9922535192968" cy="300.06622780587554" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="303" data-strategy="5" cx="1126.6985349808326" cy="272.5594068659634" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="350" data-strategy="5" cx="1095.0619179569687" cy="282.7782094705658" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="351" data-strategy="5" cx="1121.7042597984703" cy="282.5089853751011" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="352" data-strategy="5" cx="1118.0022970578214" cy="310.1543986256733" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="353" data-strategy="5" cx="1144.508985811164" cy="311.50880388653917" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="354" data-strategy="5" cx="1163.3172062173303" cy="302.48966429108367" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="355" data-strategy="5" cx="1121.738393207595" cy="299.8326213397954" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="356" data-strategy="5" cx="1090.043621716984" cy="315.81567106460597" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="357" data-strategy="5" cx="1092.4062097028993" cy="311.8758979035255" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="358" data-strategy="5" cx="1118.975310602175" cy="315.84109095378517" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="359" data-strategy="5" cx="1170.7219928739041" cy="319.36291474916766" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="360" data-strategy="5" cx="1126.6045998950026" cy="303.5849318813488" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="361" data-strategy="5" cx="1099.3122324317085" cy="307.29296301840736" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="362" data-strategy="5" cx="1126.3832978613152" cy="304.4095838761876" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="363" data-strategy="5" cx="1143.7412032632478" cy="303.6599019094916" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="364" data-strategy="5" cx="1154.8036216642777" cy="299.5844136380761" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="365" data-strategy="5" cx="1111.2470648078906" cy="279.3562529423692" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="366" data-strategy="5" cx="1116.360197067093" cy="255.0973855886042" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="367" data-strategy="5" cx="1095.3263798759604" cy="267.9792044640959" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="368" data-strategy="5" cx="1103.3580374975056" cy="269.1726361451042" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="369" data-strategy="5" cx="1134.2854329012737" cy="244.1259025780708" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="370" data-strategy="5" cx="1109.6791546416437" cy="217.1948186072366" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="371" data-strategy="5" cx="1096.2722133907396" cy="207.12115883489653" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="372" data-strategy="5" cx="1090.7903251234786" cy="200.37444060371266" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="373" data-strategy="5" cx="1114.8796902025565" cy="199.355609545554" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="374" data-strategy="5" cx="1097.5117329501659" cy="251.37602333904624" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="375" data-strategy="5" cx="1129.7330138961966" cy="227.8326175049929" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="376" data-strategy="5" cx="1105.8027127384278" cy="218.57904693529673" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="377" data-strategy="5" cx="1117.533237282846" cy="217.17756279855084" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="378" data-strategy="5" cx="1115.2048820279865" cy="217.58615388370097" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="379" data-strategy="5" cx="1116.6292903805102" cy="179.12873316742977" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="380" data-strategy="5" cx="1170.5511155189781" cy="167.1374945744516" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="381" data-strategy="5" cx="1111.5293756729152" cy="159.18861929096437" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="382" data-strategy="5" cx="1099.5231207111033" cy="179.10454855122327" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="383" data-strategy="5" cx="1109.781114211133" cy="208.8006061145639" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="384" data-strategy="5" cx="1170.7873903858613" cy="216.43606634329112" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="385" data-strategy="5" cx="1099.336450667915" cy="229.91125238834812" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="386" data-strategy="5" cx="1147.1453750738804" cy="211.22711410518272" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="387" data-strategy="5" cx="1096.0195125509163" cy="238.29919914693014" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="388" data-strategy="5" cx="1117.7799840664186" cy="224.5615585619251" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="389" data-strategy="5" cx="1157.9746093321912" cy="207.88843743659336" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="390" data-strategy="5" cx="1124.0724003090281" cy="233.6688011177816" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="391" data-strategy="5" cx="1123.6337958110933" cy="265.9904587166287" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="392" data-strategy="5" cx="1088.3579725827615" cy="300.4753995066091" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="393" data-strategy="5" cx="1091.9976005841656" cy="302.51824815194374" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="394" data-strategy="5" cx="1142.9857956773867" cy="311.2713834348563" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="406" data-strategy="5" cx="1125.9633669235836" cy="297.4167343947741" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="411" data-strategy="5" cx="1100.1470712680348" cy="300.53075034309813" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="416" data-strategy="5" cx="1163.4768265148116" cy="247.3474284358271" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="421" data-strategy="5" cx="1124.5990195268441" cy="249.71701807407683" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="426" data-strategy="5" cx="1130.6020051536896" cy="235.26797407094043" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="432" data-strategy="5" cx="1155.8630758210595" cy="259.1606729754893" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="433" data-strategy="5" cx="1145.0242052627934" cy="251.83437386220987" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="434" data-strategy="5" cx="1107.34195457086" cy="267.64725656948497" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="435" data-strategy="5" cx="1125.6515596144266" cy="294.1383070140077" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="436" data-strategy="5" cx="1091.4088982758597" cy="272.61433463132164" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="437" data-strategy="5" cx="1151.0484841170464" cy="303.0856021244818" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="438" data-strategy="5" cx="1126.824079245456" cy="311.8097739908535" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="439" data-strategy="5" cx="1170.316636514163" cy="286.1633064921583" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="440" data-strategy="5" cx="1094.9915636229298" cy="285.51576178462756" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="441" data-strategy="5" cx="1117.236030740369" cy="295.5625767384513" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="442" data-strategy="5" cx="1090.120691194598" cy="291.9066863403074" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="464" data-strategy="5" cx="1115.886331073107" cy="274.831774111591" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="469" data-strategy="5" cx="1167.7593626849116" cy="249.06324609359734" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="473" data-strategy="5" cx="1126.5535691162438" cy="250.90457990186073" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="474" data-strategy="5" cx="1144.0533726292588" cy="255.5544324936387" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="478" data-strategy="5" cx="1117.2205069040806" cy="229.11065181683594" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="479" data-strategy="5" cx="1154.5680996645956" cy="217.28647785330205" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="481" data-strategy="5" cx="1125.9462321919018" cy="208.41887921967688" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="482" data-strategy="5" cx="1088.1644015826816" cy="190.82644853937143" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="483" data-strategy="5" cx="1113.7557968228398" cy="162.26428795975198" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="484" data-strategy="5" cx="1172.6461304550683" cy="183.2706536321336" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="485" data-strategy="5" cx="1089.1740702593993" cy="180.13165423592724" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="486" data-strategy="5" cx="1110.6783959189193" cy="219.53459561077696" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="487" data-strategy="5" cx="1114.3598223928238" cy="223.10161315113714" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="488" data-strategy="5" cx="1090.5145985815777" cy="184.79115608215568" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="489" data-strategy="5" cx="1163.4644270702006" cy="209.46019322349005" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="490" data-strategy="5" cx="1131.920616976783" cy="246.7828441596205" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="491" data-strategy="5" cx="1112.180517896162" cy="250.33553225448557" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="492" data-strategy="5" cx="1156.9319830888535" cy="239.28439012900648" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="493" data-strategy="5" cx="1099.4296910703595" cy="245.1038594040246" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="494" data-strategy="5" cx="1109.758681135884" cy="225.96385067773707" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="495" data-strategy="5" cx="1094.7013086426884" cy="159.7482678279547" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="496" data-strategy="5" cx="1089.6859565736202" cy="194.56001449924796" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="497" data-strategy="5" cx="1124.285235384734" cy="204.55099791296732" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="498" data-strategy="5" cx="1164.331116505697" cy="205.2512265936237" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="499" data-strategy="5" cx="1111.0302401576264" cy="262.16936440078865" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="500" data-strategy="5" cx="1134.258321508941" cy="304.6342570424976" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="501" data-strategy="5" cx="1138.4923984827735" cy="288.3849174741472" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="502" data-strategy="5" cx="1158.4659213491152" cy="283.7694621932657" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="503" data-strategy="5" cx="1089.69166988551" cy="316.71641566357" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="526" data-strategy="5" cx="1086.4581655818079" cy="124.49346247194805" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="534" data-strategy="5" cx="1163.3886926690848" cy="185.29015089565223" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="574" data-strategy="5" cx="1088.0633411782474" cy="154.77507114903696" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="575" data-strategy="5" cx="1171.0785776084451" cy="179.38663677795446" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="576" data-strategy="5" cx="1149.1393697926767" cy="229.22536472299464" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="577" data-strategy="5" cx="1165.3358899576585" cy="221.91307227120313" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="578" data-strategy="5" cx="1159.203616275711" cy="296.4252294356024" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="579" data-strategy="5" cx="1142.6811174687523" cy="319.3545109174847" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="580" data-strategy="5" cx="1150.09724217282" cy="316.0258661767627" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="581" data-strategy="5" cx="1153.1095326505301" cy="291.51002587240987" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="582" data-strategy="5" cx="1139.501844211212" cy="278.5873407525112" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="583" data-strategy="5" cx="1124.1136322369478" cy="310.3211892153382" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="584" data-strategy="5" cx="1117.9892208968774" cy="301.2318842307704" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="585" data-strategy="5" cx="1088.4292647404438" cy="349.52653744153935" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="586" data-strategy="5" cx="1162.0133354078016" cy="319.8982707456861" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="587" data-strategy="5" cx="1117.0344838518456" cy="291.0632959146664" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="588" data-strategy="5" cx="1156.0580889949533" cy="270.25331217090445" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="589" data-strategy="5" cx="1172.1509500716468" cy="297.01814193506283" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="590" data-strategy="5" cx="1102.9282863923347" cy="293.74709859814584" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="591" data-strategy="5" cx="1119.8995667604704" cy="277.00039359067864" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="592" data-strategy="5" cx="1117.9633135177853" cy="334.0540030859419" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="593" data-strategy="5" cx="1166.8881854676301" cy="348.19526241833" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="594" data-strategy="5" cx="1149.1617484580288" cy="343.9996477014837" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="595" data-strategy="5" cx="1137.6343798093226" cy="334.18589607479845" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="596" data-strategy="5" cx="1123.6308936830897" cy="364.13000928517334" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="597" data-strategy="5" cx="1087.0215444786893" cy="400.9554544506839" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="598" data-strategy="5" cx="1094.6845608081078" cy="394.53517742063593" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="599" data-strategy="5" cx="1152.592139428811" cy="410.296950111825" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="600" data-strategy="5" cx="1139.2356132584184" cy="442.11506865696117" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="601" data-strategy="5" cx="1131.3265281060023" cy="434.7480295590279" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="602" data-strategy="5" cx="1140.5297361595087" cy="406.284439774058" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="603" data-strategy="5" cx="1166.172497744014" cy="397.66944628879645" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="604" data-strategy="5" cx="1139.0153898681697" cy="389.9741845170365" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="605" data-strategy="5" cx="1137.1401177527375" cy="383.2210929684486" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="606" data-strategy="5" cx="1120.5470706645592" cy="390.0388325250533" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="607" data-strategy="5" cx="1159.4633892169998" cy="400.12215336998764" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="608" data-strategy="5" cx="1141.100757743506" cy="361.78377309562916" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="609" data-strategy="5" cx="1162.3922341924922" cy="355.4040508411937" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="610" data-strategy="5" cx="1100.5258230732254" cy="350.52057617800114" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="611" data-strategy="5" cx="1094.15727488062" cy="356.71443483085795" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="612" data-strategy="5" cx="1120.8681214526573" cy="332.59795405173526" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="613" data-strategy="5" cx="1112.8669337146268" cy="351.20316982083835" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="614" data-strategy="5" cx="1148.665374031095" cy="351.531065982119" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="615" data-strategy="5" cx="1111.0453091829347" cy="343.94382620163066" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="616" data-strategy="5" cx="1112.2070369313437" cy="331.2267976301897" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="617" data-strategy="5" cx="1110.8636094851192" cy="317.8618537135193" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="618" data-strategy="5" cx="1164.2458056234243" cy="334.174789305939" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="619" data-strategy="5" cx="1153.6052498846097" cy="317.8050209144734" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="620" data-strategy="5" cx="1122.367819898194" cy="296.8387590648004" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="621" data-strategy="5" cx="1124.4024152870504" cy="284.1424533953944" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="622" data-strategy="5" cx="1112.8171939650726" cy="261.49267068878316" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="623" data-strategy="5" cx="1156.6701017233477" cy="243.45071820050686" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="624" data-strategy="5" cx="1120.8135993764292" cy="254.5463490601422" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="625" data-strategy="5" cx="1170.9398133688896" cy="294.85932880747913" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="626" data-strategy="5" cx="1114.1209106404986" cy="279.16352629205005" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="627" data-strategy="5" cx="1152.0815570240313" cy="288.2149807360462" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="628" data-strategy="5" cx="1109.0003186776532" cy="286.4585087095512" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="629" data-strategy="5" cx="1164.3800455108908" cy="283.1196235583277" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="630" data-strategy="5" cx="1119.3929021908707" cy="291.4860322954027" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="631" data-strategy="5" cx="1111.5031878285427" cy="297.2331465112285" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="632" data-strategy="5" cx="1159.603075835329" cy="300.0432311313579" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="633" data-strategy="5" cx="1114.2684818726707" cy="310.3883860237722" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="634" data-strategy="5" cx="1164.0438363555552" cy="331.2502699873643" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="635" data-strategy="5" cx="1148.4992207995429" cy="341.21061648672645" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="636" data-strategy="5" cx="1111.4069734118993" cy="349.13945088134045" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="637" data-strategy="5" cx="1122.6949078955367" cy="327.0289161704339" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="638" data-strategy="5" cx="1169.8664745691628" cy="312.57968548995757" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="639" data-strategy="5" cx="1108.3998702087633" cy="318.36211472567544" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="640" data-strategy="5" cx="1103.3554403870266" cy="308.0327303255719" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="641" data-strategy="5" cx="1120.5540887290204" cy="300.3923231980906" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="642" data-strategy="5" cx="1159.2761316526805" cy="323.78026580215146" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="643" data-strategy="5" cx="1096.7357281953891" cy="325.2428087137707" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="644" data-strategy="5" cx="1142.5780062476344" cy="336.0589594960202" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="645" data-strategy="5" cx="1137.3546874195263" cy="321.16741534885534" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="646" data-strategy="5" cx="1116.4873966418097" cy="300.4966765969328" r="3" fill="#94a3b8" fill-opacity=".65"></circle><circle class="point" data-window="647" data-strategy="5" cx="1156.2840329983994" cy="300.56503580904507" r="3" fill="#94a3b8" fill-opacity=".65"></circle><text x="1129.75" y="572" text-anchor="middle" >BH｜動能法人持有 24 日</text></svg></div></details></section>\n<section><details><summary>查看逐筆明細</summary><p class="note">搜尋只影響明細，不改變圖表。</p><input id="search" placeholder="搜尋編號或日期"><p id="tableCount"></p><div class="table-wrap"><table><thead id="head"><tr><th>窗口</th><th>開始日</th><th>結束日</th><th>共同狀態</th><th>REF｜流動性等權</th><th>A｜每週動能</th><th>B｜動能＋法人</th><th>C｜動能＋低波動</th><th>AH｜動能持有 24 日</th><th>BH｜動能法人持有 24 日</th></tr></thead><tbody id="rows"></tbody></table></div></details></section>\n<section><details><summary>圖表怎麼看</summary><p class="note">箱體是第 25–75 百分位，桃紅線是中位數，鬚為 1.5×IQR 範圍內最遠點。所有符合篩選的點（含離群值）都會保留。移到點上可查看日期、報酬和費用。左右抖動不代表日期；重疊窗口不能當成獨立樣本。箱形圖採六策略共同有效窗口，期末不清倉。</p></details></section>\n<section id="strategyGuide">\n<h2>六種決策方式怎麼做？</h2>\n<p class="note">這裡比較的是固定量化規則：目前沒有讀取新聞，也沒有讓 LLM 決定買賣。每個窗口都從 10 億元現金重新開始，持有 24 個交易日。</p>\n<div class="strategy-cards">\n<article><h3>REF｜流動性等權</h3><p>先看過去 20 個交易日的平均成交金額，選出合格股票中交易最活躍的 25 檔。初次等權買入後，持有到第 24 日。</p><p class="note">作為比較基準：用來看其他選股規則是否比「買一組流動性好的股票」更有幫助。成交活躍不代表一定會上漲。</p></article>\n<article><h3>A｜每週動能</h3><p>分別依過去 20 日與 60 日的除權息調整報酬，計算股票在同組中的排名。兩種排名各占一半，選綜合分數最高的 25 檔。</p><p class="note">每 5 個交易日重新選股、調整到等權，並非固定星期一。想法是延續近期強勢；若行情反轉，原本強勢的股票也可能回落。</p></article>\n<article><h3>B｜動能＋法人</h3><p>以 A 的動能分數占 75%，再加入 25% 法人排名。法人訊號是過去 20 日「外資與投信的淨買入股數合計 ÷ 成交股數合計」，比例越高，排名越前。</p><p class="note">選分數最高的 25 檔，每 5 個交易日調整。這是在動能之外檢查法人資金方向；法人買入不保證後續上漲。</p></article>\n<article><h3>C｜動能＋低波動</h3><p>以 A 的動能分數占 75%，另用過去 20 日每日調整報酬的標準差衡量波動；波動越小，排名越前，這部分占 25%。</p><p class="note">選分數最高的 25 檔，每 5 個交易日調整。希望在強勢股票中偏向走勢較平穩者，但「低波動」不等於不會跌，也不保證回撤較小。</p></article>\n<article><h3>AH｜動能持有 24 日</h3><p>初次選股完全使用 A 的規則，買入 25 檔後持有到第 24 日，不進行中途重新選股或等權調整。</p><p class="note">與 A 比較，可以看中途調整是否值得額外交易成本。成本較低，但也無法中途換掉轉弱股票。</p></article>\n<article><h3>BH｜動能法人持有 24 日</h3><p>初次選股完全使用 B 的規則，買入 25 檔後持有到第 24 日，不進行中途重新選股或等權調整。</p><p class="note">與 B 比較，檢查只在建倉時使用法人確認是否已足夠，以及降低換手後的表現。</p></article>\n</div>\n<p class="note"><strong>共同設定：</strong>下單只使用前一交易日及之前的訊號。建倉／調整時，股票目標合計占可交易淨值 90%，25 檔各占 3.6%；股數按前收換算為整千股，當日 VWAP 成交，收盤估值。成交價格、費用與整千股取整會使實際權重偏離目標。</p>\n<p class="note"><strong>交易費用：</strong>買賣手續費各 0.1425%，賣出另扣 0.3% 證交稅。第 24 日僅估值，不假設清倉；應收配息配股計入總資產，但不能當成可下單現金或股數。</p>\n<p class="note"><strong>分數比例與持股權重不同：</strong>B、C 的 75%／25% 是選股分數的組合比例，不是把資金分成兩組。六種策略都從相同合格股票池選股；股票池要求價格與法人訊號資料完整，且先排除近期無法解釋的公司行動。</p>\n</section>\n</main><div id="tooltip" hidden></div>\n<script id="payload" type="application/json">__PAYLOAD__</script><script>\nfunction quantile(values,p){const a=[...values].sort((a,b)=>a-b);if(!a.length)return null;const h=(a.length-1)*p,lo=Math.floor(h);return a[lo]+(a[Math.ceil(h)]-a[lo])*(h-lo)}\nfunction filteredWindows(windows,f){return windows.filter(w=>{const md=w.start.slice(5);return (!f.year||w.start.slice(0,4)===f.year)&&f.months.includes(Number(w.start.slice(5,7)))&&(!f.from||w.start>=f.from)&&(!f.to||w.start<=f.to)&&(!f.seasonFrom&&!f.seasonTo||!f.seasonFrom&&md<=f.seasonTo||!f.seasonTo&&md>=f.seasonFrom||f.seasonFrom&&f.seasonTo&&(f.seasonFrom<=f.seasonTo?md>=f.seasonFrom&&md<=f.seasonTo:md>=f.seasonFrom||md<=f.seasonTo))})}\nfunction chartStats(values){if(!values.length)return null;const q1=quantile(values,.25),med=quantile(values,.5),q3=quantile(values,.75),iqr=q3-q1;return{q1,med,q3,lo:Math.min(...values.filter(v=>v>=q1-1.5*iqr)),hi:Math.max(...values.filter(v=>v<=q3+1.5*iqr))}}\nwindow.BoxplotModel={quantile,filteredWindows,chartStats};\n// UI\nconst DATA=JSON.parse(document.querySelector(\'#payload\').textContent);\nconst $=id=>document.getElementById(id),esc=s=>String(s).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c])),pct=v=>(v*100).toFixed(3)+\'%\';\nlet selected=[];\nfor(const y of [...new Set(DATA.windows.map(w=>w.start.slice(0,4)))])$(\'year\').insertAdjacentHTML(\'beforeend\',\'<option value="\'+y+\'">\'+y+\'</option>\');\nfor(let m=1;m<=12;m++)$(\'month\').insertAdjacentHTML(\'beforeend\',\'<option value="\'+m+\'">\'+m+\' 月</option>\');\n$(\'head\').innerHTML=\'<tr>\'+[\'窗口\',\'開始日\',\'結束日\',\'共同狀態\',...DATA.strategies.map(s=>s.name)].map(h=>\'<th>\'+esc(h)+\'</th>\').join(\'\')+\'</tr>\';\nfunction filters(){return{year:$(\'year\').value,months:$(\'month\').value?[Number($(\'month\').value)]:Array.from({length:12},(_,i)=>i+1),from:$(\'from\').value,to:$(\'to\').value,seasonFrom:$(\'scope\').value===\'competition\'?\'10-27\':\'\',seasonTo:$(\'scope\').value===\'competition\'?\'11-27\':\'\'}}\nfunction validMonthDay(s){if(!s)return true;if(!/^\\d{2}-\\d{2}$/.test(s))return false;const d=new Date(\'2000-\'+s+\'T00:00:00Z\');return !Number.isNaN(d.valueOf())&&d.toISOString().slice(5,10)===s}\nfunction line(x1,y1,x2,y2,color,width,extra){return \'<line x1="\'+x1+\'" y1="\'+y1+\'" x2="\'+x2+\'" y2="\'+y2+\'" stroke="\'+color+\'" stroke-width="\'+width+\'" \'+(extra||\'\')+\'/>\'}\nfunction text(x,y,value,extra){return \'<text x="\'+x+\'" y="\'+y+\'" text-anchor="middle" \'+(extra||\'\')+\'>\'+esc(value)+\'</text>\'}\nfunction drawChart(items,included){\n const W=1250,H=620,L=82,R=25,T=35,B=90,all=DATA.windows.flatMap(w=>w.results.map(r=>r.return*100));\n const low=Math.floor(Math.min(0,...all)/5)*5-5,high=Math.ceil(Math.max(0,...all)/5)*5+5;\n const yp=v=>T+(high-v)/(high-low)*(H-T-B),xp=j=>L+(j+.5)*(W-L-R)/6;\n let out=\'<svg viewBox="0 0 \'+W+\' \'+H+\'" role="img" aria-label="\'+(included?\'有效窗口箱形圖\':\'排除窗口散點圖\')+\'">\';\n for(let v=Math.ceil(low/10)*10;v<=high;v+=10){if(v===0)continue;out+=line(L,yp(v),W-R,yp(v),\'#e2e8f0\',1)+text(L-32,yp(v)+5,v.toFixed(1)+\'%\')}\n out+=line(L,yp(0),W-R,yp(0),\'#111827\',2,\'class="zero-baseline" stroke-dasharray="7 5"\')+text(L-32,yp(0)+5,\'0%\',\'font-weight="bold"\');\n for(let j=0;j<6;j++){\n  const s=DATA.strategies[j],x=xp(j),stats=included?chartStats(items.map(w=>w.results[j].return*100)):null;\n  if(stats){out+=line(x,yp(stats.lo),x,yp(stats.hi),s.color,2)+\'<rect x="\'+(x-38)+\'" y="\'+yp(stats.q3)+\'" width="76" height="\'+(yp(stats.q1)-yp(stats.q3))+\'" fill="\'+s.color+\'" fill-opacity=".12" stroke="\'+s.color+\'"/>\';for(const v of [stats.lo,stats.hi])out+=line(x-38,yp(v),x+38,yp(v),s.color,2)}\n  for(const w of items){const r=w.results[j];out+=\'<circle class="point" data-window="\'+w.id+\'" data-strategy="\'+j+\'" cx="\'+(x+r.jitter*150)+\'" cy="\'+yp(r.return*100)+\'" r="3" fill="\'+(included?s.color:\'#94a3b8\')+\'" fill-opacity=".65"></circle>\'}\n  if(stats)out+=line(x-44,yp(stats.med),x+44,yp(stats.med),\'#be185d\',3.5,\'class="median-line"\');\n  out+=text(x,H-48,s.short);if(stats)out+=text(x,H-24,\'中位數 \'+stats.med.toFixed(2)+\'%\',\'class="median-label" style="fill:#be185d"\');\n }\n if(!items.length)out+=text(W/2,H/2,\'沒有符合條件的\'+(included?\'共同有效\':\'排除\')+\'窗口\',\'style="font-size:22px"\');\n return out+\'</svg>\';\n}\nfunction pointText(w,j,included){const r=w.results[j];return \'窗口 #\'+w.id+\'｜\'+DATA.strategies[j].name+\'\\n\'+w.start+\' 至 \'+w.end+\'\\n\'+(included?\'報酬\':\'暫算報酬\')+\'：\'+pct(r.return)+\'\\n費用/本金：\'+pct(r.cost)+\'\\n\'+(included?\'共同有效，納入箱形圖\':r.invalid?\'未採納：本策略資料也不可判定\':\'未採納：其他策略缺漏，排除共同窗口\')}\nfunction renderTable(){const q=$(\'search\').value.trim(),rows=selected.filter(w=>[w.id,w.start,w.end].join(\' \').includes(q));$(\'rows\').innerHTML=rows.map(w=>\'<tr>\'+[w.id,w.start,w.end,w.common?\'納入\':\'排除\',...w.results.map(r=>pct(r.return)+(r.invalid?\'（不可判定，暫算）\':!w.common?\'（未納入共同比較）\':\'\'))].map(c=>\'<td>\'+esc(c)+\'</td>\').join(\'\')+\'</tr>\').join(\'\');$(\'tableCount\').textContent=\'顯示 \'+rows.length+\'／\'+selected.length+\' 列（全部 650 個窗口）\'}\nfunction render(){$(\'competition\').setAttribute(\'aria-pressed\',String($(\'scope\').value===\'competition\'));$(\'reset\').setAttribute(\'aria-pressed\',String($(\'scope\').value!==\'competition\'));$(\'rangeNote\').textContent=$(\'scope\').value===\'competition\'?\'比賽期間：各年 10/27～11/27 開始的 24 日窗口，結束日可能超出此範圍。\':\'日期依窗口開始日判斷。\';const f=filters();let error=\'\';if(f.from&&f.to&&f.from>f.to)error=\'開始日期的「從」不能晚於「至」。\';$(\'error\').textContent=error;selected=error?[]:filteredWindows(DATA.windows,f);const good=selected.filter(w=>w.common),bad=selected.filter(w=>!w.common);$(\'status\').textContent=\'顯示 \'+good.length+\' 個有效窗口（選中 \'+selected.length+\' 個，排除 \'+bad.length+\' 個）\';$(\'excludedHeading\').textContent=\'（\'+bad.length+\' 個）\';$(\'validChart\').innerHTML=drawChart(good,true);$(\'excludedChart\').innerHTML=drawChart(bad,false);$(\'tooltip\').hidden=true;renderTable()}\nfunction reset(){$(\'scope\').value=\'all\';for(const id of [\'year\',\'month\',\'from\',\'to\',\'search\'])$(id).value=\'\';$(\'rangeDetails\').open=false}\nfor(const id of [\'scope\',\'year\',\'month\',\'from\',\'to\'])$(id).addEventListener(\'input\',render);\n$(\'search\').addEventListener(\'input\',renderTable);$(\'reset\').addEventListener(\'click\',()=>{reset();render()});$(\'competition\').addEventListener(\'click\',()=>{reset();$(\'scope\').value=\'competition\';render()});\ndocument.addEventListener(\'pointerover\',e=>{const p=e.target.closest?.(\'.point\');if(!p)return;const w=DATA.windows[Number(p.dataset.window)-1],j=Number(p.dataset.strategy);$(\'tooltip\').textContent=pointText(w,j,w.common);$(\'tooltip\').hidden=false;positionTooltip(e)});\nfunction positionTooltip(e){const t=$(\'tooltip\');if(t.hidden)return;t.style.left=Math.max(8,Math.min(e.clientX+15,innerWidth-t.offsetWidth-12))+\'px\';t.style.top=Math.max(8,Math.min(e.clientY+15,innerHeight-t.offsetHeight-12))+\'px\'}\ndocument.addEventListener(\'pointermove\',positionTooltip);document.addEventListener(\'pointerout\',e=>{if(e.target.classList?.contains(\'point\'))$(\'tooltip\').hidden=true});\nrender();\n</script></html>\n'
assert filter_template.count('__PAYLOAD__')==1
filtered_html=filter_template.replace('__PAYLOAD__',json.dumps(filter_payload,ensure_ascii=False,allow_nan=False).replace('</','<\\/'))
html_path=root/'動能基準研究_互動箱形圖_2026-10-13.html'
html_path.write_text(filtered_html,encoding='utf-8')
print('Updated original HTML. Windows:',len(filter_payload['windows']),'common valid:',sum(w['common'] for w in filter_payload['windows']))
print('Expected October starts:',sum(w['start'][5:7]=='10' for w in filter_payload['windows']))
print('Expected 2025 October starts:',sum(w['start'][:7]=='2025-10' for w in filter_payload['windows']))
print('Expected recurring competition-season starts:',sum('10-27'<=w['start'][5:]<='11-27' for w in filter_payload['windows']))


In [ ]:
html_path=root/'動能基準研究_互動箱形圖_2026-10-13.html'
png_path=root/'動能基準研究_箱形圖_全部窗口_2026-10-13.png'
seasonal_png=root/'動能基準研究_競賽季節_各年份箱形圖與散點_2026-10-13.png'
seasonal_report=root/'動能基準研究_競賽季節_歷年比較說明_2026-10-13.md'
print('Output paths updated to momentum-baseline filenames; analysis state retained.')